# Haptic Ground Truth — Colab Pipeline

Same four Sound2Hap algorithms plus **E**, the original rule-based thunder/rain/RMS mapper.

Convert **3–5 minute** video into **gated candidate haptic tracks** for human-in-the-loop evaluation.

**Runtime:** **T4 GPU** recommended for context detection (AST + ViViT). Sound2Hap A–D and rule-based E can run on CPU.

**Output:** mono **8 kHz** haptic WAV files + `events.json` + `algorithm_e_rule_based.json`

**Setup:** Run all cells top-to-bottom. Upload a video when prompted — no Google Drive needed.

| Phase | Component |
|-------|-----------|
| **1** | Tokenization (100Hz) + Frozen Context Detectors + AST/ViViT encoders |
| **2** | Frozen fusion → `events.json` → gated audio |
| **3** | Sound2Hap A–D |
| **4** | Rule-based E (ungated video + audio RMS) |

In [ ]:
# Install system + Python dependencies
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q numpy scipy librosa soundfile audioread resampy torch torchaudio matplotlib mosqito pyyaml transformers accelerate decord av opencv-python Pillow
# Optional: PANNs gives true framewise SED (~10 ms frames). Without it the
# detector falls back to densely-strided AST (100 ms frames). GPU runtime advised.
!pip install -q panns-inference


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/haptic-groundtruth")
PKG_DIR = PROJECT_ROOT / "haptic_gt"
PKG_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "__init__.py": """\"\"\"Ground-truth haptic track generation from video audio.\"\"\"

from .pipeline import generate_candidate_tracks

__all__ = ["generate_candidate_tracks"]
""",
    "algorithms/__init__.py": """\"\"\"Sound2Hap signal-processing algorithms plus the original rule-based mapper.\"\"\"
""",
    "algorithms/freq_shift.py": """\"\"\"
Frequency shifting audio-to-vibration (Sound2Hap / Okazaki et al., 2015).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

from pathlib import Path
from typing import Union

import librosa
import numpy as np
import soundfile as sf
import torch
from scipy.signal import butter, lfilter

from haptic_gt.utils.normalization import normalize_audio

VIB_SR = 8000


def _butter_bandpass(sr: int, center_hz: float = 250.0, q: float = 1.0, order: int = 4):
    bw = center_hz / q
    low_hz = max(center_hz - bw / 2, 1.0)
    high_hz = min(center_hz + bw / 2, sr / 2 - 1)
    wn = [low_hz / (sr / 2), high_hz / (sr / 2)]
    return butter(order, wn, btype="band")


def _butter_highpass(sr: int, cutoff_hz: float = 10.0, order: int = 2):
    wn = cutoff_hz / (sr / 2)
    return butter(order, wn, btype="high")


def process_file(
    in_wav: Union[str, Path],
    out_wav: Union[str, Path],
    centre_hz: float = 250.0,
    q: float = 1.0,
) -> None:
    y, sr = librosa.load(in_wav, sr=None, mono=True)

    wav_tensor = torch.from_numpy(y).float().unsqueeze(0)
    y_norm_t = normalize_audio(wav_tensor, normalize=True, strategy="peak")
    y = y_norm_t.squeeze(0).numpy()

    y_1ot = librosa.effects.pitch_shift(y, sr=sr, n_steps=-12, res_type="kaiser_best")
    y_2ot = librosa.effects.pitch_shift(y, sr=sr, n_steps=-24, res_type="kaiser_best")
    mix = y + y_1ot + y_2ot

    rms = np.sqrt(np.mean(mix**2) + 1e-12)
    mix /= rms * np.sqrt(2)

    b_hp, a_hp = _butter_highpass(sr, cutoff_hz=10.0)
    mix = lfilter(b_hp, a_hp, mix)

    b, a = _butter_bandpass(sr, centre_hz, q)
    mix_bp = lfilter(b, a, mix)
    mix_bp = librosa.resample(mix_bp, orig_sr=sr, target_sr=VIB_SR)
    mix_bp = np.clip(mix_bp, -1.0, 1.0)

    out_path = Path(out_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, mix_bp.astype(np.float32), VIB_SR, subtype="PCM_16")
""",
    "algorithms/haptic_gen.py": """\"\"\"
HapticGen-style RMS-driven NCO synthesis (Sound2Hap / Sung et al., CHI 2025).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from haptic_gt.utils.normalization import normalize_audio

WANTED_BIN_SIZE_SEC = 0.010
BASE_FREQ = 200.0
VIB_SR = 8000


def amp_env_on_wav_norm(
    wav_norm: np.ndarray,
    input_sample_rate: int,
    output_sample_rate: int,
) -> np.ndarray:
    wav_norm = wav_norm.squeeze()
    num_samples = len(wav_norm)
    duration_sec = num_samples / input_sample_rate
    samples_per_bin = int(WANTED_BIN_SIZE_SEC * input_sample_rate)
    num_bins = num_samples // samples_per_bin

    wav_chunks = np.array_split(wav_norm, num_bins)
    rms_bins = np.array([np.sqrt(np.mean(chunk**2)) for chunk in wav_chunks])
    rms_max = np.max(rms_bins)
    rms_norm = np.sqrt(2)
    rms_amplify = max(1.0, min(1.2, 1.0 / (rms_max * rms_norm)))
    rms_norm_amp = rms_norm * rms_amplify
    out_samples = int(duration_sec * output_sample_rate)

    phase_acc = 0.0
    output = np.zeros(out_samples)
    for i in range(out_samples):
        t = i / output_sample_rate
        t_prog = t / duration_sec
        bin_fi = t_prog * num_bins
        bin_lo = int(bin_fi)
        bin_hi = min(num_bins - 1, int(math.ceil(bin_fi)))
        bin_fr = bin_fi - bin_lo
        rms_val = (
            rms_bins[bin_lo] * (1.0 - bin_fr) + rms_bins[bin_hi] * bin_fr
        ) * rms_norm_amp
        freq_offset = (rms_val - 0.3) * 100.0
        phase_delta = 2.0 * math.pi * (BASE_FREQ + freq_offset) / output_sample_rate
        phase_acc = (phase_acc + phase_delta) % (2.0 * math.pi)
        output[i] = rms_val * math.sin(phase_acc)

    return output


def process_file(input_path: str | Path, output_path: str | Path) -> None:
    wav_data, sr = sf.read(input_path)
    wav_tensor = torch.from_numpy(wav_data).float().unsqueeze(0)
    wav_norm_tensor = normalize_audio(
        wav_tensor,
        normalize=True,
        strategy="peak",
        peak_clip_headroom_db=0,
        peak_normalize_db_clamp=0,
    )
    wav = wav_norm_tensor.squeeze(0).numpy()
    env_signal = amp_env_on_wav_norm(wav, sr, VIB_SR)

    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, env_signal, VIB_SR, subtype="PCM_16")
""",
    "algorithms/percept.py": """\"\"\"
Perception-level audio-to-vibration translator (Sound2Hap / Lee & Choi, CHI 2013).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from scipy.signal import find_peaks

from haptic_gt.utils.normalization import normalize_audio

AUDIO_SR = 44100
VIB_SR = 8000
FRAME_S = 4096
F1 = 175.0
F2 = 210.0
CR = 0.035
OR = 0.40
CV = 1
CL = 0.1
OL = 3.8
C_FULLBAND = 0.065
F_FULLBAND = 6400
C_BASS = 1.91
F_BASS = 200
C = 1.37

_ISO_FREQ = np.array(
    [
        25, 31.5, 40, 50, 63, 80, 100, 125, 160, 200, 250, 315, 400, 500, 630,
        800, 1000, 1250, 1600, 2000, 2500, 3150, 4000, 5000, 6300,
    ]
)
_ISO_SPL60 = np.array(
    [
        104.23, 99.08, 94.18, 89.96, 85.94, 82.05, 78.65, 75.56, 72.47, 69.86,
        67.53, 65.39, 63.45, 62.05, 60.81, 59.89, 60.01, 62.15, 63.19, 59.96,
        57.26, 56.42, 57.57, 60.89, 66.36,
    ]
)


def iso60phon(f: np.ndarray) -> np.ndarray:
    return np.interp(f, _ISO_FREQ, _ISO_SPL60, left=_ISO_SPL60[0], right=_ISO_SPL60[-1])


def auditory_loudness(frame: np.ndarray, content: str) -> float:
    if content == "music":
        c_use, f_max = C_BASS, F_BASS
    else:
        c_use, f_max = C_FULLBAND, F_FULLBAND

    mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(frame.size, 1 / AUDIO_SR)
    mask = (freqs >= 25) & (freqs <= f_max)
    mag = mag[mask]
    freqs = freqs[mask]

    db = 20 * np.log10(C * mag + 1e-12)
    af = iso60phon(freqs)
    loudness = c_use * np.sum(db / af)
    return max(0.0, loudness)


def auditory_roughness(frame: np.ndarray, peak_db: float = -40.0) -> float:
    mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(frame.size, 1 / AUDIO_SR)
    mask = (freqs >= 25) & (freqs <= 6400)
    mag = mag[mask]
    freqs = freqs[mask]

    db = 20 * np.log10(mag + 1e-12)
    thresh = db.max() + peak_db
    peaks, _ = find_peaks(db, height=thresh)

    f = freqs[peaks]
    x = mag[peaks]
    roughness = 0.0
    for i in range(len(f)):
        for j in range(i + 1, len(f)):
            f1, f2 = f[i], f[j]
            x1, x2 = x[i], x[j]
            xm, xM = min(x1, x2), max(x1, x2)
            fd = abs(f2 - f1)
            s = 0.24 / (0.0207 * min(f1, f2) + 18.96)
            term = ((xm * xM) ** 0.1 / 2.0) * (2 * xm / (xm + xM)) ** 3.11
            roughness += term * (math.exp(-3.5 * s * fd) - math.exp(-5.75 * s * fd))
    return roughness


def perceptual_targets(la: float, ra: float, content: str) -> tuple[float, float]:
    if content == "music":
        iv = CL * la - OL
    else:
        iv = CR * math.sqrt(la) * (ra**2) - OR
    rv = CV * ra
    return max(0, iv), rv


def amplitudes_from_percepts(iv: float, rv: float) -> tuple[float, float]:
    if iv <= 0.0:
        return 0.0, 0.0

    rv_max = (801.0 / 113.0) + 0.529 * iv + 0.479
    rv_adj = min(rv, rv_max)
    disc = max(0.0, 801.0 - 113.0 * (rv_adj - 0.529 * iv - 0.479))
    r1 = (28.3 + math.sqrt(disc)) / 56.3
    r2 = (28.3 - math.sqrt(disc)) / 56.3
    valid = [s for s in (r1, r2) if 0.0 <= s <= 1.0]
    s = min(valid) if valid else (28.3 / 56.3)

    a = ((25.8 * s**2 - 25.5 * s + rv_adj - 0.203) / 3.98) ** 2
    a2 = a * s
    a1 = a - a2
    return a1, a2


def synth_vibration(a1: float, a2: float, n_samples: int) -> np.ndarray:
    t = np.arange(n_samples) / VIB_SR
    return a1 * np.sin(2 * math.pi * F1 * t) + a2 * np.sin(2 * math.pi * F2 * t)


def read_wav_mono_44k(fname: str | Path) -> np.ndarray:
    wav_data, _ = sf.read(fname)
    if wav_data.ndim > 1:
        wav_data = wav_data.mean(axis=1)
    wav_tensor = torch.from_numpy(wav_data).float().unsqueeze(0)
    wav_norm_tensor = normalize_audio(
        wav_tensor,
        normalize=True,
        strategy="peak",
        peak_clip_headroom_db=0,
        peak_normalize_db_clamp=0,
    )
    return wav_norm_tensor.squeeze(0).numpy().astype("float32")


def process_file(
    in_wav: str | Path,
    out_wav: str | Path,
    content: str = "game",
) -> None:
    audio = read_wav_mono_44k(in_wav)
    hop_s = FRAME_S
    n_out_total = int(np.ceil(len(audio) * VIB_SR / AUDIO_SR))
    vib_full = np.zeros(n_out_total, dtype=np.float32)

    for start in range(0, len(audio), hop_s):
        block = audio[start : start + FRAME_S]
        if block.size == 0:
            break
        if block.size < FRAME_S:
            block = np.pad(block, (0, FRAME_S - block.size), "constant")

        la = auditory_loudness(block, content)
        ra = auditory_roughness(block)
        iv, rv = perceptual_targets(la, ra, content)
        a1, a2 = amplitudes_from_percepts(iv, rv)

        n_out = int(round(FRAME_S * VIB_SR / AUDIO_SR))
        vib_seg = synth_vibration(a1, a2, n_out)
        rms_seg = np.sqrt(np.mean(vib_seg**2) + 1e-12)
        vib_seg /= rms_seg * np.sqrt(2)

        out_start = int(round(start * VIB_SR / AUDIO_SR))
        out_end = out_start + n_out
        if out_end > n_out_total:
            vib_full[out_start:] += vib_seg[: n_out_total - out_start]
        else:
            vib_full[out_start:out_end] += vib_seg

    vib_full = np.clip(vib_full, -1.0, 1.0)
    out_path = Path(out_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, vib_full, VIB_SR, subtype="PCM_16")
""",
    "algorithms/pitch_match.py": """\"\"\"
Pitch Match audio-to-vibration (Sound2Hap / Kim et al., IEEE ToH 2023).

Python version used for Sound2Hap web tool. MATLAB version used in the study.

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import get_window, resample_poly

try:
    from mosqito.functions.loudness_zwtv._loudness_zwtv import loudness_zwtv

    MOSQITO_AVAILABLE = True
except Exception:
    MOSQITO_AVAILABLE = False

VIB_SR = 8000


@dataclass
class Config:
    regressionCoeffs: dict
    vibrationFreqRange: tuple
    binSizeMs: float
    overlapRatio: float
    smoothingWindow: int
    inputSampleRate: int
    outputSampleRate: int


def get_config() -> Config:
    return Config(
        regressionCoeffs={2: -0.005, 3: 0.003, 9: -0.015, 12: 0.008, 24: 0.008},
        vibrationFreqRange=(50.0, 398.0),
        binSizeMs=10.0,
        overlapRatio=0.5,
        smoothingWindow=3,
        inputSampleRate=44100,
        outputSampleRate=VIB_SR,
    )


def normalize_audio(audio: np.ndarray, do_normalize: bool) -> np.ndarray:
    scale_peak = 10 ** (-1 / 20)
    normalize_peak = 1.0
    wav_max = np.max(np.abs(audio)) + 1e-12
    rescaling = min(max(1.0, normalize_peak / wav_max), scale_peak / wav_max)
    if do_normalize or (rescaling < 1.0):
        audio = audio * rescaling
    return audio


def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))


def _specific_and_total_loudness_bark(audio_bin: np.ndarray, sr: int):
    if not MOSQITO_AVAILABLE:
        env = np.abs(audio_bin)
        total = float(np.mean(env))
        spec24 = np.zeros(24, dtype=np.float32)
        spec24[0] = total
        return spec24, total

    try:
        results = loudness_zwtv(audio_bin, sr, field_type="free")
        n_time = np.asarray(results["N"]).reshape(-1)
        n_spec = np.asarray(results["N_specific"])
        total_loudness = float(np.mean(n_time)) if n_time.size else 0.0

        if n_spec.ndim == 2 and n_spec.shape[1] >= 240:
            spec_time_mean = np.mean(n_spec, axis=0)
            spec24 = np.zeros(24, dtype=np.float32)
            for i in range(24):
                start = i * 10
                end = start + 10
                spec24[i] = float(np.sum(spec_time_mean[start:end]))
        else:
            if n_spec.ndim == 1:
                vec = n_spec
            else:
                vec = np.mean(n_spec, axis=0) if n_spec.size else np.zeros(240)
            idx = np.linspace(0, len(vec) - 1, 24)
            spec24 = np.interp(idx, np.arange(len(vec)), vec).astype(np.float32)

        spec24[~np.isfinite(spec24)] = 0.0
        return spec24, total_loudness
    except Exception:
        env = np.abs(audio_bin)
        total = float(np.mean(env))
        spec24 = np.zeros(24, dtype=np.float32)
        spec24[0] = total
        return spec24, total


def predict_vibration_frequency(specific_loudness_24: np.ndarray, cfg: Config) -> float:
    predicted = 0.0
    for bark_band, coeff in cfg.regressionCoeffs.items():
        idx = int(bark_band) - 1
        if 0 <= idx < len(specific_loudness_24):
            predicted += coeff * float(specific_loudness_24[idx]) * 1000.0
    predicted = abs(predicted)
    vmin, vmax = cfg.vibrationFreqRange
    return float(np.clip(predicted, vmin, vmax))


def analyze_audio_bins(audio: np.ndarray, sr: int, cfg: Config):
    bin_size = int(round(cfg.binSizeMs * sr / 1000.0))
    hop = max(1, int(round(bin_size * (1.0 - cfg.overlapRatio))))
    if bin_size < 2:
        bin_size = 2
    starts = np.arange(0, max(1, len(audio) - bin_size + 1), hop, dtype=int)
    if starts.size == 0:
        starts = np.array([0], dtype=int)

    times = (starts + bin_size / 2.0) / float(sr)
    freqs = np.zeros(starts.size, dtype=np.float32)
    amps = np.zeros(starts.size, dtype=np.float32)
    win = get_window("hann", bin_size, fftbins=False).astype(np.float32)

    for i, s in enumerate(starts):
        e = min(s + bin_size, len(audio))
        chunk = np.zeros(bin_size, dtype=np.float32)
        seg = audio[s:e]
        chunk[: len(seg)] = seg
        chunk *= win

        if rms(chunk) < 1e-3:
            freqs[i] = freqs[i - 1] if i > 0 else np.mean(cfg.vibrationFreqRange)
            amps[i] = 0.0
            continue

        spec24, loud = _specific_and_total_loudness_bark(chunk, sr)
        freqs[i] = predict_vibration_frequency(spec24, cfg)
        amps[i] = float(loud)

    if cfg.smoothingWindow > 1 and len(freqs) > cfg.smoothingWindow:
        k = cfg.smoothingWindow
        kernel = np.ones(k, dtype=np.float32) / k
        freqs = np.convolve(freqs, kernel, mode="same")

    return times.astype(np.float64), freqs.astype(np.float64), amps.astype(np.float64)


def generate_time_varying_vibration(audio: np.ndarray, sr: int, cfg: Config):
    bin_t, bin_f, bin_a = analyze_audio_bins(audio, sr, cfg)
    t = np.arange(len(audio), dtype=np.float64) / float(sr)

    if len(bin_t) == 1:
        f_inst = np.full_like(t, bin_f[0], dtype=np.float64)
        a_inst = np.full_like(t, bin_a[0], dtype=np.float64)
    else:
        try:
            from scipy.interpolate import PchipInterpolator

            f_inst = PchipInterpolator(bin_t, bin_f, extrapolate=True)(t)
        except Exception:
            f_inst = np.interp(t, bin_t, bin_f, left=bin_f[0], right=bin_f[-1])
        a_inst = np.interp(t, bin_t, bin_a, left=bin_a[0], right=bin_a[-1])

    rms_current = rms(a_inst)
    rms_norm = np.sqrt(2.0)
    rms_amplify = max(1.0, min(1.2, 1.0 / (rms_current * rms_norm))) if rms_current > 0 else 1.0
    a_inst = a_inst * (rms_norm * rms_amplify) if rms_current > 0 else np.full_like(t, 0.1)

    dt = 1.0 / float(sr)
    phi = np.empty_like(t)
    phi[0] = 0.0
    phi[1:] = 2.0 * np.pi * np.cumsum(f_inst[:-1]) * dt
    v = a_inst * np.sin(phi)

    fade_len = int(round(0.01 * sr))
    if len(v) > 2 * fade_len and fade_len > 0:
        fade_in = np.linspace(0.0, 1.0, fade_len)
        fade_out = np.linspace(1.0, 0.0, fade_len)
        v[:fade_len] *= fade_in
        v[-fade_len:] *= fade_out

    return v.astype(np.float32), f_inst.astype(np.float32), a_inst.astype(np.float32)


def generate_vibration_signal(audio: np.ndarray, sr: int, cfg: Config):
    v, f_arr, a_arr = generate_time_varying_vibration(audio, sr, cfg)
    analysis_info = {
        "method": "time_varying",
        "duration": len(audio) / float(sr),
        "freqMean": float(np.mean(f_arr)),
        "freqRange": (float(np.min(f_arr)), float(np.max(f_arr))),
        "freqStd": float(np.std(f_arr)),
    }
    return v, f_arr, a_arr, analysis_info


def _read_mono(path: str | Path):
    x, sr = sf.read(path, always_2d=False)
    x = x.astype(np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    return x, sr


def _write_int16_wav(path: str | Path, y: np.ndarray, sr: int):
    y = y / (np.max(np.abs(y)) + 1e-12)
    sf.write(path, y, sr, subtype="PCM_16")


def process_file(
    input_file: str | Path,
    output_file: str | Path,
    cfg: Config | None = None,
) -> dict:
    cfg = cfg or get_config()
    audio, sr = _read_mono(input_file)
    duration = len(audio) / float(sr)
    audio = normalize_audio(audio, True)

    v, f_arr, a_arr, info = generate_vibration_signal(audio, sr, cfg)
    fs_out = cfg.outputSampleRate
    if sr != fs_out:
        frac = Fraction(fs_out, sr).limit_denominator(1000)
        v = resample_poly(v, frac.numerator, frac.denominator)

    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    _write_int16_wav(out_path, v, fs_out)

    return {
        "inputFile": str(input_file),
        "outputFile": str(output_file),
        "duration": duration,
        "originalSr": sr,
        "targetSr": fs_out,
        "analysisInfo": info,
        "mosqitoAvailable": MOSQITO_AVAILABLE,
    }
""",
    "algorithms/rule_based.py": """\"\"\"Original rule-based haptic map (thunder / rain / audio RMS).

Port of ``vibrator-android/model_json/process_video.py``. Unlike Sound2Hap A–D
this runs on the ungated mix (and video frames when present), then writes a
40 ms intensity map plus an 8 kHz carrier WAV the Android A–E switcher can play.
\"\"\"

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import librosa
import numpy as np
import soundfile as sf

from haptic_gt.audio_io import VIB_SR

WINDOW_SIZE_MS = 40
CARRIER_HZ = 175.0

LIGHTNING_BRIGHTNESS_SCALE_MAX = 50.0
LIGHTNING_THRESHOLD_VISUAL_LOW = 140.0
LIGHTNING_THRESHOLD_VISUAL_HIGH = 160.0
RAIN_DIFF_THRESHOLD = 20
RAIN_RATIO_MAX = 0.12
RAIN_INTENSITY_MAX = 200.0
RAIN_INTENSITY_THRESHOLD = 30.0
AUDIO_RMS_MAX = 0.5
AUDIO_THUNDER_INTENSITY_THRESHOLD = 100.0
MIN_INTENSITY = 16


def mix_window(
    *,
    max_lightning: float,
    max_rain: float,
    avg_audio: float,
) -> tuple[str, int]:
    \"\"\"Choose event type and 0–255 intensity for one 40 ms window.\"\"\"
    event_type = "none"
    intensity = 0

    if max_lightning > LIGHTNING_THRESHOLD_VISUAL_LOW and avg_audio > AUDIO_THUNDER_INTENSITY_THRESHOLD:
        event_type = "thunder_both"
        intensity = 255
    elif max_lightning > LIGHTNING_THRESHOLD_VISUAL_HIGH:
        event_type = "thunder_visual"
        intensity = 255
    elif avg_audio > AUDIO_THUNDER_INTENSITY_THRESHOLD and max_lightning > LIGHTNING_THRESHOLD_VISUAL_LOW:
        event_type = "thunder_audio"
        intensity = int(min(255, avg_audio * 1.5))
    elif max_rain > RAIN_INTENSITY_THRESHOLD:
        event_type = "rain"
        intensity = int(max_rain)
    elif avg_audio > MIN_INTENSITY:
        event_type = "audio"
        intensity = int(avg_audio)
    else:
        event_type = "none"
        intensity = 0

    if intensity < MIN_INTENSITY:
        intensity = 0
        if event_type != "none":
            event_type = "none"
    return event_type, int(intensity)


def _audio_intensity(rms: float) -> float:
    return float(np.interp(rms, [0.0, AUDIO_RMS_MAX], [0.0, 255.0]))


def _rms_windows(audio: np.ndarray, sr: int, window_ms: int = WINDOW_SIZE_MS) -> np.ndarray:
    hop = max(1, int(round(sr * window_ms / 1000.0)))
    n = int(np.ceil(len(audio) / hop))
    out = np.zeros(n, dtype=np.float64)
    for i in range(n):
        chunk = audio[i * hop : min(len(audio), (i + 1) * hop)]
        if chunk.size:
            out[i] = float(np.sqrt(np.mean(np.square(chunk)) + 1e-12))
    return out


def _analyze_audio_only(audio: np.ndarray, sr: int) -> list[dict[str, Any]]:
    rms = _rms_windows(audio, sr)
    events: list[dict[str, Any]] = []
    for i, val in enumerate(rms):
        avg_audio = _audio_intensity(float(val))
        event_type, intensity = mix_window(
            max_lightning=0.0, max_rain=0.0, avg_audio=avg_audio
        )
        events.append(
            {
                "start_ms": int(i * WINDOW_SIZE_MS),
                "type": event_type,
                "intensity": intensity,
                "max_lightning": 0.0,
                "max_rain": 0.0,
                "avg_audio": float(avg_audio),
            }
        )
    return events


def _analyze_video(
    video_path: Path,
    audio: np.ndarray,
    sr: int,
) -> list[dict[str, Any]] | None:
    try:
        import cv2
    except ImportError:
        return None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if fps <= 1e-6 or total_frames <= 0 or frame_width <= 0:
        cap.release()
        return None

    frame_length = max(1, int(round(sr / fps)))
    audio_rms = librosa.feature.rms(
        y=audio.astype(np.float32),
        frame_length=frame_length,
        hop_length=frame_length,
    )[0]

    frames_per_window = max(1, int(fps * (WINDOW_SIZE_MS / 1000.0)))
    last_brightness = 0.0
    prev_down = None
    frame_metrics: list[dict[str, float]] = []

    for frame_idx in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        down = cv2.resize(gray, (max(1, frame_width // 4), max(1, frame_height // 4)))
        if prev_down is None:
            prev_down = down
        diff = cv2.absdiff(down, prev_down)
        prev_down = down
        _, diff_th = cv2.threshold(diff, RAIN_DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)
        rain_motion = int(np.count_nonzero(diff_th))

        avg_brightness = float(np.mean(gray))
        brightness_spike = max(0.0, avg_brightness - last_brightness)
        last_brightness = avg_brightness
        lightning_intensity = float(
            np.interp(
                brightness_spike,
                [0.0, LIGHTNING_BRIGHTNESS_SCALE_MAX],
                [0.0, 255.0],
            )
        )
        rain_ratio = rain_motion / float(down.shape[0] * down.shape[1])
        rain_intensity = float(
            np.interp(rain_ratio, [0.0, RAIN_RATIO_MAX], [0.0, RAIN_INTENSITY_MAX])
        )
        audio_val = float(audio_rms[frame_idx]) if frame_idx < len(audio_rms) else 0.0
        frame_metrics.append(
            {
                "lightning_intensity": lightning_intensity,
                "rain_intensity": rain_intensity,
                "audio_intensity": _audio_intensity(audio_val),
                "brightness": avg_brightness,
            }
        )

    cap.release()
    if not frame_metrics:
        return None

    num_windows = int(np.ceil(len(frame_metrics) / frames_per_window))
    events: list[dict[str, Any]] = []
    for w in range(num_windows):
        window_frames = frame_metrics[w * frames_per_window : (w + 1) * frames_per_window]
        if not window_frames:
            events.append(
                {
                    "start_ms": int(w * WINDOW_SIZE_MS),
                    "type": "none",
                    "intensity": 0,
                    "max_lightning": 0.0,
                    "max_rain": 0.0,
                    "avg_audio": 0.0,
                }
            )
            continue
        max_lightning = max(f["lightning_intensity"] for f in window_frames)
        max_rain = max(f["rain_intensity"] for f in window_frames)
        avg_audio = float(np.mean([f["audio_intensity"] for f in window_frames]))
        event_type, intensity = mix_window(
            max_lightning=max_lightning, max_rain=max_rain, avg_audio=avg_audio
        )
        events.append(
            {
                "start_ms": int(w * WINDOW_SIZE_MS),
                "type": event_type,
                "intensity": intensity,
                "max_lightning": float(max_lightning),
                "max_rain": float(max_rain),
                "avg_audio": avg_audio,
            }
        )
    return events


def windows_to_track(events: list[dict[str, Any]]) -> dict[str, int]:
    return {str(int(e["start_ms"])): int(e["intensity"]) for e in events}


def windows_to_wav(
    events: list[dict[str, Any]],
    duration_sec: float,
    *,
    sr: int = VIB_SR,
    carrier_hz: float = CARRIER_HZ,
) -> np.ndarray:
    n = max(1, int(round(duration_sec * sr)))
    t = np.arange(n, dtype=np.float64) / sr
    carrier = np.sin(2.0 * np.pi * carrier_hz * t)
    amp = np.zeros(n, dtype=np.float64)
    hop = max(1, int(round(sr * WINDOW_SIZE_MS / 1000.0)))
    for i, ev in enumerate(events):
        s0 = i * hop
        s1 = min(n, (i + 1) * hop)
        if s1 <= s0:
            break
        amp[s0:s1] = float(ev["intensity"]) / 255.0
    if events:
        last_end = min(n, len(events) * hop)
        if last_end < n:
            amp[last_end:] = float(events[-1]["intensity"]) / 255.0
    return (carrier * amp).astype(np.float32)


def analyze(
    source_wav: str | Path,
    *,
    video_path: str | Path | None = None,
) -> list[dict[str, Any]]:
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    events = None
    if video_path is not None:
        events = _analyze_video(Path(video_path), audio, int(sr))
    if events is None:
        events = _analyze_audio_only(audio, int(sr))
    return events


def process_file(
    source_wav: str | Path,
    output_wav: str | Path,
    *,
    video_path: str | Path | None = None,
    json_path: str | Path | None = None,
) -> dict[str, Any]:
    \"\"\"Write the rule-based haptic WAV (and optional JSON map).\"\"\"
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    duration_sec = len(audio) / float(sr)

    events = analyze(source_wav, video_path=video_path)
    haptic = windows_to_wav(events, duration_sec)
    out_wav = Path(output_wav)
    out_wav.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_wav, haptic, VIB_SR, subtype="PCM_16")

    payload = {
        "window_size_ms": WINDOW_SIZE_MS,
        "track": windows_to_track(events),
        "events": events,
        "source": "rule_based",
    }
    if json_path is not None:
        out_json = Path(json_path)
        out_json.parent.mkdir(parents=True, exist_ok=True)
        out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        payload["json_path"] = str(out_json)
    payload["wav_path"] = str(out_wav)
    return payload
""",
    "audio_io.py": """\"\"\"Audio extraction, loading, and export (Sound2Hap-compatible rates).\"\"\"

from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

INPUT_SR = 44_100
VIB_SR = 8_000


def _require_ffmpeg() -> str:
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            "ffmpeg not found. Install it first "
            "(Colab: !apt-get -qq install ffmpeg)."
        )
    return ffmpeg


def extract_audio_from_video(
    video_path: str | Path,
    output_path: str | Path,
    sr: int = INPUT_SR,
) -> Path:
    \"\"\"Extract mono 16-bit PCM WAV at 44.1 kHz from a video file.\"\"\"
    video_path = Path(video_path)
    output_path = Path(output_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    ffmpeg = _require_ffmpeg()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        ffmpeg,
        "-y",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(sr),
        "-sample_fmt",
        "s16",
        str(output_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\\n{result.stderr}")
    return output_path


def prepare_source_wav(
    audio_path: str | Path,
    output_path: str | Path,
    sr: int = INPUT_SR,
) -> Path:
    \"\"\"Convert/load audio to mono 16-bit PCM WAV at 44.1 kHz.\"\"\"
    audio_path = Path(audio_path)
    output_path = Path(output_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    audio, _ = librosa.load(audio_path, sr=sr, mono=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_path, np.clip(audio, -1.0, 1.0), sr, subtype="PCM_16")
    return output_path


def save_haptic(path: str | Path, audio: np.ndarray, sr: int = VIB_SR) -> Path:
    \"\"\"Write a mono haptic track as 16-bit PCM WAV (default 8 kHz).\"\"\"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, np.clip(audio, -1.0, 1.0), sr, subtype="PCM_16")
    return path
""",
    "context/__init__.py": """\"\"\"Frozen multimodal context detection for haptic gating.\"\"\"

from haptic_gt.context.detector import EventResult, detect_events
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.manual_events import events_from_manual

__all__ = ["DetectedEvent", "EventResult", "detect_events", "events_from_manual"]
""",
    "context/context_detectors.py": """\"\"\"Frozen transformer-based context detectors for sudden symbolic events.\"\"\"

from __future__ import annotations

from dataclasses import dataclass

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category


@dataclass
class SymbolicToken:
    time_sec: float
    label: str
    confidence: float
    modality: str  # "audio" | "video"
    category: str | None = None


IMPULSIVE_AUDIO_KEYWORDS = (
    "thunder",
    "explosion",
    "gunshot",
    "gunfire",
    "machine gun",
    "fireworks",
    "bang",
    "boom",
    "artillery",
    "fusillade",
    "cap gun",
)

IMPULSIVE_VIDEO_KEYWORDS = (
    "shooting",
    "explod",
    "fire",
    "smash",
    "hit",
    "crash",
)

SUSTAINED_AUDIO_KEYWORDS = (
    "vehicle",
    "engine",
    "truck",
    "motor vehicle",
    "idling",
    "tank",
    "rain",
    "wind",
    "storm",
)

SUSTAINED_VIDEO_KEYWORDS = (
    "driving",
    "motorcycl",
    "riding",
)


def _is_impulsive_label(label: str, keywords: tuple[str, ...]) -> bool:
    low = label.lower()
    return any(k in low for k in keywords)


def symbolic_tokens_from_scores(
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
    *,
    threshold: float | None = None,
) -> list[SymbolicToken]:
    \"\"\"
    Emit symbolic tokens from a single encoder pass.

    Impulsive labels use context_detector_threshold; sustained labels (vehicle,
    weather-non-thunder) use the lower sustained_encoder_threshold so they are
    not silently filtered out.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    impulsive_thresh = threshold if threshold is not None else taxonomy.context_detector_threshold
    sustained_thresh = taxonomy.sustained_encoder_threshold
    tokens: list[SymbolicToken] = []

    for enc in encoder_scores:
        if enc.source == "audio":
            is_impulsive = _is_impulsive_label(enc.label, IMPULSIVE_AUDIO_KEYWORDS)
            is_sustained = _is_impulsive_label(enc.label, SUSTAINED_AUDIO_KEYWORDS)
            modality = "audio"
        else:
            is_impulsive = _is_impulsive_label(enc.label, IMPULSIVE_VIDEO_KEYWORDS)
            is_sustained = _is_impulsive_label(enc.label, SUSTAINED_VIDEO_KEYWORDS)
            modality = "video"

        if not is_impulsive and not is_sustained:
            continue

        effective_thresh = impulsive_thresh if is_impulsive else sustained_thresh
        if enc.score < effective_thresh:
            continue

        cat = match_label_to_category(taxonomy, enc.label, modality)
        if cat is None:
            continue

        tokens.append(
            SymbolicToken(
                time_sec=enc.time_sec,
                label=enc.label,
                confidence=enc.score,
                modality=modality,
                category=cat,
            )
        )

    return tokens


def detect_symbolic_tokens(
    video_path,
    audio_16k,
    video_frames,
    taxonomy: Taxonomy | None = None,
    *,
    window_sec: float = 0.5,
    hop_sec: float = 0.5,
    duration_sec: float | None = None,
    threshold: float | None = None,
    encoder_scores: list[EncoderScore] | None = None,
) -> list[SymbolicToken]:
    \"\"\"
    Emit symbolic tokens for sudden AV events.

    When encoder_scores is provided, filters that list (no extra model calls).
    Otherwise falls back to legacy full-timeline classification (deprecated).
    \"\"\"
    if encoder_scores is not None:
        return symbolic_tokens_from_scores(
            encoder_scores,
            taxonomy=taxonomy,
            threshold=threshold,
        )

    if duration_sec is None:
        raise ValueError("duration_sec is required when encoder_scores is not provided")

    from pathlib import Path

    from haptic_gt.context.encoders import run_encoder_pass

    taxonomy = taxonomy or load_taxonomy()
    scores = run_encoder_pass(
        Path(video_path),
        audio_16k,
        video_frames,
        window_sec=window_sec,
        hop_sec=hop_sec,
        duration_sec=duration_sec,
        full_scan=True,
    )
    return symbolic_tokens_from_scores(scores, taxonomy=taxonomy, threshold=threshold)
""",
    "context/debug_events.py": """\"\"\"Debug helpers: event timing table + per-event audio/metadata extraction.\"\"\"

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR


@dataclass
class EventDebugRow:
    event_id: str
    category: str
    label: str
    start_sec: float
    peak_sec: float
    end_sec: float
    duration_sec: float
    confidence: float
    audio_score: float | None
    video_score: float | None
    sources: list[str]
    included_in_gate: bool
    clip_wav: str | None = None
    # Filled by user during diagnosis (optional)
    actual_peak_sec: float | None = None
    timing_note: str | None = None


def load_events_payload(events_json: str | Path) -> dict:
    path = Path(events_json)
    return json.loads(path.read_text(encoding="utf-8"))


def events_to_debug_rows(payload: dict) -> list[EventDebugRow]:
    rows: list[EventDebugRow] = []
    for i, ev in enumerate(payload.get("events", []), start=1):
        event_id = str(ev.get("event_id") or f"event_{i:03d}")
        start = float(ev.get("start_sec", 0.0))
        peak = float(ev.get("peak_sec", start))
        end = float(ev.get("end_sec", peak))
        rows.append(
            EventDebugRow(
                event_id=event_id,
                category=str(ev.get("category", "")),
                label=str(ev.get("label", "")),
                start_sec=start,
                peak_sec=peak,
                end_sec=end,
                duration_sec=max(0.0, end - start),
                confidence=float(ev.get("confidence", 0.0)),
                audio_score=ev.get("audio_score"),
                video_score=ev.get("video_score"),
                sources=list(ev.get("sources") or []),
                included_in_gate=bool(ev.get("included_in_gate", False)),
            )
        )
    return rows


def format_events_table(rows: list[EventDebugRow]) -> str:
    \"\"\"Plain-text timing table for Colab / terminal diagnosis.\"\"\"
    if not rows:
        return "(no events)"

    header = (
        f"{'id':<12} {'cat':<14} {'start':>8} {'peak':>8} {'end':>8} "
        f"{'dur':>6} {'conf':>6} {'audio':>6} {'vid':>6} {'gate':>5} label"
    )
    lines = [header, "-" * len(header)]
    for r in rows:
        audio = f"{r.audio_score:.3f}" if r.audio_score is not None else "-"
        video = f"{r.video_score:.3f}" if r.video_score is not None else "-"
        lines.append(
            f"{r.event_id:<12} {r.category:<14} "
            f"{r.start_sec:8.3f} {r.peak_sec:8.3f} {r.end_sec:8.3f} "
            f"{r.duration_sec:6.3f} {r.confidence:6.3f} {audio:>6} {video:>6} "
            f"{'Y' if r.included_in_gate else 'N':>5} {r.label}"
        )
    return "\\n".join(lines)


def extract_event_debug_clips(
    source_wav: str | Path,
    events_json: str | Path,
    output_dir: str | Path,
    *,
    sample_rate: int = INPUT_SR,
    pad_sec: float = 0.25,
) -> list[EventDebugRow]:
    \"\"\"
    Write one WAV + JSON sidecar per event for accurate listening/diagnosis.

    Layout:
      debug_events/
        event_001_vehicle/
          clip.wav
          meta.json
        ...
        timing_table.txt
        events_debug.json
    \"\"\"
    source_wav = Path(source_wav)
    events_json = Path(events_json)
    output_dir = Path(output_dir)
    debug_root = output_dir / "debug_events"
    debug_root.mkdir(parents=True, exist_ok=True)

    payload = load_events_payload(events_json)
    rows = events_to_debug_rows(payload)

    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate
    duration_sec = len(audio) / sr

    exported: list[EventDebugRow] = []
    for row in rows:
        folder = debug_root / f"{row.event_id}_{row.category}"
        folder.mkdir(parents=True, exist_ok=True)
        clip_path = folder / "clip.wav"

        s0 = max(0.0, row.start_sec - pad_sec)
        s1 = min(duration_sec, row.end_sec + pad_sec)
        i0 = max(0, int(s0 * sr))
        i1 = min(len(audio), int(s1 * sr))
        clip = audio[i0:i1]
        sf.write(clip_path, clip, sr, subtype="PCM_16")

        meta = asdict(row)
        meta.update(
            {
                "clip_wav": str(clip_path.relative_to(output_dir)),
                "clip_start_sec": round(s0, 3),
                "clip_end_sec": round(s1, 3),
                "pad_sec": pad_sec,
                "sample_rate": sr,
                # User fills these after listening / watching video:
                "actual_peak_sec": None,
                "timing_error_sec": None,
                "note": None,
            }
        )
        (folder / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

        row.clip_wav = meta["clip_wav"]
        exported.append(row)

    table = format_events_table(exported)
    (debug_root / "timing_table.txt").write_text(table + "\\n", encoding="utf-8")

    summary = {
        "source_audio": str(source_wav.name),
        "gate_categories_used": payload.get("gate_categories_used", []),
        "instructions": (
            "Compare peak_sec to the real event in the video/audio. "
            "Fill actual_peak_sec and timing_error_sec (= detected_peak - actual_peak) "
            "in each meta.json or reply with a table."
        ),
        "events": [asdict(r) for r in exported],
    }
    (debug_root / "events_debug.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )
    return exported


def print_event_timing_report(
    events_json: str | Path,
    *,
    source_wav: str | Path | None = None,
    output_dir: str | Path | None = None,
    extract_clips: bool = True,
) -> list[EventDebugRow]:
    \"\"\"Print timing table and optionally extract per-event debug clips.\"\"\"
    payload = load_events_payload(events_json)
    rows = events_to_debug_rows(payload)
    print("=== Detected event timing table ===")
    print(format_events_table(rows))
    print()
    print("Fill actual peaks like:")
    print("  event_001 actual_peak_sec=...  (or 'ok' if correct)")
    print("  event_004 actual_peak_sec=15.20")
    print()

    if extract_clips and source_wav is not None and output_dir is not None:
        exported = extract_event_debug_clips(source_wav, events_json, output_dir)
        print(f"Wrote debug clips under: {Path(output_dir) / 'debug_events'}")
        return exported
    return rows
""",
    "context/detector.py": """\"\"\"Orchestrate Phase 1 + Phase 2 context detection.\"\"\"

from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

from haptic_gt.context.context_detectors import symbolic_tokens_from_scores
from haptic_gt.context.encoders import run_encoder_pass
from haptic_gt.context.frozen_fusion import DetectedEvent, dedupe_events_by_peak, fuse_events
from haptic_gt.context.mask import (
    apply_gate,
    event_included_in_gate,
    events_for_haptic_gate,
    resolve_gate_categories,
)
from haptic_gt.context.onset_refine import refine_event_timing
from haptic_gt.context.proposals import propose_all_windows
from haptic_gt.context.impulsive_nms import (
    measure_impulsive_attacks,
    snap_impulsive_peaks_to_attacks,
    suppress_impulsive_overlaps,
)
from haptic_gt.context.impulsive_promote import promote_impulsive_transients
from haptic_gt.context.rumble_filter import filter_sustained_rumble_bursts
from haptic_gt.context.sed_events import (
    events_from_frame_posteriors,
    split_impulsive_by_posterior_peaks,
)
from haptic_gt.context.sed_frames import (
    compute_frame_posteriors,
    posteriors_to_encoder_scores,
)
from haptic_gt.context.sustained_merge import merge_sustained_events
from haptic_gt.context.taxonomy import load_taxonomy
from haptic_gt.context.tokenization import tokenize_video_audio
from haptic_gt.context.visual_flash import align_impulsive_events_to_flashes

EVENTS_JSON_NAME = "events.json"
GATED_AUDIO_NAME = "gated_audio.wav"


@dataclass
class EventResult:
    events: list[DetectedEvent] = field(default_factory=list)
    no_events_detected: bool = True
    no_haptic_events: bool = True
    timeline_hz: int = 100
    gate_categories_used: list[str] = field(default_factory=list)
    gated_wav: Path | None = None
    events_json: Path | None = None
    haptic_outputs: dict[str, str] = field(default_factory=dict)
    detector_info: dict = field(default_factory=dict)
    sustained_gate: dict = field(default_factory=dict)

    def to_dict(self, *, output_dir: Path | None = None) -> dict:
        taxonomy = load_taxonomy()
        gate_cats = self.gate_categories_used or resolve_gate_categories(taxonomy)

        def _rel(path: str | None) -> str | None:
            if path is None or output_dir is None:
                return path
            try:
                return str(Path(path).relative_to(output_dir))
            except ValueError:
                return path

        event_rows = []
        for i, e in enumerate(self.events, start=1):
            row = {
                "event_id": f"event_{i:03d}",
                "category": e.category,
                "label": e.label,
                "start_sec": round(e.start_sec, 3),
                "peak_sec": round(e.peak_sec, 3),
                "end_sec": round(e.end_sec, 3),
                "confidence": round(e.confidence, 4),
                "context_token": e.context_token,
                "audio_score": e.audio_score,
                "video_score": e.video_score,
                "sources": e.sources,
                "included_in_gate": event_included_in_gate(
                    e, taxonomy, gate_categories=gate_cats
                ),
            }
            # Frame posteriors report the decode threshold for every promoted
            # shot, so strength has to be read off the attack instead.
            if e.attack_rel_max is not None:
                row["attack_rel_max"] = e.attack_rel_max
            if e.attack_prominence is not None:
                row["attack_prominence"] = e.attack_prominence
            event_rows.append(row)

        haptic_out = {
            key: _rel(val) for key, val in self.haptic_outputs.items() if val
        }

        return {
            "detector": self.detector_info,
            "sustained_gate": self.sustained_gate,
            "no_events_detected": self.no_events_detected,
            "no_haptic_events": self.no_haptic_events,
            "gate_categories_used": gate_cats,
            "timeline_hz": self.timeline_hz,
            "haptic_outputs": haptic_out,
            "events": event_rows,
        }


def detect_events(
    video_path: str | Path,
    source_wav: str | Path,
    output_dir: str | Path | None = None,
    *,
    taxonomy_path: str | Path | None = None,
    window_sec: float | None = None,
    hop_sec: float = 0.25,
    write_gated: bool = True,
    gate_categories: list[str] | None = None,
    full_scan: bool = False,
) -> EventResult:
    \"\"\"
    Run frozen context detection.

    Default (``sed_enabled``): frame-level SED posteriors → hysteresis decoding →
    spectral-flux onset refinement → events.json → optional gated_audio.wav.

    Legacy path (``full_scan`` or ``sed_enabled: false``): sparse onset proposals →
    window tagging → frozen fusion. Kept for comparison; its onsets are only as
    precise as the 1 s classifier window.
    \"\"\"
    video_path = Path(video_path)
    source_wav = Path(source_wav)
    taxonomy = load_taxonomy(taxonomy_path)
    window_sec = window_sec if window_sec is not None else taxonomy.proposal_window_sec
    gate_cats = resolve_gate_categories(taxonomy, gate_categories)

    tokens_data = tokenize_video_audio(video_path, source_wav, timeline_hz=taxonomy.timeline_hz)

    if taxonomy.sed_enabled and not full_scan:
        return _detect_via_frame_sed(
            tokens_data=tokens_data,
            source_wav=source_wav,
            video_path=video_path,
            taxonomy=taxonomy,
            gate_cats=gate_cats,
            output_dir=output_dir,
            write_gated=write_gated,
        )

    if full_scan:
        proposal_windows = None
        encoder_scores = run_encoder_pass(
            video_path,
            tokens_data.audio_16k,
            tokens_data.video_frames,
            window_sec=max(0.75, window_sec),
            hop_sec=hop_sec,
            duration_sec=tokens_data.duration_sec,
            full_scan=True,
        )
    else:
        proposal_windows = propose_all_windows(source_wav, taxonomy)
        encoder_scores = run_encoder_pass(
            video_path,
            tokens_data.audio_16k,
            tokens_data.video_frames,
            window_sec=window_sec,
            hop_sec=hop_sec,
            duration_sec=tokens_data.duration_sec,
            proposal_windows=proposal_windows,
            timeline_hz=taxonomy.timeline_hz,
        )

    tokens = symbolic_tokens_from_scores(encoder_scores, taxonomy=taxonomy)

    events = fuse_events(tokens, encoder_scores, taxonomy=taxonomy)
    events = refine_event_timing(events, source_wav, taxonomy)
    # Onset snap can collapse late rumble + early muzzle onto the same peak
    events = dedupe_events_by_peak(events)
    # Cannon in engine bed: AST often says vehicle; flux + explosion score recovers shots
    events = promote_impulsive_transients(
        events, source_wav, encoder_scores, taxonomy
    )
    # SED/refine often sit on a decay bump; snap back to the muzzle in-window
    events = snap_impulsive_peaks_to_attacks(events, source_wav, taxonomy)
    events = align_impulsive_events_to_flashes(events, video_path, taxonomy)
    # Picture flash can lead the boom by ~0.2 s; snap again onto the sharp attack
    events = snap_impulsive_peaks_to_attacks(events, source_wav, taxonomy)
    # Peaks are on attacks/flashes now; only the spans need rebuilding around them
    events = refine_event_timing(
        events, source_wav, taxonomy, relocate_impulsive_peaks=False
    )
    events = dedupe_events_by_peak(events)
    # One accent per blast: a shot plus a bump in its own decay is one bang
    events = suppress_impulsive_overlaps(events, source_wav, taxonomy)
    # Collapse fragmented vehicle chips into longer rumble spans
    events = merge_sustained_events(events, taxonomy)
    # Keep loud rumble islands; do not re-merge (that glues bursts across quiet gaps)
    gate_report: dict = {}
    events = filter_sustained_rumble_bursts(
        events, source_wav, taxonomy, report=gate_report
    )
    events = dedupe_events_by_peak(events)

    return _finalize(
        events,
        source_wav=source_wav,
        taxonomy=taxonomy,
        gate_cats=gate_cats,
        output_dir=output_dir,
        write_gated=write_gated,
        detector_info={"mode": "window_tagging", "use_video": taxonomy.use_video},
        sustained_gate=gate_report,
    )


def _finalize(
    events: list[DetectedEvent],
    *,
    source_wav: Path,
    taxonomy,
    gate_cats: list[str],
    output_dir: str | Path | None,
    write_gated: bool,
    detector_info: dict | None = None,
    sustained_gate: dict | None = None,
) -> EventResult:
    \"\"\"Build EventResult and write events.json / gated audio.\"\"\"
    events = measure_impulsive_attacks(events, source_wav, taxonomy)
    gate_events = events_for_haptic_gate(events, taxonomy, gate_categories=gate_cats)
    result = EventResult(
        events=events,
        no_events_detected=len(events) == 0,
        no_haptic_events=len(gate_events) == 0,
        timeline_hz=taxonomy.timeline_hz,
        gate_categories_used=gate_cats,
        detector_info=detector_info or {"mode": "window_tagging", "use_video": taxonomy.use_video},
        sustained_gate=sustained_gate or {},
    )

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        events_json = output_dir / EVENTS_JSON_NAME
        events_json.write_text(
            json.dumps(result.to_dict(output_dir=output_dir), indent=2),
            encoding="utf-8",
        )
        result.events_json = events_json

        if write_gated and gate_events:
            gated = output_dir / GATED_AUDIO_NAME
            apply_gate(
                source_wav,
                gated,
                events,
                taxonomy=taxonomy,
                gate_categories=gate_cats,
            )
            result.gated_wav = gated
            result.haptic_outputs["gated_audio"] = str(gated)

    return result


def _detect_via_frame_sed(
    *,
    tokens_data,
    source_wav: Path,
    video_path: Path,
    taxonomy,
    gate_cats: list[str],
    output_dir: str | Path | None,
    write_gated: bool,
) -> EventResult:
    \"\"\"
    Frame-level SED path (default).

    Per-category posteriors on a 100 ms grid (10 ms with PANNs) are decoded with
    median filtering + hysteresis, so onsets come from the model's time axis
    instead of one score per 1 s window. Spectral-flux refinement then sharpens
    impulsive onsets to sample accuracy.

    Video is not fused here on purpose — see ``use_video`` in taxonomy.yaml.
    \"\"\"
    frames = compute_frame_posteriors(tokens_data.audio_16k, taxonomy)
    encoder_scores = posteriors_to_encoder_scores(frames, taxonomy)

    events = events_from_frame_posteriors(frames, taxonomy)
    events = split_impulsive_by_posterior_peaks(events, frames, taxonomy)
    events = refine_event_timing(events, source_wav, taxonomy)
    events = dedupe_events_by_peak(events)
    events = promote_impulsive_transients(events, source_wav, encoder_scores, taxonomy)
    # SED/refine often sit on a decay bump; snap back to the muzzle in-window
    events = snap_impulsive_peaks_to_attacks(events, source_wav, taxonomy)
    events = align_impulsive_events_to_flashes(events, video_path, taxonomy)
    # Picture flash can lead the boom by ~0.2 s; snap again onto the sharp attack
    events = snap_impulsive_peaks_to_attacks(events, source_wav, taxonomy)
    # Peaks are on attacks/flashes now; only the spans need rebuilding around them
    events = refine_event_timing(
        events, source_wav, taxonomy, relocate_impulsive_peaks=False
    )
    events = dedupe_events_by_peak(events)
    # One accent per blast: a shot plus a bump in its own decay is one bang
    events = suppress_impulsive_overlaps(events, source_wav, taxonomy)
    events = merge_sustained_events(events, taxonomy)
    gate_report: dict = {}
    events = filter_sustained_rumble_bursts(
        events, source_wav, taxonomy, report=gate_report
    )
    events = dedupe_events_by_peak(events)

    return _finalize(
        events,
        source_wav=source_wav,
        taxonomy=taxonomy,
        gate_cats=gate_cats,
        output_dir=output_dir,
        write_gated=write_gated,
        sustained_gate=gate_report,
        detector_info={
            "mode": "frame_sed",
            "backend": frames.backend,
            "frame_hop_sec": round(frames.hop_sec, 4),
            "window_sec": taxonomy.sed_window_sec,
            "onset_high": taxonomy.sed_onset_high,
            "onset_low": taxonomy.sed_onset_low,
            "use_video": taxonomy.use_video,
        },
    )
""",
    "context/encoders.py": """\"\"\"Frozen AST and ViViT encoders (requires_grad=False).\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

from haptic_gt.context.proposals import ProposalWindow

AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
VIVIT_MODEL_ID = "google/vivit-b-16x2-kinetics400"

_audio_pipe: Any | None = None
_video_pipe: Any | None = None


def _get_device() -> int:
    try:
        import torch

        return 0 if torch.cuda.is_available() else -1
    except Exception:
        return -1


def get_audio_pipeline():
    global _audio_pipe
    if _audio_pipe is None:
        from transformers import pipeline

        _audio_pipe = pipeline(
            "audio-classification",
            model=AST_MODEL_ID,
            device=_get_device(),
        )
    return _audio_pipe


def get_video_pipeline():
    global _video_pipe
    if _video_pipe is None:
        from transformers import pipeline

        _video_pipe = pipeline(
            "video-classification",
            model=VIVIT_MODEL_ID,
            device=_get_device(),
        )
    return _video_pipe


@dataclass
class EncoderScore:
    time_sec: float
    label: str
    score: float
    source: str  # "audio" | "video"


def _top_predictions(raw: list[dict], top_k: int = 5) -> list[tuple[str, float]]:
    out: list[tuple[str, float]] = []
    for item in raw[:top_k]:
        label = str(item.get("label", ""))
        if label.startswith("LABEL_"):
            label = label.replace("LABEL_", "")
        out.append((label, float(item.get("score", 0.0))))
    return out


def classify_audio_window(
    audio_16k: np.ndarray,
    start_sec: float,
    end_sec: float,
    *,
    top_k: int = 5,
) -> list[EncoderScore]:
    \"\"\"Run frozen AST on an audio slice.\"\"\"
    sr = 16_000
    s0 = max(0, int(start_sec * sr))
    s1 = min(len(audio_16k), int(end_sec * sr))
    if s1 - s0 < sr // 10:
        return []

    clip = audio_16k[s0:s1]
    pipe = get_audio_pipeline()
    preds = pipe(clip, top_k=top_k)
    mid = 0.5 * (start_sec + end_sec)
    return [
        EncoderScore(time_sec=mid, label=label, score=score, source="audio")
        for label, score in _top_predictions(preds, top_k)
    ]


def classify_video_clip_file(
    video_path: Path,
    center_sec: float,
    *,
    top_k: int = 5,
) -> list[EncoderScore]:
    \"\"\"
    Run frozen ViViT on a short clip from the video file.

    The HF video-classification pipeline samples frames internally from the
    full file; for temporal windows we extract a sub-clip with ffmpeg when
  possible, otherwise classify the whole file (coarse fallback).
    \"\"\"
    pipe = get_video_pipeline()
    try:
        preds = pipe(str(video_path), top_k=top_k)
    except Exception:
        return []
    return [
        EncoderScore(time_sec=center_sec, label=label, score=score, source="video")
        for label, score in _top_predictions(preds, top_k)
    ]


def classify_video_frames(
    frames: np.ndarray,
    center_sec: float,
    *,
    top_k: int = 5,
) -> list[EncoderScore]:
    \"\"\"Run ViViT on a pre-extracted frame tensor [T, C, H, W] in [0,1].

    NOTE: the video branch is disabled by default (``use_video: false``). ViViT is
    trained on Kinetics-400 human actions, whose labels ("driving car",
    "exploding firecrackers") do not correspond to the acoustic events we gate on,
    so ``video_score`` was null on every event and fusion never used it. To bring
    video back, swap in an audio-visual model (VGGSound-style) or a learned head
    on VideoMAE/CLIP features, then set ``use_video: true``.
    \"\"\"
    if frames.size == 0:
        return []
    try:
        import torch
    except ImportError:
        return []

    pipe = get_video_pipeline()
    # Pipeline expects list of PIL or tensor; pass uint8 frame list
    tensor = torch.from_numpy(frames).float()
    if tensor.ndim != 4:
        return []
    # Sample up to 32 frames uniformly
    n = tensor.shape[0]
    if n > 32:
        idx = np.linspace(0, n - 1, 32).astype(int)
        tensor = tensor[idx]
    # Convert to uint8 HWC list for pipeline compatibility
    frames_u8 = (tensor.permute(0, 2, 3, 1).numpy() * 255.0).clip(0, 255).astype(np.uint8)
    try:
        from PIL import Image

        pil_frames = [Image.fromarray(f) for f in frames_u8]
        preds = pipe(pil_frames, top_k=top_k)
    except Exception:
        return []

    return [
        EncoderScore(time_sec=center_sec, label=label, score=score, source="video")
        for label, score in _top_predictions(preds, top_k)
    ]


def run_encoder_on_windows(
    windows: list[ProposalWindow],
    video_path: Path,
    audio_16k: np.ndarray,
    video_frames: np.ndarray,
    *,
    timeline_hz: int = 100,
    window_sec: float = 1.0,
    top_k: int = 8,
) -> list[EncoderScore]:
    \"\"\"Classify frozen AST + ViViT only at proposal-centered windows.\"\"\"
    if not windows:
        return []

    duration_sec = len(audio_16k) / 16_000
    half = window_sec / 2.0
    scores: list[EncoderScore] = []

    from haptic_gt.context.taxonomy import load_taxonomy

    use_video = load_taxonomy().use_video

    for win in windows:
        t0 = max(0.0, win.center_sec - half)
        t1 = min(duration_sec, win.center_sec + half)
        scores.extend(classify_audio_window(audio_16k, t0, t1, top_k=top_k))

        if not use_video:
            continue

        bin_start = int(t0 * timeline_hz)
        bin_end = int(t1 * timeline_hz)
        bin_end = min(bin_end, len(video_frames))
        if bin_end > bin_start:
            clip = video_frames[bin_start:bin_end]
            scores.extend(
                classify_video_frames(clip, center_sec=win.center_sec, top_k=top_k)
            )
        else:
            scores.extend(
                classify_video_clip_file(video_path, center_sec=win.center_sec, top_k=top_k)
            )

    return scores


def run_encoder_pass(
    video_path: Path,
    audio_16k: np.ndarray,
    video_frames: np.ndarray,
    *,
    window_sec: float = 1.0,
    hop_sec: float = 0.5,
    duration_sec: float,
    top_k: int = 8,
    full_scan: bool = False,
    proposal_windows: list[ProposalWindow] | None = None,
    timeline_hz: int = 100,
) -> list[EncoderScore]:
    \"\"\"
    Classify with frozen AST + ViViT.

    Default: proposal windows only. Set full_scan=True for legacy timeline sweep.
    \"\"\"
    if not full_scan and proposal_windows is not None:
        return run_encoder_on_windows(
            proposal_windows,
            video_path,
            audio_16k,
            video_frames,
            timeline_hz=timeline_hz,
            window_sec=window_sec,
            top_k=top_k,
        )

    from haptic_gt.context.taxonomy import load_taxonomy

    use_video = load_taxonomy().use_video

    scores: list[EncoderScore] = []
    t = 0.0
    while t < duration_sec:
        end = min(duration_sec, t + window_sec)
        scores.extend(classify_audio_window(audio_16k, t, end, top_k=top_k))

        if not use_video:
            t += hop_sec
            continue

        bin_start = int(t * 100)
        bin_end = int(end * 100)
        bin_end = min(bin_end, len(video_frames))
        if bin_end > bin_start:
            clip = video_frames[bin_start:bin_end]
            scores.extend(
                classify_video_frames(clip, center_sec=0.5 * (t + end), top_k=top_k)
            )
        else:
            scores.extend(
                classify_video_clip_file(video_path, center_sec=0.5 * (t + end), top_k=top_k)
            )
        t += hop_sec
    return scores
""",
    "context/event_aggregation.py": """\"\"\"Merge symbolic tokens into event primitives with start/peak/end.\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.context_detectors import SymbolicToken
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category


@dataclass
class EventPrimitive:
    category: str | None
    label: str
    start_sec: float
    peak_sec: float
    end_sec: float
    confidence: float
    sources: list[str] = field(default_factory=list)
    context_token: bool = False
    audio_score: float | None = None
    video_score: float | None = None


def _merge_tokens(
    items: list[SymbolicToken],
    taxonomy: Taxonomy,
) -> list[EventPrimitive]:
    \"\"\"Peak-split impulsive tokens; gap-merge sustained ones.\"\"\"
    if not items:
        return []

    by_category: dict[str, list[SymbolicToken]] = {}
    for tok in items:
        cat = tok.category or "unknown"
        by_category.setdefault(cat, []).append(tok)

    primitives: list[EventPrimitive] = []
    for cat_name, group in by_category.items():
        cat_cfg = taxonomy.categories.get(cat_name)
        if cat_cfg and cat_cfg.impulsive:
            scores = [
                EncoderScore(
                    time_sec=t.time_sec,
                    label=t.label,
                    score=t.confidence,
                    source=t.modality,
                )
                for t in group
            ]
            primitives.extend(
                _peak_split_scores(
                    scores,
                    category=cat_name,
                    taxonomy=taxonomy,
                    context_token=True,
                )
            )
            continue

        sorted_items = sorted(group, key=lambda x: x.time_sec)
        merge_gap = taxonomy.sustained_merge_gap_sec
        clusters: list[list[SymbolicToken]] = [[sorted_items[0]]]
        for tok in sorted_items[1:]:
            if tok.time_sec - clusters[-1][-1].time_sec <= merge_gap:
                clusters[-1].append(tok)
            else:
                clusters.append([tok])
        for cluster in clusters:
            peak_tok = max(cluster, key=lambda x: x.confidence)
            half_win = 0.35
            primitives.append(
                EventPrimitive(
                    category=cat_name if cat_name != "unknown" else None,
                    label=peak_tok.label,
                    start_sec=max(0.0, cluster[0].time_sec - half_win),
                    peak_sec=peak_tok.time_sec,
                    end_sec=cluster[-1].time_sec + half_win,
                    confidence=peak_tok.confidence,
                    sources=sorted({g.modality for g in cluster}),
                    context_token=True,
                )
            )
    return primitives


def _local_peaks(
    times: list[float],
    scores: list[float],
    *,
    min_score: float,
    min_distance_sec: float,
) -> list[int]:
    \"\"\"Indices of local maxima separated by at least min_distance_sec.\"\"\"
    if not times:
        return []

    # Collapse duplicate timestamps to max score
    buckets: dict[float, float] = {}
    for t, s in zip(times, scores):
        buckets[t] = max(buckets.get(t, 0.0), s)
    ordered_t = sorted(buckets)
    ordered_s = [buckets[t] for t in ordered_t]

    candidates: list[int] = []
    n = len(ordered_t)
    for i in range(n):
        if ordered_s[i] < min_score:
            continue
        left = ordered_s[i - 1] if i > 0 else -1.0
        right = ordered_s[i + 1] if i + 1 < n else -1.0
        if ordered_s[i] >= left and ordered_s[i] >= right:
            candidates.append(i)

    # Greedy keep highest peaks first, enforce spacing
    candidates.sort(key=lambda i: ordered_s[i], reverse=True)
    kept: list[int] = []
    for i in candidates:
        if all(abs(ordered_t[i] - ordered_t[j]) >= min_distance_sec for j in kept):
            kept.append(i)
    kept.sort()
    return kept


def _peak_split_scores(
    scores: list[EncoderScore],
    *,
    category: str,
    taxonomy: Taxonomy,
    context_token: bool = False,
) -> list[EventPrimitive]:
    \"\"\"One short event per local score peak (for gunshots / explosions).\"\"\"
    if not scores:
        return []

    min_score = taxonomy.impulsive_encoder_threshold
    half = taxonomy.impulsive_event_half_width_sec
    min_dist = taxonomy.impulsive_min_peak_distance_sec

    times = [s.time_sec for s in scores]
    vals = [s.score for s in scores]
    peak_idxs = _local_peaks(
        times, vals, min_score=min_score, min_distance_sec=min_dist
    )

    buckets: dict[float, float] = {}
    for t, s in zip(times, vals):
        buckets[t] = max(buckets.get(t, 0.0), s)
    ordered_t = sorted(buckets)

    if not peak_idxs:
        best = max(scores, key=lambda s: s.score)
        if best.score < min_score:
            return []
        peak_times = {best.time_sec}
    else:
        peak_times = {ordered_t[i] for i in peak_idxs}

    peak_scores: list[EncoderScore] = []
    for t in sorted(peak_times):
        nearby = [s for s in scores if abs(s.time_sec - t) <= 0.01]
        if not nearby:
            continue
        peak_scores.append(max(nearby, key=lambda s: s.score))

    primitives: list[EventPrimitive] = []
    for peak in peak_scores:
        nearby = [
            s
            for s in scores
            if abs(s.time_sec - peak.time_sec) <= half + 0.15
            and s.score >= min_score * 0.8
        ]
        sources = sorted({s.source for s in (nearby or [peak])})
        audio_scores = [s.score for s in nearby if s.source == "audio"] or (
            [peak.score] if peak.source == "audio" else []
        )
        video_scores = [s.score for s in nearby if s.source == "video"] or (
            [peak.score] if peak.source == "video" else []
        )
        primitives.append(
            EventPrimitive(
                category=category,
                label=peak.label,
                start_sec=max(0.0, peak.time_sec - half),
                peak_sec=peak.time_sec,
                end_sec=peak.time_sec + half,
                confidence=peak.score,
                sources=sources,
                context_token=context_token,
                audio_score=max(audio_scores) if audio_scores else None,
                video_score=max(video_scores) if video_scores else None,
            )
        )
    return primitives


def _merge_encoder_scores(
    scores: list[EncoderScore],
    taxonomy: Taxonomy,
    *,
    gap_sec: float | None = None,
) -> list[EventPrimitive]:
    \"\"\"Aggregate encoder scores by taxonomy category.\"\"\"
    if not scores:
        return []

    gap = gap_sec if gap_sec is not None else taxonomy.sustained_merge_gap_sec
    by_category: dict[str, list[EncoderScore]] = {}
    for s in scores:
        cat = match_label_to_category(taxonomy, s.label, s.source)
        if cat is None:
            continue
        by_category.setdefault(cat, []).append(s)

    primitives: list[EventPrimitive] = []
    for cat_name, group in by_category.items():
        cat_cfg = taxonomy.categories.get(cat_name)
        if cat_cfg and cat_cfg.impulsive:
            primitives.extend(
                _peak_split_scores(group, category=cat_name, taxonomy=taxonomy)
            )
            continue

        # Sustained categories (vehicle, weather): gap-merge above-threshold hits
        threshold = getattr(taxonomy, "sustained_encoder_threshold", taxonomy.encoder_threshold)
        group = sorted(
            [s for s in group if s.score >= threshold],
            key=lambda x: x.time_sec,
        )
        if not group:
            continue
        clusters: list[list[EncoderScore]] = [[group[0]]]
        for item in group[1:]:
            if item.time_sec - clusters[-1][-1].time_sec <= gap:
                clusters[-1].append(item)
            else:
                clusters.append([item])
        for cluster in clusters:
            peak = max(cluster, key=lambda x: x.score)
            sources = sorted({c.source for c in cluster})
            audio_scores = [c.score for c in cluster if c.source == "audio"]
            video_scores = [c.score for c in cluster if c.source == "video"]
            primitives.append(
                EventPrimitive(
                    category=cat_name,
                    label=peak.label,
                    start_sec=max(0.0, cluster[0].time_sec - 0.25),
                    peak_sec=peak.time_sec,
                    end_sec=cluster[-1].time_sec + 0.25,
                    confidence=peak.score,
                    sources=sources,
                    context_token=False,
                    audio_score=max(audio_scores) if audio_scores else None,
                    video_score=max(video_scores) if video_scores else None,
                )
            )
    return primitives


def _overlaps(a: EventPrimitive, b: EventPrimitive, *, margin_sec: float = 0.35) -> bool:
    if a.category != b.category:
        return False
    # Impulsive: only treat as same event if peaks are very close
    if abs(a.peak_sec - b.peak_sec) <= margin_sec:
        return True
    return not (a.end_sec + margin_sec < b.start_sec or b.end_sec + margin_sec < a.start_sec)


def aggregate_events(
    tokens: list[SymbolicToken],
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
) -> list[EventPrimitive]:
    \"\"\"Combine context symbolic tokens and encoder scores into primitives.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    token_primitives = _merge_tokens(tokens, taxonomy)
    encoder_primitives = _merge_encoder_scores(encoder_scores, taxonomy)

    if not token_primitives:
        return encoder_primitives
    if not encoder_primitives:
        return token_primitives

    merged = list(token_primitives)
    for enc in encoder_primitives:
        if any(_overlaps(enc, tok) for tok in token_primitives):
            continue
        merged.append(enc)
    merged.sort(key=lambda p: p.start_sec)
    return merged
""",
    "context/frozen_fusion.py": """\"\"\"Phase 2 frozen cross-modal fusion (rule-based, no training).\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.event_aggregation import EventPrimitive, aggregate_events
from haptic_gt.context.context_detectors import SymbolicToken
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category


@dataclass
class DetectedEvent:
    category: str
    label: str
    start_sec: float
    peak_sec: float
    end_sec: float
    confidence: float
    context_token: bool = False
    audio_score: float | None = None
    video_score: float | None = None
    sources: list[str] = field(default_factory=list)
    #: Attack level as a fraction of the clip's strongest attack, and how far
    #: the attack rose above the moment before it. Impulsive events only.
    attack_rel_max: float | None = None
    attack_prominence: float | None = None


def _best_encoder_scores(
    scores: list[EncoderScore],
    taxonomy: Taxonomy,
    center_sec: float,
    tolerance_sec: float = 0.75,
) -> dict[str, tuple[str, float, str]]:
    \"\"\"Return best audio/video score per category near center_sec.\"\"\"
    best: dict[str, tuple[str, float, str]] = {}
    for s in scores:
        if abs(s.time_sec - center_sec) > tolerance_sec:
            continue
        cat = match_label_to_category(taxonomy, s.label, s.source)
        if cat is None:
            continue
        prev = best.get(cat)
        if prev is None or s.score > prev[1]:
            best[cat] = (s.label, s.score, s.source)
    return best


def dedupe_events_by_peak(
    events: list[DetectedEvent],
    *,
    margin_sec: float = 0.35,
) -> list[DetectedEvent]:
    \"\"\"Keep the highest-confidence event when peaks collide after refine/snap.\"\"\"
    if not events:
        return []
    ordered = sorted(events, key=lambda e: (e.category, e.peak_sec, -e.confidence))
    merged: list[DetectedEvent] = []
    for ev in ordered:
        if merged and ev.category == merged[-1].category:
            if abs(ev.peak_sec - merged[-1].peak_sec) <= margin_sec:
                if ev.confidence > merged[-1].confidence:
                    merged[-1] = ev
                continue
        merged.append(ev)
    merged.sort(key=lambda e: e.start_sec)
    return merged


def fuse_events(
    tokens: list[SymbolicToken],
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Frozen cross-modal fusion at event level.

    1. Aggregate tokens/scores into primitives
    2. Map labels to taxonomy categories
    3. Apply per-category fusion rules and thresholds
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    primitives = aggregate_events(tokens, encoder_scores, taxonomy)
    detected: list[DetectedEvent] = []

    for prim in primitives:
        cat_name = prim.category
        if cat_name is None:
            cat_name = match_label_to_category(taxonomy, prim.label, "audio")
        if cat_name is None:
            cat_name = match_label_to_category(taxonomy, prim.label, "video")
        if cat_name is None or cat_name not in taxonomy.categories:
            continue

        cat_cfg = taxonomy.categories[cat_name]
        nearby = _best_encoder_scores(encoder_scores, taxonomy, prim.peak_sec)
        audio_score = prim.audio_score
        video_score = prim.video_score
        if cat_name in nearby:
            lbl, sc, src = nearby[cat_name]
            if src == "audio":
                audio_score = max(audio_score or 0.0, sc)
            else:
                video_score = max(video_score or 0.0, sc)
            if prim.label == lbl or prim.label in lbl:
                pass
            elif audio_score is None and video_score is None:
                prim.label = lbl

        has_context = prim.context_token
        if cat_cfg.impulsive:
            cat_threshold = taxonomy.impulsive_encoder_threshold
        elif hasattr(taxonomy, "sustained_encoder_threshold"):
            cat_threshold = taxonomy.sustained_encoder_threshold
        else:
            cat_threshold = taxonomy.encoder_threshold
        has_audio = (audio_score or 0.0) >= cat_threshold
        has_video = (video_score or 0.0) >= cat_threshold

        if cat_cfg.require_context_or_both:
            passes = has_context or (has_audio and has_video)
            if not passes and has_audio and cat_cfg.impulsive:
                passes = (audio_score or 0.0) >= cat_threshold
            if not passes:
                continue
        else:
            weighted = (
                cat_cfg.audio_weight * (audio_score or 0.0)
                + cat_cfg.video_weight * (video_score or 0.0)
            )
            score_floor = max(prim.confidence, weighted)
            if not has_context and score_floor < cat_threshold:
                continue

        if has_context and (has_audio or has_video):
            prim.confidence = min(1.0, prim.confidence * 1.05)

        detected.append(
            DetectedEvent(
                category=cat_name,
                label=prim.label,
                start_sec=prim.start_sec,
                peak_sec=prim.peak_sec,
                end_sec=prim.end_sec,
                confidence=prim.confidence,
                context_token=prim.context_token,
                audio_score=audio_score,
                video_score=video_score,
                sources=prim.sources,
            )
        )

    return dedupe_events_by_peak(detected)
""",
    "context/impulsive_nms.py": """\"\"\"Keep one accent per blast, and keep accents from running into each other.

A cannon does not stop making spectral flux when it stops firing: measured on a
tank clip, the 0.9 s after the loudest shot holds dozens of local maxima at
0.3-0.65 of that shot's level, because the blast decay and its reverb ride over
the engine. None of them is a separate bang -- they rise only 1.0-2.0x above the
moment before them, where every real shot in the same clip rose 4.7-9.1x.

Two things then go wrong downstream:

1. Two events survive closer than ``impulsive_min_peak_distance_sec``, because
   the generic peak dedupe uses its own fixed margin, so one blast is reported
   as a shot plus a second shot somewhere in its own tail.
2. Refinement extends every span by its decay tail, so neighbouring accents
   overlap and the renderer hits the same bang twice.
\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.onset_refine import _spectral_flux, local_flux_ratio
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy

#: Accent timing is only meaningful to about this tolerance, so an attack is
#: looked for within it rather than exactly on the reported peak.
_ATTACK_WINDOW_SEC = 0.025


class _Attacks:
    \"\"\"Spectral flux for one clip, queried at event peaks.\"\"\"

    def __init__(self, source_wav: Path, taxonomy: Taxonomy) -> None:
        audio, sr = sf.read(source_wav, always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        self.audio = audio.astype(np.float32)
        self.sr = int(sr)
        self.duration_sec = len(self.audio) / float(sr)
        self.times, self.flux = _spectral_flux(
            self.audio, self.sr, hop_ms=taxonomy.onset_flux_hop_ms
        )
        self.flux_peak = float(np.max(self.flux)) if self.flux.size else 0.0

    def _attack_index(self, t: float) -> int:
        \"\"\"Strongest flux bin within the accent's own lead time.

        An attack can be a single 5 ms bin, so reading the bin nearest the peak
        reports less than half the real level when timing is off by one hop.
        \"\"\"
        core = np.flatnonzero(np.abs(self.times - t) <= _ATTACK_WINDOW_SEC)
        if core.size == 0:
            return int(np.argmin(np.abs(self.times - t)))
        return int(core[int(np.argmax(self.flux[core]))])

    def level(self, t: float) -> float:
        if not self.flux.size or not self.times.size:
            return 0.0
        return float(self.flux[self._attack_index(t)])

    def rel_max(self, t: float) -> float:
        if self.flux_peak <= 0.0:
            return 0.0
        return self.level(t) / self.flux_peak

    def prominence(self, t: float) -> float:
        if not self.flux.size or not self.times.size:
            return 0.0
        return local_flux_ratio(
            self.times, self.flux, float(self.times[self._attack_index(t)])
        )


#: Decay bumps sit 0.2-0.4 s after a muzzle; this is the window in which there
#: is only one bang. Prefer the earliest sharp attack that is still loud enough
#: (muzzle), not the absolute loudest bin (often a boom 0.2-0.3 s later).
_SNAP_PROMINENCE = 3.0
_SNAP_EARLIEST_FRAC = 0.65


def _local_max_indices(flux: np.ndarray) -> np.ndarray:
    if flux.size < 3:
        return np.array([], dtype=int)
    return np.flatnonzero((flux[1:-1] >= flux[:-2]) & (flux[1:-1] >= flux[2:])) + 1


def snap_impulsive_peaks_to_attacks(
    events: list[DetectedEvent],
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"Move each impulsive peak onto the muzzle in its own spacing window.

    Frame SED and the first refine pass often land 0.2-0.4 s late, on a bump in
    the blast's own decay. Promote then skips the real attack because it is
    inside the 0.40 s neighbour gate, so the haptic accent fires on the tail.
    Inside ``impulsive_min_peak_distance_sec`` there is only one bang: pick the
    earliest sharp attack that is still ≥65% of the loudest sharp peak in that
    window (prominence ≥ 3.0). Loudest-alone was landing on the boom after the
    muzzle and making the first haptic feel late.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    if not any(_is_impulsive(e, taxonomy) for e in events):
        return list(events)

    attacks = _Attacks(Path(source_wav), taxonomy)
    if attacks.flux.size < 3:
        return list(events)

    radius = taxonomy.impulsive_min_peak_distance_sec
    pre_roll = taxonomy.impulsive_pre_roll_sec
    half = taxonomy.impulsive_event_half_width_sec
    peak_idxs = _local_max_indices(attacks.flux)

    out: list[DetectedEvent] = []
    for ev in events:
        if not _is_impulsive(ev, taxonomy):
            out.append(ev)
            continue
        best_t = _best_sharp_attack(
            attacks, peak_idxs, ev.peak_sec, radius=radius
        )
        if best_t is None or abs(best_t - ev.peak_sec) < 1e-4:
            out.append(ev)
            continue
        sources = list(ev.sources)
        if "flux_snap" not in sources:
            sources.append("flux_snap")
        start = max(0.0, best_t - pre_roll)
        end = min(attacks.duration_sec, max(ev.end_sec, best_t + half))
        out.append(
            DetectedEvent(
                category=ev.category,
                label=ev.label,
                start_sec=start,
                peak_sec=best_t,
                end_sec=max(end, start + 0.2),
                confidence=ev.confidence,
                context_token=ev.context_token,
                audio_score=ev.audio_score,
                video_score=ev.video_score,
                sources=sources,
                attack_rel_max=ev.attack_rel_max,
                attack_prominence=ev.attack_prominence,
            )
        )
    return out


def _best_sharp_attack(
    attacks: _Attacks,
    peak_idxs: np.ndarray,
    t: float,
    *,
    radius: float,
) -> float | None:
    \"\"\"Earliest adequate sharp local maximum within ``radius`` of ``t``.\"\"\"
    if peak_idxs.size == 0:
        return None
    nearby = peak_idxs[np.abs(attacks.times[peak_idxs] - t) <= radius]
    if nearby.size == 0:
        return None
    scored: list[tuple[float, float, float]] = []
    for idx in nearby:
        peak_t = float(attacks.times[idx])
        promin = local_flux_ratio(attacks.times, attacks.flux, peak_t)
        if promin < _SNAP_PROMINENCE:
            continue
        scored.append((float(attacks.flux[idx]), promin, peak_t))
    if not scored:
        return None
    loudest = max(row[0] for row in scored)
    floor = _SNAP_EARLIEST_FRAC * loudest
    adequate = [row for row in scored if row[0] >= floor]
    adequate.sort(key=lambda row: row[2])
    return adequate[0][2]


def _is_impulsive(ev: DetectedEvent, taxonomy: Taxonomy) -> bool:
    cfg = taxonomy.categories.get(ev.category)
    return bool(cfg is not None and cfg.impulsive)


def suppress_impulsive_overlaps(
    events: list[DetectedEvent],
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"One accent per blast: enforce peak spacing, then stop spans overlapping.

    Ranking is by attack level rather than confidence, because frame posteriors
    from PANNs come out flat (every promoted shot reports the decode threshold),
    so confidence cannot say which of two neighbours is the real blast.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    impulsive = [e for e in events if _is_impulsive(e, taxonomy)]
    if len(impulsive) < 2:
        return list(events)

    attacks = _Attacks(Path(source_wav), taxonomy)
    min_dist = taxonomy.impulsive_min_peak_distance_sec

    by_strength = sorted(
        impulsive,
        key=lambda e: (attacks.level(e.peak_sec), e.confidence),
        reverse=True,
    )
    kept: list[DetectedEvent] = []
    for ev in by_strength:
        if any(
            other.category == ev.category and abs(other.peak_sec - ev.peak_sec) < min_dist
            for other in kept
        ):
            continue
        kept.append(ev)

    kept.sort(key=lambda e: e.peak_sec)
    for i in range(len(kept) - 1):
        cur, nxt = kept[i], kept[i + 1]
        if cur.end_sec > nxt.start_sec:
            kept[i] = DetectedEvent(
                category=cur.category,
                label=cur.label,
                start_sec=cur.start_sec,
                peak_sec=cur.peak_sec,
                end_sec=max(nxt.start_sec, cur.peak_sec + 1e-3),
                confidence=cur.confidence,
                context_token=cur.context_token,
                audio_score=cur.audio_score,
                video_score=cur.video_score,
                sources=list(cur.sources),
                attack_rel_max=cur.attack_rel_max,
                attack_prominence=cur.attack_prominence,
            )

    out = [e for e in events if not _is_impulsive(e, taxonomy)]
    out += kept
    out.sort(key=lambda e: e.start_sec)
    return out


def measure_impulsive_attacks(
    events: list[DetectedEvent],
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"Record how strong and how sharp each accent's attack actually was.

    Frame posteriors are reported as the decode threshold for every promoted
    shot, which hides the difference between the loudest cannon and a distant
    impact. These two numbers are what the promotion thresholds are tuned on.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    if not any(_is_impulsive(e, taxonomy) for e in events):
        return list(events)

    attacks = _Attacks(Path(source_wav), taxonomy)
    out: list[DetectedEvent] = []
    for ev in events:
        if not _is_impulsive(ev, taxonomy):
            out.append(ev)
            continue
        ev.attack_rel_max = round(attacks.rel_max(ev.peak_sec), 4)
        ev.attack_prominence = round(attacks.prominence(ev.peak_sec), 3)
        out.append(ev)
    return out
""",
    "context/impulsive_promote.py": """\"\"\"Recover cannon/gunshot transients that AST labeled as vehicle.\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.onset_refine import _spectral_flux, local_flux_ratio
from haptic_gt.context.proposals import _local_peak_indices
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category

# AST confidence that marks an event as a real shot for level calibration
_CONFIRMED_CONF = 0.55
# When no confirmed shot exists, treat this fraction of clip max flux as the level
_FALLBACK_REF = 0.35
# Nearby AST explosion/gunshot score required to promote. Model backing lowers
# the level a shot must reach, not the rise: the score comes from a window about
# a second wide, so it reads "explosion" all the way through a blast's ring-out
# and cannot say whether this particular moment is a new bang or the old one
# still fading. Sharpness has to answer that on its own.
_PROMOTE_AST_FLOOR = 0.20
_AST_REF_FRAC = 0.22
_AST_PROMINENCE = 4.0
# No AST support: must look like the confirmed shots
_FLUX_ONLY_REF_FRAC = 0.60
_FLUX_ONLY_PROMINENCE = 4.0
# Demotion: a fused "explosion" this far below shot level is rumble
_DEMOTE_REF_FRAC = 0.18

# An "anchor" blast, recognised by its own signature rather than by a model score.
# Confidence scales differ per backend (AST window scores run high, PANNs frame
# posteriors low), so a fixed 0.55 silently fails on some backends and leaves the
# shot level guessed from clip max -- which puts track clanks near "shot loud".
_ANCHOR_REL_MAX = 0.40
_ANCHOR_PROMINENCE = 4.0
# Inside a volley the decay of the previous blast inflates the pre-attack
# baseline, so prominence collapses on shots that are plainly loud. Near an
# anchor, lean on absolute level instead -- but not on level alone: a blast's own
# ring-out holds bumps at 0.47-0.65 of the shot that made them, which is as loud
# as a real shot and would fire a second accent into the first bang's tail. The
# bar sits above the loudest such bump (2.9x) and below the weakest real blast
# (4.0x) measured on a tank clip.
_VOLLEY_WINDOW_SEC = 2.0
_VOLLEY_REF_FRAC = 0.50
_VOLLEY_PROMINENCE = 3.0

# A shell landing away from the volley is quieter than the cannon firing next to
# the camera, so it never reaches the flux-only level gate -- but it is far
# sharper than anything the drive makes. Supported impacts (AST / in-volley) may
# sit near 3x; isolated flux-only spikes need a much higher bar so end-of-clip
# SFX does not become a phantom fire.
_IMPACT_PROMINENCE = 3.0
_IMPACT_REF_FRAC = 0.40
_ISOLATED_IMPACT_PROMINENCE = 6.0


def _flux_at(times: np.ndarray, flux: np.ndarray, peak_t: float) -> float:
    if flux.size == 0 or times.size == 0:
        return 0.0
    idx = int(np.argmin(np.abs(times - peak_t)))
    return float(flux[idx])


def _is_sharp_transient(
    times: np.ndarray,
    flux: np.ndarray,
    peak_t: float,
    flux_peak: float,
    *,
    ref_flux: float | None = None,
    local_ratio: float = 2.0,
) -> bool:
    \"\"\"True blast: near shot level with a sharp attack. Rumble is neither.\"\"\"
    if flux_peak < 1e-12:
        return False
    ref = ref_flux if ref_flux and ref_flux > 1e-12 else flux_peak * _FALLBACK_REF
    v = _flux_at(times, flux, peak_t)
    if v < _DEMOTE_REF_FRAC * ref:
        return False
    return local_flux_ratio(times, flux, peak_t) >= local_ratio


def _best_impulsive_score(
    scores: list[EncoderScore],
    taxonomy: Taxonomy,
    peak_t: float,
    win_sec: float,
) -> tuple[str, str, float] | None:
    nearby = [s for s in scores if abs(s.time_sec - peak_t) <= win_sec]
    best: tuple[str, str, float] | None = None
    for s in nearby:
        cat = match_label_to_category(taxonomy, s.label, s.source)
        if cat is None:
            continue
        cfg = taxonomy.categories.get(cat)
        if cfg is None or not cfg.impulsive:
            continue
        if best is None or s.score > best[2]:
            best = (cat, s.label, float(s.score))
    return best


def promote_impulsive_transients(
    events: list[DetectedEvent],
    source_wav: str | Path,
    encoder_scores: list[EncoderScore],
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    If a sharp flux peak has explosion/gunshot evidence, emit an impulsive
    event even when vehicle scored higher (engine bed under the muzzle).

    Quieter volley shots are found by local flux, not 32% of the loudest bang.
    Rumble-like AST explosions (tank drive) are dropped.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    duration = len(audio) / float(sr)

    times, flux = _spectral_flux(audio, sr, hop_ms=taxonomy.onset_flux_hop_ms)
    if flux.size == 0:
        return list(events)

    flux_peak = float(np.max(flux))
    if flux_peak < 1e-12:
        return list(events)

    hop_sec = taxonomy.onset_flux_hop_ms / 1000.0
    min_dist = taxonomy.impulsive_min_peak_distance_sec
    min_frames = max(1, int(round(min_dist / hop_sec)))
    floor = flux_peak * 0.06
    idxs = _local_peak_indices(flux, min_score=floor, min_distance_frames=min_frames)

    clip_cat = "explosion"
    clip_label = "Explosion"
    clip_imp_score = 0.0
    for s in encoder_scores:
        cat = match_label_to_category(taxonomy, s.label, s.source)
        if cat is None:
            continue
        cfg = taxonomy.categories.get(cat)
        if cfg is None or not cfg.impulsive:
            continue
        if s.score > clip_imp_score:
            clip_imp_score = float(s.score)
            clip_cat = cat
            clip_label = s.label

    def _impulsive(ev: DetectedEvent) -> bool:
        cfg = taxonomy.categories.get(ev.category)
        return bool(cfg is not None and cfg.impulsive)

    # Calibrate "how loud is a shot in THIS clip". Track clanks are a small
    # fraction of that level; volley shots are not.
    ref_levels = [
        _flux_at(times, flux, e.peak_sec)
        for e in events
        if _impulsive(e) and e.confidence >= _CONFIRMED_CONF
    ]
    ref_levels = [v for v in ref_levels if v > 0.0]

    # Failing that, anchor on peaks that look like blasts on their own terms.
    anchors = [
        float(times[i])
        for i in idxs
        if float(flux[i]) >= _ANCHOR_REL_MAX * flux_peak
        and local_flux_ratio(times, flux, float(times[i])) >= _ANCHOR_PROMINENCE
    ]
    if not ref_levels and anchors:
        ref_levels = [_flux_at(times, flux, t) for t in anchors]

    ref_confirmed = len(ref_levels) >= 1
    ref_flux = (
        float(np.median(ref_levels)) if ref_confirmed else flux_peak * _FALLBACK_REF
    )
    if ref_flux < 1e-12:
        ref_flux = flux_peak * _FALLBACK_REF

    # Drop rumble-like fused explosions (drive noise scored as blast)
    surviving: list[DetectedEvent] = []
    for ev in events:
        if _impulsive(ev) and not _is_sharp_transient(
            times, flux, ev.peak_sec, flux_peak, ref_flux=ref_flux
        ):
            continue
        surviving.append(ev)

    existing_imp = [e for e in surviving if _impulsive(e)]
    half = taxonomy.impulsive_event_half_width_sec
    new_events: list[DetectedEvent] = []

    for idx in idxs:
        peak_t = float(times[idx])
        if any(abs(e.peak_sec - peak_t) <= 0.40 for e in existing_imp + new_events):
            continue
        v = _flux_at(times, flux, peak_t)
        prominence = local_flux_ratio(times, flux, peak_t)
        best = _best_impulsive_score(encoder_scores, taxonomy, peak_t, 0.7)
        has_nearby = best is not None and best[2] >= _PROMOTE_AST_FLOOR
        # With AST backing a quieter volley shot is enough; without it the peak
        # must look like the confirmed shots, not like a track clank.
        in_volley = any(
            0.05 < abs(peak_t - a) <= _VOLLEY_WINDOW_SEC for a in anchors
        )
        # Flux-only fills gaps near SED/fusion events, not phantom end-of-clip SFX.
        near_seed = any(
            abs(peak_t - e.peak_sec) <= _VOLLEY_WINDOW_SEC
            for e in surviving
            if _impulsive(e)
        )
        sharp_impact = (
            prominence >= _IMPACT_PROMINENCE and v >= _IMPACT_REF_FRAC * ref_flux
        )
        if sharp_impact and (has_nearby or in_volley):
            ok = True
        elif sharp_impact and prominence >= _ISOLATED_IMPACT_PROMINENCE:
            ok = True
        elif in_volley and v >= _VOLLEY_REF_FRAC * ref_flux:
            ok = prominence >= _VOLLEY_PROMINENCE
        elif has_nearby:
            ok = v >= _AST_REF_FRAC * ref_flux and prominence >= _AST_PROMINENCE
        else:
            ok = (
                ref_confirmed
                and near_seed
                and clip_imp_score >= 0.35
                and v >= _FLUX_ONLY_REF_FRAC * ref_flux
                and prominence >= _FLUX_ONLY_PROMINENCE
            )
        if not ok:
            continue
        if has_nearby:
            cat, label, score = best
        else:
            cat, label = clip_cat, clip_label
            score = max(0.30, clip_imp_score * 0.55)
        start = max(0.0, peak_t - taxonomy.impulsive_pre_roll_sec)
        end = min(duration, peak_t + half)
        new_events.append(
            DetectedEvent(
                category=cat,
                label=label,
                start_sec=start,
                peak_sec=peak_t,
                end_sec=max(end, start + 0.2),
                confidence=max(score, taxonomy.impulsive_encoder_threshold),
                context_token=False,
                audio_score=score,
                video_score=None,
                sources=["audio", "flux"],
            )
        )
        existing_imp.append(new_events[-1])

    if not new_events:
        return surviving

    kept: list[DetectedEvent] = list(new_events)
    for ev in surviving:
        # A short sustained chip sitting on a blast is that blast mislabelled, so
        # it goes. A rumble bed that merely happens to peak under one shot is the
        # engine, and dropping it would stop the vibration the moment the cannon
        # fires -- length is what tells them apart.
        short_chip = (ev.end_sec - ev.start_sec) <= taxonomy.sustained_max_gate_sec
        if not _impulsive(ev) and short_chip:
            if any(abs(ev.peak_sec - imp.peak_sec) <= 0.35 for imp in new_events):
                continue
            if any(
                imp.start_sec - 0.1 <= ev.peak_sec <= imp.end_sec + 0.1
                for imp in new_events
            ):
                continue
        kept.append(ev)
    kept.sort(key=lambda e: e.start_sec)
    return kept
""",
    "context/manual_events.py": """\"\"\"Parse hand-labeled events for HITL ground-truth generation.\"\"\"

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def _as_event_dicts(spec: list[dict[str, Any]] | dict[str, Any] | str | Path) -> list[dict[str, Any]]:
    if isinstance(spec, (str, Path)):
        payload = json.loads(Path(spec).read_text(encoding="utf-8"))
        if isinstance(payload, list):
            return list(payload)
        if isinstance(payload, dict):
            events = payload.get("events")
            if isinstance(events, list):
                return list(events)
            if "category" in payload and "peak_sec" in payload:
                return [payload]
        raise ValueError(f"Unsupported manual events JSON shape: {spec}")
    if isinstance(spec, dict):
        if "events" in spec and isinstance(spec["events"], list):
            return list(spec["events"])
        return [spec]
    return list(spec)


def events_from_manual(
    spec: list[dict[str, Any]] | dict[str, Any] | str | Path,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Build DetectedEvent list from hand labels.

    Each event needs ``category`` and ``peak_sec``. ``start_sec`` / ``end_sec``
    default to a short window around the peak when omitted.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    rows = _as_event_dicts(spec)
    if not rows:
        return []

    half = taxonomy.impulsive_event_half_width_sec
    pre_roll = taxonomy.impulsive_pre_roll_sec
    events: list[DetectedEvent] = []

    for row in rows:
        category = str(row["category"]).strip()
        if category not in taxonomy.categories:
            raise ValueError(
                f"Unknown category {category!r}. "
                f"Expected one of: {sorted(taxonomy.categories)}"
            )
        peak = float(row["peak_sec"])
        start = float(row["start_sec"]) if row.get("start_sec") is not None else max(0.0, peak - pre_roll)
        end = float(row["end_sec"]) if row.get("end_sec") is not None else peak + half
        if end < start:
            raise ValueError(f"end_sec ({end}) must be >= start_sec ({start})")
        peak = min(max(peak, start), end)
        label = str(row.get("label") or category)
        confidence = float(row.get("confidence", 1.0))
        events.append(
            DetectedEvent(
                category=category,
                label=label,
                start_sec=start,
                peak_sec=peak,
                end_sec=end,
                confidence=confidence,
                context_token=False,
                audio_score=None,
                video_score=None,
                sources=["manual"],
            )
        )

    events.sort(key=lambda e: e.start_sec)
    return events


def vehicle_events_from_peaks(
    peaks_sec: list[float],
    *,
    half_width_sec: float = 0.45,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Build vehicle rumble events from hand-marked peak times (HITL GT).

    Use when auto vehicle detection cannot match ears — calib showed many true
    rumbles have almost no RMS rise, while bed false-positives often do.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    if "vehicle" not in taxonomy.categories:
        raise ValueError("taxonomy has no vehicle category")
    events: list[DetectedEvent] = []
    for peak in sorted(float(p) for p in peaks_sec):
        start = max(0.0, peak - half_width_sec * 0.35)
        end = peak + half_width_sec
        events.append(
            DetectedEvent(
                category="vehicle",
                label="vehicle_rumble",
                start_sec=start,
                peak_sec=peak,
                end_sec=end,
                confidence=1.0,
                context_token=False,
                audio_score=None,
                video_score=None,
                sources=["manual"],
            )
        )
    return events
""",
    "context/mask.py": """\"\"\"Build gated audio from detected event segments.\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def resolve_gate_categories(
    taxonomy: Taxonomy,
    gate_categories: list[str] | None = None,
) -> list[str]:
    \"\"\"Return categories used for haptic gating.\"\"\"
    if gate_categories is not None:
        return list(gate_categories)
    return [
        name
        for name, cat in taxonomy.categories.items()
        if cat.include_in_haptic_gate
    ]


def events_for_haptic_gate(
    events: list[DetectedEvent],
    taxonomy: Taxonomy,
    gate_categories: list[str] | None = None,
) -> list[DetectedEvent]:
    \"\"\"Events that should contribute to gated_audio.wav.\"\"\"
    allowed = set(resolve_gate_categories(taxonomy, gate_categories))
    gated: list[DetectedEvent] = []
    for ev in events:
        if ev.category in allowed:
            gated.append(ev)
    return gated


def event_included_in_gate(
    event: DetectedEvent,
    taxonomy: Taxonomy,
    gate_categories: list[str] | None = None,
) -> bool:
    allowed = set(resolve_gate_categories(taxonomy, gate_categories))
    return event.category in allowed


def build_event_mask(
    duration_samples: int,
    sample_rate: int,
    events: list[DetectedEvent],
    *,
    fade_ms: float = 10.0,
) -> np.ndarray:
    \"\"\"Binary mask with short linear fades at segment edges.\"\"\"
    mask = np.zeros(duration_samples, dtype=np.float32)
    fade = max(1, int(sample_rate * fade_ms / 1000.0))

    for ev in events:
        s0 = max(0, int(ev.start_sec * sample_rate))
        s1 = min(duration_samples, int(ev.end_sec * sample_rate))
        if s1 <= s0:
            continue
        mask[s0:s1] = 1.0
        f0 = min(fade, (s1 - s0) // 2)
        if f0 > 0:
            ramp = np.linspace(0.0, 1.0, f0, dtype=np.float32)
            mask[s0 : s0 + f0] = np.maximum(mask[s0 : s0 + f0], ramp)
            mask[s1 - f0 : s1] = np.maximum(mask[s1 - f0 : s1], ramp[::-1])

    return mask


def apply_gate(
    source_wav: str | Path,
    output_wav: str | Path,
    events: list[DetectedEvent],
    *,
    sample_rate: int = INPUT_SR,
    taxonomy: Taxonomy | None = None,
    gate_categories: list[str] | None = None,
) -> Path:
    \"\"\"Write gated mono WAV keeping only selected event segments.\"\"\"
    source_wav = Path(source_wav)
    output_wav = Path(output_wav)
    taxonomy = taxonomy or load_taxonomy()
    gate_events = events_for_haptic_gate(events, taxonomy, gate_categories=gate_categories)

    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate

    mask = build_event_mask(len(audio), sr, gate_events)
    gated = audio * mask

    output_wav.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_wav, np.clip(gated, -1.0, 1.0), sr, subtype="PCM_16")
    return output_wav
""",
    "context/onset_refine.py": """\"\"\"Refine event peak/start/end using spectral-flux onset alignment.\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def _envelope_rms(
    audio: np.ndarray,
    sr: int,
    *,
    hop_ms: float = 5.0,
) -> tuple[np.ndarray, np.ndarray]:
    hop = max(1, int(sr * hop_ms / 1000))
    n_frames = max(1, (len(audio) + hop - 1) // hop)
    env: list[float] = []
    times: list[float] = []
    for i in range(n_frames):
        s0 = i * hop
        s1 = min(len(audio), s0 + hop)
        chunk = audio[s0:s1]
        env.append(float(np.sqrt(np.mean(chunk**2) + 1e-12)))
        times.append((s0 + s1) / 2.0 / sr)
    return np.asarray(times, dtype=np.float64), np.asarray(env, dtype=np.float64)


def _spectral_flux(
    audio: np.ndarray,
    sr: int,
    *,
    hop_ms: float = 5.0,
    n_fft: int = 512,
    hf_weight: float = 1.5,
) -> tuple[np.ndarray, np.ndarray]:
    \"\"\"
    Positive spectral flux with high-frequency emphasis.

    Better than RMS for impulsive onsets (gunshot / explosion attacks).
    \"\"\"
    hop = max(1, int(sr * hop_ms / 1000))
    if len(audio) < n_fft:
        audio = np.pad(audio, (0, n_fft - len(audio)))

    window = np.hanning(n_fft).astype(np.float32)
    n_frames = 1 + max(0, (len(audio) - n_fft) // hop)
    if n_frames < 2:
        return np.array([0.0]), np.array([0.0])

    freqs = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    # Emphasize mid/high bands typical of blast attacks
    band_w = np.ones_like(freqs, dtype=np.float32)
    band_w[freqs >= 500.0] = hf_weight
    band_w[freqs >= 2000.0] = hf_weight * 1.25
    band_w[freqs < 80.0] = 0.35

    mags: list[np.ndarray] = []
    times: list[float] = []
    for i in range(n_frames):
        s0 = i * hop
        s1 = s0 + n_fft
        if s1 > len(audio):
            break
        frame = audio[s0:s1] * window
        mag = np.abs(np.fft.rfft(frame)).astype(np.float32) * band_w
        mags.append(mag)
        times.append((s0 + n_fft / 2.0) / sr)

    if len(mags) < 2:
        return np.asarray(times, dtype=np.float64), np.zeros(len(times), dtype=np.float64)

    flux = np.zeros(len(mags), dtype=np.float64)
    for i in range(1, len(mags)):
        diff = mags[i] - mags[i - 1]
        flux[i] = float(np.sum(np.maximum(diff, 0.0)))

    return np.asarray(times, dtype=np.float64), flux


def local_flux_ratio(
    times: np.ndarray,
    flux: np.ndarray,
    peak_t: float,
    *,
    pre_sec: float = 0.65,
    gap_sec: float = 0.04,
    min_pre_sec: float = 0.20,
    max_ratio: float = 999.0,
) -> float:
    \"\"\"Peak flux vs median flux just before it (sharp attack vs rumble).

    Returns 0 without enough pre-context: the clip's first frames always look
    like a huge attack (silence to signal) and must not count as a transient.
    After digital silence the baseline is exactly 0, so the ratio is capped
    rather than allowed to blow up to 1e14.
    \"\"\"
    if flux.size == 0 or times.size == 0:
        return 0.0
    idx = int(np.argmin(np.abs(times - peak_t)))
    peak_v = float(flux[idx])
    pre_mask = (times >= peak_t - pre_sec) & (times < peak_t - gap_sec)
    pre = flux[pre_mask]
    pre_times = times[pre_mask]
    if pre.size == 0:
        return 0.0
    if float(pre_times[-1] - pre_times[0]) < min_pre_sec:
        return 0.0
    baseline = float(np.median(pre))
    if baseline <= 0.0:
        baseline = float(np.mean(pre))
    if baseline <= 0.0:
        return max_ratio if peak_v > 0.0 else 0.0
    return min(peak_v / baseline, max_ratio)


def _pick_onset_from_flux(
    times: np.ndarray,
    flux: np.ndarray,
    *,
    min_ratio: float = 0.45,
    prefer_earliest: bool = False,
    early_rel: float = 0.55,
) -> float | None:
    \"\"\"Pick onset from flux peaks.

    Default: strongest peak, earliest among near-ties.
    ``prefer_earliest``: earliest peak that is still comparable to the strongest
    one in the window (``early_rel``). A weak earlier bump — a track clank
    before the shot — must not steal the onset.
    \"\"\"
    if flux.size == 0:
        return None
    peak = float(np.max(flux))
    if peak < 1e-12:
        return None

    floor = peak * min_ratio
    candidates: list[int] = []
    for i in range(1, len(flux) - 1):
        if flux[i] < floor:
            continue
        if flux[i] >= flux[i - 1] and flux[i] >= flux[i + 1]:
            candidates.append(i)
    if not candidates:
        return float(times[int(np.argmax(flux))])

    if prefer_earliest:
        best_val = max(float(flux[i]) for i in candidates)
        pool = [i for i in candidates if float(flux[i]) >= best_val * early_rel]
        earliest = min(pool or candidates, key=lambda i: float(times[i]))
        return float(times[earliest])

    # Prefer the highest flux; among near-ties, earliest attack (muzzle), not later rumble
    best_val = max(float(flux[i]) for i in candidates)
    strong = [i for i in candidates if float(flux[i]) >= best_val * 0.92]
    earliest = min(strong, key=lambda i: float(times[i]))
    return float(times[earliest])


def _find_acoustic_peak(
    audio: np.ndarray,
    sr: int,
    center_sec: float,
    radius_sec: float,
) -> float:
    \"\"\"Peak RMS sample time near the classifier's rough center.\"\"\"
    s0 = max(0, int((center_sec - radius_sec) * sr))
    s1 = min(len(audio), int((center_sec + radius_sec) * sr))
    if s1 <= s0:
        return center_sec

    seg = audio[s0:s1]
    times, env = _envelope_rms(seg, sr)
    if env.size == 0:
        return center_sec
    peak_idx = int(np.argmax(env))
    return float(s0 / sr + times[peak_idx])


def _find_impulsive_peak(
    audio: np.ndarray,
    sr: int,
    center_sec: float,
    taxonomy: Taxonomy,
    *,
    duration_sec: float,
) -> float:
    \"\"\"
    Snap impulsive events to spectral-flux onset near the classifier hint.

    Uses high-frequency-weighted spectral flux (not RMS max), which better
    matches gunshot / explosion attacks. Falls back to RMS if flux is weak.

    On short clips only, look back to t=0 so a late AST/ViViT hit (e.g. rumble)
    can still snap to an earlier muzzle blast. Mixed clips keep a local window
    so tank-drive clanks are not stolen as the cannon onset.
    \"\"\"
    back = taxonomy.impulsive_onset_back_sec
    forward = taxonomy.impulsive_onset_forward_sec
    if duration_sec <= taxonomy.impulsive_short_clip_sec:
        back = max(back, center_sec)
    hop_ms = taxonomy.onset_flux_hop_ms
    s0 = max(0, int((center_sec - back) * sr))
    s1 = min(len(audio), int((center_sec + forward) * sr))
    if s1 <= s0:
        return center_sec

    seg = audio[s0:s1]
    times, flux = _spectral_flux(seg, sr, hop_ms=hop_ms)
    onset_rel = _pick_onset_from_flux(
        times,
        flux,
        min_ratio=taxonomy.onset_flux_min_ratio,
        prefer_earliest=True,
        early_rel=taxonomy.onset_flux_early_rel,
    )
    if onset_rel is not None:
        return float(s0 / sr + onset_rel)

    # Fallback: RMS peak in the same window
    times_rms, env = _envelope_rms(seg, sr, hop_ms=max(2.5, hop_ms))
    if env.size == 0:
        return center_sec
    return float(s0 / sr + times_rms[int(np.argmax(env))])


def _extend_decay_tail(
    audio: np.ndarray,
    sr: int,
    peak_sec: float,
    *,
    threshold_ratio: float,
    max_tail_sec: float,
    duration_sec: float,
) -> float:
    \"\"\"Extend gate end while RMS stays above a fraction of the peak.\"\"\"
    peak_idx = int(peak_sec * sr)
    hop = max(1, int(sr * 0.005))
    lo = max(0, peak_idx - hop)
    hi = min(len(audio), peak_idx + hop)
    peak_val = float(np.sqrt(np.mean(audio[lo:hi] ** 2) + 1e-12))
    if peak_val < 1e-8:
        return peak_sec

    thr = peak_val * threshold_ratio
    end_idx = min(len(audio), int((peak_sec + max_tail_sec) * sr))
    i = peak_idx
    while i < end_idx:
        s1 = min(len(audio), i + hop)
        v = float(np.sqrt(np.mean(audio[i:s1] ** 2) + 1e-12))
        if v < thr:
            break
        i += hop
    return min(duration_sec, i / sr)


def refine_event_timing(
    events: list[DetectedEvent],
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
    *,
    relocate_impulsive_peaks: bool = True,
) -> list[DetectedEvent]:
    \"\"\"
    Snap classifier window centers to acoustic onsets and fix gate spans.

    Impulsive: spectral-flux onset; short pre-roll + decay tail.
    Sustained: RMS peak; span capped for gating metadata.

    Set ``relocate_impulsive_peaks`` false to rebuild spans around peaks that are
    already on an attack. Searching again would walk them off it: the window is
    wider than a shot's decay, so on a volley the earliest-comparable rule
    reaches back into the previous blast's tail and two accents collapse onto one
    moment -- and the moment it lands on was never checked for sharpness.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    duration_sec = len(audio) / sr

    radius = taxonomy.impulsive_onset_search_radius_sec
    half = taxonomy.impulsive_event_half_width_sec
    pre_roll = taxonomy.impulsive_pre_roll_sec
    max_tail = taxonomy.impulsive_decay_tail_sec
    decay_thr = taxonomy.impulsive_decay_threshold
    sustained_max = taxonomy.sustained_max_gate_sec

    refined: list[DetectedEvent] = []
    for ev in events:
        cat_cfg = taxonomy.categories.get(ev.category)
        impulsive = bool(cat_cfg and cat_cfg.impulsive)

        if impulsive:
            acoustic_peak = (
                _find_impulsive_peak(
                    audio, sr, ev.peak_sec, taxonomy, duration_sec=duration_sec
                )
                if relocate_impulsive_peaks
                else ev.peak_sec
            )
        else:
            acoustic_peak = _find_acoustic_peak(audio, sr, ev.peak_sec, radius)

        if impulsive:
            peak = acoustic_peak
            start = max(0.0, peak - pre_roll)
            # Keep some post-onset content for algorithm context
            min_end = peak + half
            decay_end = _extend_decay_tail(
                audio,
                sr,
                peak,
                threshold_ratio=decay_thr,
                max_tail_sec=max_tail,
                duration_sec=duration_sec,
            )
            end = min(duration_sec, max(min_end, decay_end))
        else:
            peak = acoustic_peak
            # Keep classifier span; do NOT recenter around peak (that invents
            # rumble during quiet gaps between intermittent bursts).
            start = max(0.0, ev.start_sec)
            end = min(duration_sec, max(ev.end_sec, peak + 0.05))
            if end < start:
                end = min(duration_sec, start + 0.25)
            # Soft cap: trim the quieter tail, keep the onset side
            if end - start > sustained_max and sustained_max > 0:
                end = min(end, start + sustained_max)
            peak = min(max(peak, start), end)

        refined.append(
            DetectedEvent(
                category=ev.category,
                label=ev.label,
                start_sec=start,
                peak_sec=peak,
                end_sec=end,
                confidence=ev.confidence,
                context_token=ev.context_token,
                audio_score=ev.audio_score,
                video_score=ev.video_score,
                sources=list(ev.sources),
            )
        )
    return refined
""",
    "context/proposals.py": """\"\"\"Cheap RMS onset proposals before frozen transformer classification.\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.audio_io import INPUT_SR
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


@dataclass
class ProposalWindow:
    center_sec: float
    start_sec: float
    end_sec: float
    rms_score: float


def _envelope_rms(
    audio: np.ndarray,
    sr: int,
    *,
    hop_ms: float,
) -> tuple[np.ndarray, np.ndarray]:
    hop = max(1, int(sr * hop_ms / 1000))
    n_frames = max(1, (len(audio) + hop - 1) // hop)
    env: list[float] = []
    times: list[float] = []
    for i in range(n_frames):
        s0 = i * hop
        s1 = min(len(audio), s0 + hop)
        chunk = audio[s0:s1]
        env.append(float(np.sqrt(np.mean(chunk**2) + 1e-12)))
        times.append((s0 + s1) / 2.0 / sr)
    return np.asarray(times, dtype=np.float64), np.asarray(env, dtype=np.float64)


def _local_peak_indices(
    values: np.ndarray,
    *,
    min_score: float,
    min_distance_frames: int,
) -> list[int]:
    candidates: list[int] = []
    n = len(values)
    for i in range(n):
        if values[i] < min_score:
            continue
        left = values[i - 1] if i > 0 else -1.0
        right = values[i + 1] if i + 1 < n else -1.0
        if values[i] >= left and values[i] >= right:
            candidates.append(i)

    candidates.sort(key=lambda i: values[i], reverse=True)
    kept: list[int] = []
    for i in candidates:
        if all(abs(i - j) >= min_distance_frames for j in kept):
            kept.append(i)
    kept.sort()
    return kept


def merge_proposal_windows(
    windows: list[ProposalWindow],
    *,
    merge_gap_sec: float = 0.35,
    min_center_gap_sec: float | None = None,
) -> list[ProposalWindow]:
    \"\"\"
    Merge overlapping or nearly-adjacent proposal windows.

    Distinct transient centers at least ``min_center_gap_sec`` apart are kept
    separate even when padded windows touch. Without that guard, a quiet early
    cannon and a louder later rumble collapse into one late-centered proposal.
    \"\"\"
    if not windows:
        return []

    ordered = sorted(windows, key=lambda w: w.center_sec)
    merged: list[ProposalWindow] = [ordered[0]]
    for win in ordered[1:]:
        prev = merged[-1]
        centers_far = (
            min_center_gap_sec is not None
            and abs(win.center_sec - prev.center_sec) >= min_center_gap_sec
        )
        windows_touch = win.start_sec <= prev.end_sec + merge_gap_sec
        if windows_touch and not centers_far:
            merged[-1] = ProposalWindow(
                center_sec=win.center_sec
                if win.rms_score > prev.rms_score
                else prev.center_sec,
                start_sec=min(prev.start_sec, win.start_sec),
                end_sec=max(prev.end_sec, win.end_sec),
                rms_score=max(prev.rms_score, win.rms_score),
            )
        else:
            merged.append(win)
    return merged


def propose_onsets(
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
    *,
    sample_rate: int = INPUT_SR,
) -> list[ProposalWindow]:
    \"\"\"
    Find candidate transient times via short-hop RMS local maxima.

    Uses adaptive thresholding relative to the clip's RMS envelope.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)

    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate

    if len(audio) == 0:
        return []

    hop_ms = taxonomy.proposal_rms_hop_ms
    pad_sec = taxonomy.proposal_search_pad_sec
    min_dist_sec = taxonomy.impulsive_min_peak_distance_sec

    times, env = _envelope_rms(audio, sr, hop_ms=hop_ms)
    if env.size == 0:
        return []

    peak_val = float(np.max(env))
    if peak_val < 1e-8:
        return []

    threshold = peak_val * taxonomy.proposal_threshold_ratio
    hop_sec = hop_ms / 1000.0
    min_dist_frames = max(1, int(round(min_dist_sec / hop_sec)))

    peak_idxs = _local_peak_indices(
        env,
        min_score=threshold,
        min_distance_frames=min_dist_frames,
    )

    duration_sec = len(audio) / sr
    windows: list[ProposalWindow] = []
    for idx in peak_idxs:
        center = float(times[idx])
        windows.append(
            ProposalWindow(
                center_sec=center,
                start_sec=max(0.0, center - pad_sec),
                end_sec=min(duration_sec, center + pad_sec),
                rms_score=float(env[idx]),
            )
        )

    # Spectral-flux proposals catch sharp HF attacks that are quieter in RMS
    # (common for early muzzle blast vs later rumble / echo).
    from haptic_gt.context.onset_refine import _spectral_flux

    flux_times, flux = _spectral_flux(
        audio, sr, hop_ms=taxonomy.onset_flux_hop_ms
    )
    if flux.size > 0:
        flux_peak = float(np.max(flux))
        if flux_peak >= 1e-12:
            flux_thr = flux_peak * taxonomy.onset_flux_proposal_ratio
            flux_hop_sec = taxonomy.onset_flux_hop_ms / 1000.0
            flux_min_dist = max(1, int(round(min_dist_sec / flux_hop_sec)))
            flux_idxs = _local_peak_indices(
                flux,
                min_score=flux_thr,
                min_distance_frames=flux_min_dist,
            )
            for idx in flux_idxs:
                center = float(flux_times[idx])
                windows.append(
                    ProposalWindow(
                        center_sec=center,
                        start_sec=max(0.0, center - pad_sec),
                        end_sec=min(duration_sec, center + pad_sec),
                        # Rank flux peaks competitively with RMS scores
                        rms_score=float(flux[idx] / flux_peak) * peak_val,
                    )
                )

    return merge_proposal_windows(
        windows,
        min_center_gap_sec=min_dist_sec,
    )


def propose_sustained_scan(
    duration_sec: float,
    taxonomy: Taxonomy | None = None,
    *,
    pad_sec: float | None = None,
) -> list[ProposalWindow]:
    \"\"\"Uniform timeline windows for sustained categories (engine rumble, etc.).\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    hop = taxonomy.proposal_sustained_hop_sec
    pad = pad_sec if pad_sec is not None else taxonomy.proposal_search_pad_sec
    if duration_sec <= 0 or hop <= 0:
        return []

    centers: list[float] = []
    t = hop / 2.0
    while t < duration_sec:
        centers.append(t)
        t += hop

    return [
        ProposalWindow(
            center_sec=c,
            start_sec=max(0.0, c - pad),
            end_sec=min(duration_sec, c + pad),
            rms_score=0.0,
        )
        for c in centers
    ]


def _dedupe_nearby_windows(
    windows: list[ProposalWindow],
    *,
    min_center_gap_sec: float,
) -> list[ProposalWindow]:
    \"\"\"Keep highest-scoring window when centers are very close.\"\"\"
    if not windows:
        return []

    ordered = sorted(windows, key=lambda w: w.center_sec)
    kept: list[ProposalWindow] = [ordered[0]]
    for win in ordered[1:]:
        if win.center_sec - kept[-1].center_sec < min_center_gap_sec:
            if win.rms_score > kept[-1].rms_score:
                kept[-1] = win
        else:
            kept.append(win)
    return kept


def propose_all_windows(
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
    *,
    sample_rate: int = INPUT_SR,
    include_sustained_scan: bool = True,
) -> list[ProposalWindow]:
    \"\"\"
    Onset transients plus optional sustained timeline scan for classifiers.

    Onset peaks are preserved; sustained windows fill gaps for vehicle / activity.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    onset = propose_onsets(source_wav, taxonomy, sample_rate=sample_rate)
    if not include_sustained_scan:
        return onset

    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    duration_sec = len(audio) / (sr if sr else sample_rate)
    sustained = propose_sustained_scan(duration_sec, taxonomy)

    combined = _dedupe_nearby_windows(
        onset + sustained,
        min_center_gap_sec=0.35,
    )
    return combined
""",
    "context/rumble_calib.py": """\"\"\"Calibrate vehicle rumble detection against hand-marked peaks.\"\"\"

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.onset_refine import _envelope_rms
from haptic_gt.context.sustained_salience import (
    curve_value,
    local_rms,
    salience_stats,
    salience_threshold_curve,
    scene_spans,
)
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


@dataclass
class ManualPeakReport:
    manual_peak_sec: float
    matched_event_id: str | None
    matched_peak_sec: float | None
    match_error_sec: float | None
    detected_confidence: float | None
    audio_score: float | None
    local_rms: float | None
    scene_threshold: float
    rms_rise_ratio: float | None
    would_pass_salience: bool
    would_pass_rise_1_25: bool
    would_pass_rise_1_35: bool
    would_pass_rise_1_50: bool
    would_pass_rise_1_70: bool
    status: str  # "hit" | "miss"


@dataclass
class DetectedPeakReport:
    event_id: str
    peak_sec: float
    start_sec: float
    end_sec: float
    confidence: float
    audio_score: float | None
    nearest_manual_sec: float | None
    distance_sec: float | None
    local_rms: float | None
    rms_rise_ratio: float | None
    label: str  # "tp" | "fp" | "unknown"


def _rms_rise_at(
    times: np.ndarray,
    env: np.ndarray,
    center_sec: float,
    taxonomy: Taxonomy,
) -> float | None:
    pre = taxonomy.sustained_burst_pre_sec
    post = taxonomy.sustained_burst_post_sec
    pre_mask = (times >= center_sec - pre) & (times < center_sec - 0.05)
    burst_mask = (times >= center_sec - 0.15) & (times <= center_sec + post)
    if not np.any(burst_mask):
        return None
    peak_val = float(np.max(env[burst_mask]))
    if peak_val < 1e-8:
        return None
    if np.any(pre_mask):
        baseline = float(np.percentile(env[pre_mask], 60))
    else:
        baseline = float(np.percentile(env[burst_mask], 25))
    baseline = max(baseline, peak_val * 1e-3)
    return peak_val / baseline


def calibrate_rumble_thresholds(
    source_wav: str | Path,
    manual_peaks_sec: list[float],
    events_json: str | Path | dict | None = None,
    *,
    match_tolerance_sec: float = 1.0,
    taxonomy: Taxonomy | None = None,
) -> dict:
    \"\"\"
    Compare hand-marked rumble times to detected events + local RMS.

    A mark is a hit if any vehicle span covers it (or peak is within
    ``match_tolerance_sec``). Use ``local_rms`` / ``would_pass_salience``
    to check the loudness gate.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    times, env = _envelope_rms(audio, sr, hop_ms=taxonomy.onset_flux_hop_ms)

    events: list[dict] = []
    if events_json is not None:
        if isinstance(events_json, dict):
            payload = events_json
        else:
            payload = json.loads(Path(events_json).read_text(encoding="utf-8"))
        events = [
            e
            for e in payload.get("events", [])
            if str(e.get("category", "")).lower() == "vehicle"
        ]

    stats = salience_stats(env, taxonomy)
    thr = float(stats["threshold"])
    # The detector gates on a floor per scene, so a single clip-wide number can
    # say a mark passes while the scene it lives in rejects it.
    thr_curve = salience_threshold_curve(times, env, taxonomy)
    scenes = scene_spans(times, env, taxonomy)
    peak_win = taxonomy.sustained_salience_peak_win_sec

    def thr_at(t_sec: float) -> float:
        return curve_value(times, thr_curve, t_sec)

    def _covers(ev: dict, mark: float) -> float | None:
        peak = float(ev.get("peak_sec", ev.get("start_sec", 0.0)))
        start = float(ev.get("start_sec", peak))
        end = float(ev.get("end_sec", peak))
        if start - 0.2 <= mark <= end + 0.2:
            return 0.0
        err = abs(peak - mark)
        if err <= match_tolerance_sec:
            return err
        return None

    manual_rows: list[ManualPeakReport] = []
    for m in sorted(float(x) for x in manual_peaks_sec):
        rise = _rms_rise_at(times, env, m, taxonomy)
        rms = local_rms(times, env, m, half_win_sec=peak_win)
        best_i = None
        best_err = None
        for i, ev in enumerate(events):
            err = _covers(ev, m)
            if err is None:
                continue
            if best_err is None or err < best_err:
                best_err = err
                best_i = i
        if best_i is not None and best_err is not None:
            ev = events[best_i]
            peak = float(ev.get("peak_sec", m))
            manual_rows.append(
                ManualPeakReport(
                    manual_peak_sec=m,
                    matched_event_id=str(ev.get("event_id")),
                    matched_peak_sec=peak,
                    match_error_sec=round(best_err, 3),
                    detected_confidence=float(ev.get("confidence", 0.0)),
                    audio_score=(
                        float(ev["audio_score"])
                        if ev.get("audio_score") is not None
                        else None
                    ),
                    local_rms=round(rms, 4),
                    scene_threshold=round(thr_at(m), 4),
                    rms_rise_ratio=None if rise is None else round(rise, 3),
                    would_pass_salience=bool(rms >= thr_at(m)),
                    would_pass_rise_1_25=bool(rise is not None and rise >= 1.25),
                    would_pass_rise_1_35=bool(rise is not None and rise >= 1.35),
                    would_pass_rise_1_50=bool(rise is not None and rise >= 1.50),
                    would_pass_rise_1_70=bool(rise is not None and rise >= 1.70),
                    status="hit",
                )
            )
        else:
            manual_rows.append(
                ManualPeakReport(
                    manual_peak_sec=m,
                    matched_event_id=None,
                    matched_peak_sec=None,
                    match_error_sec=None,
                    detected_confidence=None,
                    audio_score=None,
                    local_rms=round(rms, 4),
                    scene_threshold=round(thr_at(m), 4),
                    rms_rise_ratio=None if rise is None else round(rise, 3),
                    would_pass_salience=bool(rms >= thr_at(m)),
                    would_pass_rise_1_25=bool(rise is not None and rise >= 1.25),
                    would_pass_rise_1_35=bool(rise is not None and rise >= 1.35),
                    would_pass_rise_1_50=bool(rise is not None and rise >= 1.50),
                    would_pass_rise_1_70=bool(rise is not None and rise >= 1.70),
                    status="miss",
                )
            )

    detected_rows: list[DetectedPeakReport] = []
    manuals = [float(x) for x in manual_peaks_sec]
    for i, ev in enumerate(events):
        peak = float(ev.get("peak_sec", ev.get("start_sec", 0.0)))
        start = float(ev.get("start_sec", peak))
        end = float(ev.get("end_sec", peak))
        rise = _rms_rise_at(times, env, peak, taxonomy)
        rms = local_rms(times, env, peak, half_win_sec=peak_win)
        if manuals:
            nearest = min(manuals, key=lambda m: abs(m - peak))
            dist = abs(nearest - peak)
            covered = any(start - 0.2 <= m <= end + 0.2 for m in manuals)
            label = "tp" if covered or dist <= match_tolerance_sec else "fp"
        else:
            nearest, dist, label = None, None, "unknown"
        detected_rows.append(
            DetectedPeakReport(
                event_id=str(ev.get("event_id", f"event_{i+1:03d}")),
                peak_sec=peak,
                start_sec=start,
                end_sec=end,
                confidence=float(ev.get("confidence", 0.0)),
                audio_score=(
                    float(ev["audio_score"]) if ev.get("audio_score") is not None else None
                ),
                nearest_manual_sec=nearest,
                distance_sec=None if dist is None else round(dist, 3),
                local_rms=round(rms, 4),
                rms_rise_ratio=None if rise is None else round(rise, 3),
                label=label,
            )
        )

    hits = sum(1 for r in manual_rows if r.status == "hit")
    misses = sum(1 for r in manual_rows if r.status == "miss")
    fps = sum(1 for r in detected_rows if r.label == "fp")
    report = {
        "summary": {
            "manual_count": len(manual_rows),
            "hits": hits,
            "misses": misses,
            "false_positives": fps,
            "salience_threshold": round(thr, 4),
            "salience_unimodal": bool(stats["unimodal"]),
            "scenes": [
                {
                    "start_sec": round(lo, 2),
                    "end_sec": round(hi, 2),
                    "threshold": round(thr_at((lo + hi) / 2), 4),
                }
                for lo, hi in scenes
            ],
            "current_rise_ratio_setting": taxonomy.sustained_burst_rise_ratio,
            "sustained_encoder_threshold": taxonomy.sustained_encoder_threshold,
            "hint": (
                "Hits use span coverage (a 29–32s island hits 29, 30, 31, 31.9). "
                "would_pass_salience compares each mark to its own scene's floor, "
                "which is what the detector gates on; salience_threshold is the "
                "clip-wide number and only applies to single-scene clips."
            ),
        },
        "manual": [asdict(r) for r in manual_rows],
        "detected": [asdict(r) for r in detected_rows],
    }
    return report


@dataclass
class TimelineSample:
    t_sec: float
    local_rms: float
    rms_rise_ratio: float | None
    nearest_detected_peak_sec: float | None
    nearest_detected_dist_sec: float | None
    nearest_manual_dist_sec: float | None
    in_manual_window: bool
    scene_threshold: float
    above_salience: bool
    would_pass_1_10: bool
    would_pass_1_25: bool
    would_pass_1_35: bool


def scan_rumble_timeline(
    source_wav: str | Path,
    *,
    start_sec: float = 28.0,
    end_sec: float = 45.0,
    hop_sec: float = 0.1,
    events_json: str | Path | dict | None = None,
    manual_peaks_sec: list[float] | None = None,
    manual_window_sec: float = 0.4,
    taxonomy: Taxonomy | None = None,
) -> dict:
    \"\"\"
    Dense timeline scan (default 100 ms) of local RMS + rise.

    Auto rumble follows ``above_salience`` (absolute loudness), not rise.
    ``hop_sec=0.1`` is 28.0, 28.1, … (set 0.01 for 10 ms).
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    duration = len(audio) / sr
    times, env = _envelope_rms(audio, sr, hop_ms=taxonomy.onset_flux_hop_ms)

    detected_peaks: list[float] = []
    if events_json is not None:
        if isinstance(events_json, dict):
            payload = events_json
        else:
            payload = json.loads(Path(events_json).read_text(encoding="utf-8"))
        detected_peaks = [
            float(e.get("peak_sec", e.get("start_sec", 0.0)))
            for e in payload.get("events", [])
            if str(e.get("category", "")).lower() == "vehicle"
        ]

    stats = salience_stats(env, taxonomy)
    thr = float(stats["threshold"])
    thr_curve = salience_threshold_curve(times, env, taxonomy)
    scenes = scene_spans(times, env, taxonomy)
    manuals = [float(x) for x in (manual_peaks_sec or [])]
    t0 = max(0.0, float(start_sec))
    t1 = min(duration, float(end_sec))
    hop = max(0.01, float(hop_sec))
    samples: list[TimelineSample] = []
    t = t0
    while t <= t1 + 1e-9:
        rms_val = float(np.interp(t, times, env)) if env.size else 0.0
        rise = _rms_rise_at(times, env, t, taxonomy)
        near_det = None
        dist_det = None
        if detected_peaks:
            near_det = min(detected_peaks, key=lambda p: abs(p - t))
            dist_det = abs(near_det - t)
        dist_man = None
        in_man = False
        if manuals:
            nearest_m = min(manuals, key=lambda m: abs(m - t))
            dist_man = abs(nearest_m - t)
            in_man = dist_man <= manual_window_sec
        samples.append(
            TimelineSample(
                t_sec=round(t, 3),
                local_rms=round(rms_val, 6),
                rms_rise_ratio=None if rise is None else round(rise, 3),
                nearest_detected_peak_sec=near_det,
                nearest_detected_dist_sec=None if dist_det is None else round(dist_det, 3),
                nearest_manual_dist_sec=None if dist_man is None else round(dist_man, 3),
                in_manual_window=in_man,
                scene_threshold=round(curve_value(times, thr_curve, t), 4),
                above_salience=bool(rms_val >= curve_value(times, thr_curve, t)),
                would_pass_1_10=bool(rise is not None and rise >= 1.10),
                would_pass_1_25=bool(rise is not None and rise >= 1.25),
                would_pass_1_35=bool(rise is not None and rise >= 1.35),
            )
        )
        t += hop

    return {
        "summary": {
            "start_sec": t0,
            "end_sec": t1,
            "hop_sec": hop,
            "n_samples": len(samples),
            "n_in_manual_window": sum(1 for s in samples if s.in_manual_window),
            "salience_threshold": round(thr, 4),
            "salience_unimodal": bool(stats["unimodal"]),
            "scenes": [
                {
                    "start_sec": round(lo, 2),
                    "end_sec": round(hi, 2),
                    "threshold": round(curve_value(times, thr_curve, (lo + hi) / 2), 4),
                }
                for lo, hi in scenes
            ],
            "max_rise": max((s.rms_rise_ratio or 0.0) for s in samples) if samples else 0.0,
            "hint": (
                "Auto rumble follows above_salience (local RMS vs its own scene's "
                "floor), not rise. Green marks should sit in high-RMS regions."
            ),
        },
        "samples": [asdict(s) for s in samples],
    }


def format_timeline_scan(report: dict, *, only_manual_windows: bool = False) -> str:
    \"\"\"Plain-text dense scan. Optionally only rows near manual peaks.\"\"\"
    lines = [
        "=== Dense rumble scan ===",
        json.dumps(report["summary"], indent=2),
        "",
        f"{'t':>7} {'rms':>8} {'loud?':>5} {'rise':>6} {'man?':>4} {'d_man':>6} {'d_det':>6}",
    ]
    for s in report["samples"]:
        if only_manual_windows and not s["in_manual_window"]:
            continue
        rise = "-" if s["rms_rise_ratio"] is None else f"{s['rms_rise_ratio']:.2f}"
        dman = "-" if s["nearest_manual_dist_sec"] is None else f"{s['nearest_manual_dist_sec']:.2f}"
        ddet = "-" if s["nearest_detected_dist_sec"] is None else f"{s['nearest_detected_dist_sec']:.2f}"
        lines.append(
            f"{s['t_sec']:7.1f} {s['local_rms']:8.4f} "
            f"{'Y' if s.get('above_salience') else '-':>5} {rise:>6} "
            f"{'Y' if s['in_manual_window'] else '-':>4} {dman:>6} {ddet:>6}"
        )
    return "\\n".join(lines)


def format_calibration_table(report: dict) -> str:
    \"\"\"Plain-text table for Colab / terminal.\"\"\"
    lines = [
        "=== Rumble threshold calibration ===",
        json.dumps(report["summary"], indent=2),
        "",
        "Manual peaks:",
        f"{'manual':>8} {'status':<6} {'rms':>7} {'floor':>7} {'loud?':>5} "
        f"{'rise':>6} {'conf':>6} {'err':>6}",
    ]
    for r in report["manual"]:
        rise = "-" if r["rms_rise_ratio"] is None else f"{r['rms_rise_ratio']:.2f}"
        conf = "-" if r["detected_confidence"] is None else f"{r['detected_confidence']:.2f}"
        err = "-" if r["match_error_sec"] is None else f"{r['match_error_sec']:.2f}"
        rms = "-" if r.get("local_rms") is None else f"{r['local_rms']:.3f}"
        floor = "-" if r.get("scene_threshold") is None else f"{r['scene_threshold']:.3f}"
        lines.append(
            f"{r['manual_peak_sec']:8.2f} {r['status']:<6} {rms:>7} {floor:>7} "
            f"{'Y' if r.get('would_pass_salience') else '-':>5} {rise:>6} "
            f"{conf:>6} {err:>6}"
        )
    lines.extend(
        [
            "",
            "Detected vehicle events:",
            f"{'id':<12} {'start':>8} {'peak':>8} {'end':>8} {'label':<4} "
            f"{'rms':>7} {'conf':>6}",
        ]
    )
    for r in report["detected"]:
        rms = "-" if r.get("local_rms") is None else f"{r['local_rms']:.3f}"
        lines.append(
            f"{r['event_id']:<12} {r.get('start_sec', r['peak_sec']):8.2f} "
            f"{r['peak_sec']:8.2f} {r.get('end_sec', r['peak_sec']):8.2f} "
            f"{r['label']:<4} {rms:>7} {r['confidence']:6.2f}"
        )
    return "\\n".join(lines)
""",
    "context/rumble_filter.py": """\"\"\"Keep vehicle rumble where the clip is loud; drop quiet engine-bed FPs.\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.onset_refine import _envelope_rms
from haptic_gt.context.sustained_salience import apply_vehicle_salience
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def _has_energy_burst(
    times: np.ndarray,
    env: np.ndarray,
    *,
    center_sec: float,
    pre_sec: float,
    post_sec: float,
    rise_ratio: float,
) -> tuple[bool, float, float]:
    \"\"\"
    Return (ok, onset_sec, end_sec) if energy rises vs a pre-window baseline.

    Steady engine bed fails; a clear rumble swell / burst passes.
    \"\"\"
    if env.size == 0 or times.size == 0:
        return False, center_sec, center_sec

    pre_mask = (times >= center_sec - pre_sec) & (times < center_sec - 0.05)
    burst_mask = (times >= center_sec - 0.15) & (times <= center_sec + post_sec)
    if not np.any(burst_mask):
        return False, center_sec, center_sec

    burst_env = env[burst_mask]
    burst_times = times[burst_mask]
    peak_val = float(np.max(burst_env))
    if peak_val < 1e-8:
        return False, center_sec, center_sec

    if np.any(pre_mask):
        # Robust baseline: ignore short dips in the idle bed
        baseline = float(np.percentile(env[pre_mask], 60))
    else:
        baseline = float(np.percentile(burst_env, 25))

    baseline = max(baseline, peak_val * 1e-3)
    if peak_val < baseline * rise_ratio:
        return False, center_sec, center_sec

    # Onset: first frame that climbs well above baseline
    onset_thr = baseline + 0.35 * (peak_val - baseline)
    onset_idx = int(np.argmax(burst_env))  # fallback = peak
    for i, v in enumerate(burst_env):
        if v >= onset_thr:
            onset_idx = i
            break
    onset_sec = float(burst_times[onset_idx])

    # End: after peak, when energy falls near baseline again
    peak_i = int(np.argmax(burst_env))
    end_thr = baseline + 0.25 * (peak_val - baseline)
    end_idx = peak_i
    for i in range(peak_i, len(burst_env)):
        end_idx = i
        if burst_env[i] <= end_thr:
            break
    end_sec = float(burst_times[end_idx])
    if end_sec <= onset_sec:
        end_sec = onset_sec + 0.25
    return True, onset_sec, end_sec


def filter_sustained_rumble_bursts(
    events: list[DetectedEvent],
    source_wav: str | Path,
    taxonomy: Taxonomy | None = None,
    *,
    report: dict | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Gate vehicle rumble on absolute local RMS (salience), not RMS rise.

    Quiet high-rise bumps (engine bed / idle ticks) are dropped. Long loud
    plateaus AST missed are filled so auto-detect can catch marked rumbles
    on an unseen video. Impulsive events pass through unchanged.

    Pass ``report`` to record the loudness floor per scene and every proposal the
    gate dropped, so a missing rumble can be told apart from one the classifier
    never proposed.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    hop_ms = taxonomy.onset_flux_hop_ms
    times, env = _envelope_rms(audio, sr, hop_ms=hop_ms)
    duration_sec = len(audio) / float(sr)
    return apply_vehicle_salience(
        events, times, env, taxonomy, duration_sec=duration_sec, report=report
    )
""",
    "context/sed_events.py": """\"\"\"Events from frame-level posteriors: median filter + hysteresis thresholding.

This is the standard DCASE sound-event-detection decoding step. A category is
"on" once its posterior crosses the high threshold and stays on until it falls
below the low threshold, which stops one noisy frame from chopping an event in
two (or from starting one).
\"\"\"

from __future__ import annotations

import numpy as np

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.sed_frames import FramePosteriors
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def median_filter(values: np.ndarray, k: int) -> np.ndarray:
    if values.size == 0 or k <= 1:
        return values
    k = int(k) | 1
    pad = k // 2
    padded = np.pad(values, pad, mode="edge")
    out = np.empty_like(values)
    for i in range(values.size):
        out[i] = float(np.median(padded[i : i + k]))
    return out


def hysteresis_segments(
    values: np.ndarray,
    high: float,
    low: float,
) -> list[tuple[int, int]]:
    \"\"\"Frame index spans that cross ``high`` and stay above ``low``.\"\"\"
    segments: list[tuple[int, int]] = []
    n = values.size
    i = 0
    while i < n:
        if values[i] < high:
            i += 1
            continue
        start = i
        while start > 0 and values[start - 1] >= low:
            start -= 1
        end = i
        while end + 1 < n and values[end + 1] >= low:
            end += 1
        if not segments or start > segments[-1][1]:
            segments.append((start, end))
        else:
            segments[-1] = (segments[-1][0], max(segments[-1][1], end))
        i = end + 1
    return segments


def _thresholds(taxonomy: Taxonomy, category: str) -> tuple[float, float]:
    cfg = taxonomy.categories.get(category)
    high = taxonomy.sed_onset_high
    low = taxonomy.sed_onset_low
    if cfg is not None:
        if cfg.sed_high is not None:
            high = float(cfg.sed_high)
        if cfg.sed_low is not None:
            low = float(cfg.sed_low)
    return high, max(min(low, high), 0.0)


def events_from_frame_posteriors(
    frames: FramePosteriors,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"Decode frame posteriors into events with start / peak / end.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    if frames.times.size == 0:
        return []

    hop = max(frames.hop_sec, 1e-4)
    events: list[DetectedEvent] = []

    for category, raw in frames.scores.items():
        cfg = taxonomy.categories.get(category)
        if cfg is None or raw.size == 0:
            continue
        impulsive = bool(cfg.impulsive)
        med_sec = (
            taxonomy.sed_median_impulsive_sec
            if impulsive
            else taxonomy.sed_median_sustained_sec
        )
        smooth = median_filter(raw, max(1, int(round(med_sec / hop))))
        high, low = _thresholds(taxonomy, category)
        min_sec = (
            taxonomy.sed_min_event_sec_impulsive
            if impulsive
            else taxonomy.sed_min_event_sec_sustained
        )

        spans = hysteresis_segments(smooth, high, low)
        merged: list[tuple[int, int]] = []
        gap_frames = max(1, int(round(taxonomy.sed_merge_gap_sec / hop)))
        for s, e in spans:
            if merged and s - merged[-1][1] <= gap_frames:
                merged[-1] = (merged[-1][0], e)
            else:
                merged.append((s, e))

        label = cfg.audioset_labels[0] if cfg.audioset_labels else category
        for s, e in merged:
            start = float(frames.times[s]) - hop / 2.0
            end = float(frames.times[e]) + hop / 2.0
            if end - start < min_sec:
                pad = (min_sec - (end - start)) / 2.0
                start -= pad
                end += pad
            start = max(0.0, start)
            end = min(frames.duration_sec, max(end, start + min_sec))
            peak_i = s + int(np.argmax(smooth[s : e + 1]))
            events.append(
                DetectedEvent(
                    category=category,
                    label=label,
                    start_sec=start,
                    peak_sec=float(np.clip(frames.times[peak_i], start, end)),
                    end_sec=end,
                    confidence=float(np.max(smooth[s : e + 1])),
                    context_token=False,
                    audio_score=float(np.max(raw[s : e + 1])),
                    # Video is intentionally not fused; see encoders.classify_video_frames
                    video_score=None,
                    sources=["audio", "sed"],
                )
            )

    events.sort(key=lambda ev: ev.start_sec)
    return events


def split_impulsive_by_posterior_peaks(
    events: list[DetectedEvent],
    frames: FramePosteriors,
    taxonomy: Taxonomy | None = None,
) -> list[DetectedEvent]:
    \"\"\"Split one long impulsive span into one event per posterior peak.

    A volley of shots can stay above threshold continuously; DCASE decoding
    would report a single event, but each shot needs its own haptic accent.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    hop = max(frames.hop_sec, 1e-4)
    min_dist_frames = max(1, int(round(taxonomy.impulsive_min_peak_distance_sec / hop)))
    out: list[DetectedEvent] = []

    for ev in events:
        cfg = taxonomy.categories.get(ev.category)
        arr = frames.scores.get(ev.category)
        half = taxonomy.impulsive_event_half_width_sec
        if cfg is None or not cfg.impulsive or arr is None:
            out.append(ev)
            continue
        if ev.end_sec - ev.start_sec <= 2 * half:
            out.append(ev)
            continue

        mask = (frames.times >= ev.start_sec) & (frames.times <= ev.end_sec)
        idxs = np.flatnonzero(mask)
        if idxs.size < 3:
            out.append(ev)
            continue
        window = arr[idxs]
        # A flat above-threshold plateau is not a set of peaks: require a strict
        # rise and a value close to the strongest peak in the span.
        floor = float(np.max(window)) * taxonomy.sed_peak_rel
        local: list[int] = []
        for i in range(1, window.size - 1):
            if window[i] < floor:
                continue
            if window[i] > window[i - 1] and window[i] >= window[i + 1]:
                local.append(i)
        local.sort(key=lambda i: float(window[i]), reverse=True)
        kept: list[int] = []
        for i in local:
            if all(abs(i - j) >= min_dist_frames for j in kept):
                kept.append(i)
        if len(kept) <= 1:
            out.append(ev)
            continue

        for i in sorted(kept):
            peak_t = float(frames.times[idxs[i]])
            out.append(
                DetectedEvent(
                    category=ev.category,
                    label=ev.label,
                    start_sec=max(ev.start_sec, peak_t - taxonomy.impulsive_pre_roll_sec),
                    peak_sec=peak_t,
                    end_sec=min(ev.end_sec, peak_t + half),
                    confidence=float(window[i]),
                    context_token=ev.context_token,
                    audio_score=float(window[i]),
                    video_score=None,
                    sources=list(ev.sources),
                )
            )

    out.sort(key=lambda ev: ev.start_sec)
    return out
""",
    "context/sed_frames.py": """\"\"\"Frame-level sound event posteriors (DCASE-style), not clip-level tagging.

Window tagging gives one score per ~1 s window, so onsets can only be located
to about a second — that is why timing came from spectral flux alone before.
Here every category gets a posterior on a fixed frame grid, so onsets come from
the model's own time axis.

Backends
--------
``ast_dense``
    AST (AudioSet) evaluated on densely overlapping windows (hop ``sed_hop_sec``),
    using all 527 sigmoid outputs instead of a truncated top-k. Resolution is the
    hop; effective smoothing is the window length. No extra download.
``panns``
    PANNs ``Cnn14_DecisionLevelAtt`` framewise output (true frame-level SED, the
    DCASE-style option). Used when ``panns_inference`` is installed.
\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np

from haptic_gt.context.encoders import EncoderScore
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy, match_label_to_category

AST_SR = 16_000
PANNS_SR = 32_000

_ast_model: Any | None = None
_ast_extractor: Any | None = None
_panns_model: Any | None = None


@dataclass
class FramePosteriors:
    \"\"\"Per-category posterior on a uniform frame grid.\"\"\"

    times: np.ndarray  # [T] frame center times (s)
    scores: dict[str, np.ndarray]  # category -> [T] posterior
    hop_sec: float
    backend: str
    duration_sec: float

    def category_names(self) -> list[str]:
        return sorted(self.scores)

    def at(self, category: str, t: float) -> float:
        arr = self.scores.get(category)
        if arr is None or arr.size == 0 or self.times.size == 0:
            return 0.0
        return float(arr[int(np.argmin(np.abs(self.times - t)))])


def _device_str() -> str:
    try:
        import torch

        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


def _get_ast():
    \"\"\"Load AST model + feature extractor directly (all logits, batched).\"\"\"
    global _ast_model, _ast_extractor
    if _ast_model is None or _ast_extractor is None:
        import torch
        from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

        from haptic_gt.context.encoders import AST_MODEL_ID

        _ast_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)
        _ast_model = AutoModelForAudioClassification.from_pretrained(AST_MODEL_ID)
        _ast_model.eval()
        _ast_model.to(_device_str())
        for p in _ast_model.parameters():
            p.requires_grad = False
        torch.set_grad_enabled(False)
    return _ast_model, _ast_extractor


def _category_label_index(taxonomy: Taxonomy, id2label: dict) -> dict[str, list[int]]:
    \"\"\"Map each taxonomy category to the model output indices that feed it.\"\"\"
    out: dict[str, list[int]] = {name: [] for name in taxonomy.categories}
    for idx, label in id2label.items():
        cat = match_label_to_category(taxonomy, str(label), "audio")
        if cat is not None:
            out[cat].append(int(idx))
    return {k: v for k, v in out.items() if v}


def _ast_dense_posteriors(
    audio_16k: np.ndarray,
    taxonomy: Taxonomy,
    *,
    batch_size: int = 12,
) -> FramePosteriors:
    import torch

    model, extractor = _get_ast()
    device = _device_str()
    hop = taxonomy.sed_hop_sec
    win = taxonomy.sed_window_sec
    duration = len(audio_16k) / float(AST_SR)

    centers: list[float] = []
    clips: list[np.ndarray] = []
    t = 0.0
    while t <= max(duration - 1e-6, 0.0):
        s0 = int(max(0.0, t - win / 2.0) * AST_SR)
        s1 = int(min(duration, t + win / 2.0) * AST_SR)
        clip = audio_16k[s0:s1]
        if clip.size >= AST_SR // 20:
            centers.append(t)
            clips.append(clip.astype(np.float32))
        t += hop

    cat_idx = _category_label_index(taxonomy, model.config.id2label)
    scores: dict[str, list[float]] = {c: [] for c in cat_idx}

    for i in range(0, len(clips), batch_size):
        batch = clips[i : i + batch_size]
        inputs = extractor(
            batch, sampling_rate=AST_SR, return_tensors="pt", padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits = model(**inputs).logits
        # AudioSet head is multi-label: sigmoid, and keep every class
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        for row in probs:
            for cat, idxs in cat_idx.items():
                scores[cat].append(float(np.max(row[idxs])))

    return FramePosteriors(
        times=np.asarray(centers, dtype=np.float64),
        scores={c: np.asarray(v, dtype=np.float64) for c, v in scores.items()},
        hop_sec=hop,
        backend="ast_dense",
        duration_sec=duration,
    )


def _panns_posteriors(audio_16k: np.ndarray, taxonomy: Taxonomy) -> FramePosteriors:
    \"\"\"True framewise SED via PANNs Cnn14_DecisionLevelAtt (~10 ms frames).\"\"\"
    global _panns_model
    import librosa
    from panns_inference import SoundEventDetection, labels

    if _panns_model is None:
        _panns_model = SoundEventDetection(checkpoint_path=None, device=_device_str())

    audio_32k = librosa.resample(
        audio_16k.astype(np.float32), orig_sr=AST_SR, target_sr=PANNS_SR
    )
    framewise = _panns_model.inference(audio_32k[None, :])[0]  # [T, 527]
    duration = len(audio_16k) / float(AST_SR)
    n_frames = framewise.shape[0]
    hop = duration / max(n_frames, 1)
    times = (np.arange(n_frames) + 0.5) * hop

    id2label = {i: lab for i, lab in enumerate(labels)}
    cat_idx = _category_label_index(taxonomy, id2label)
    scores = {
        cat: np.max(framewise[:, idxs], axis=1).astype(np.float64)
        for cat, idxs in cat_idx.items()
    }
    return FramePosteriors(
        times=times,
        scores=scores,
        hop_sec=hop,
        backend="panns",
        duration_sec=duration,
    )


def compute_frame_posteriors(
    audio_16k: np.ndarray,
    taxonomy: Taxonomy | None = None,
    *,
    backend: str | None = None,
) -> FramePosteriors:
    \"\"\"Frame-level posteriors per taxonomy category.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    backend = backend or taxonomy.sed_backend
    if backend in ("auto", "panns"):
        try:
            return _panns_posteriors(audio_16k, taxonomy)
        except Exception:
            if backend == "panns":
                raise
    return _ast_dense_posteriors(audio_16k, taxonomy)


def posteriors_to_encoder_scores(
    frames: FramePosteriors,
    taxonomy: Taxonomy | None = None,
) -> list[EncoderScore]:
    \"\"\"Frame posteriors as sparse scores, for stages that expect AST hits.

    Each category is represented by its first taxonomy label so existing
    label-to-category matching keeps working.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    out: list[EncoderScore] = []
    for cat, arr in frames.scores.items():
        cfg = taxonomy.categories.get(cat)
        label = cfg.audioset_labels[0] if cfg and cfg.audioset_labels else cat
        for t, v in zip(frames.times, arr):
            if v <= 0.01:
                continue
            out.append(
                EncoderScore(
                    time_sec=float(t), label=label, score=float(v), source="audio"
                )
            )
    out.sort(key=lambda s: s.time_sec)
    return out
""",
    "context/shot_calib.py": """\"\"\"Calibrate impulsive (cannon / gunshot) detection against hand-marked times.

Mirrors ``rumble_calib`` for blasts: for every marked shot it reports the
nearest spectral-flux attack, how loud that attack is relative to the clip's
confirmed shots, and whether an event matched. Use it to decide whether a
timing error is in the detector or in the mark.
\"\"\"

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import soundfile as sf

from haptic_gt.context.onset_refine import _spectral_flux, local_flux_ratio
from haptic_gt.context.proposals import _local_peak_indices
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy

IMPULSIVE_CATEGORIES = ("explosion", "gunshot", "weather")
CONFIRMED_CONF = 0.55


@dataclass
class ManualShotReport:
    manual_sec: float
    matched_event_id: str | None
    matched_peak_sec: float | None
    match_error_sec: float | None
    detected_confidence: float | None
    nearest_attack_sec: float | None
    attack_offset_sec: float | None
    flux_rel_shot_level: float | None
    prominence: float | None
    status: str  # "hit" | "miss"


@dataclass
class DetectedShotReport:
    event_id: str
    peak_sec: float
    confidence: float
    audio_score: float | None
    nearest_manual_sec: float | None
    distance_sec: float | None
    flux_rel_shot_level: float | None
    prominence: float | None
    label: str  # "tp" | "fp"


def _load_events(events_json: str | Path | dict | None) -> list[dict]:
    if events_json is None:
        return []
    if isinstance(events_json, dict):
        payload = events_json
    else:
        payload = json.loads(Path(events_json).read_text(encoding="utf-8"))
    return [
        e
        for e in payload.get("events", [])
        if str(e.get("category", "")).lower() in IMPULSIVE_CATEGORIES
    ]


def _flux_profile(
    source_wav: Path, taxonomy: Taxonomy
) -> tuple[np.ndarray, np.ndarray]:
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    return _spectral_flux(audio, sr, hop_ms=taxonomy.onset_flux_hop_ms)


def _flux_at(times: np.ndarray, flux: np.ndarray, t: float) -> float:
    if flux.size == 0:
        return 0.0
    return float(flux[int(np.argmin(np.abs(times - t)))])


def _shot_level(
    times: np.ndarray, flux: np.ndarray, events: list[dict]
) -> tuple[float, bool]:
    \"\"\"Reference attack strength of this clip's confident blasts.\"\"\"
    levels = [
        _flux_at(times, flux, float(e.get("peak_sec", 0.0)))
        for e in events
        if float(e.get("confidence", 0.0)) >= CONFIRMED_CONF
    ]
    levels = [v for v in levels if v > 0.0]
    if levels:
        return float(np.median(levels)), True
    peak = float(np.max(flux)) if flux.size else 0.0
    return peak * 0.35, False


def _nearest_attack(
    times: np.ndarray,
    flux: np.ndarray,
    peak_idxs: list[int],
    t: float,
) -> float | None:
    if not peak_idxs:
        return None
    cand = min(peak_idxs, key=lambda i: abs(float(times[i]) - t))
    return float(times[cand])


def calibrate_shot_times(
    source_wav: str | Path,
    manual_shots_sec: list[float],
    events_json: str | Path | dict | None = None,
    *,
    match_tolerance_sec: float = 0.35,
    taxonomy: Taxonomy | None = None,
) -> dict:
    \"\"\"Compare hand-marked shot times to detected impulsive events + flux attacks.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    times, flux = _flux_profile(source_wav, taxonomy)
    events = _load_events(events_json)
    shot_level, confirmed = _shot_level(times, flux, events)

    hop_sec = taxonomy.onset_flux_hop_ms / 1000.0
    min_frames = max(1, int(round(taxonomy.impulsive_min_peak_distance_sec / hop_sec)))
    flux_peak = float(np.max(flux)) if flux.size else 0.0
    peak_idxs = _local_peak_indices(
        flux,
        min_score=flux_peak * 0.05,
        min_distance_frames=min_frames,
    )

    def _rel(t: float) -> float | None:
        if shot_level <= 0.0:
            return None
        return round(_flux_at(times, flux, t) / shot_level, 3)

    manual_rows: list[ManualShotReport] = []
    used: set[str] = set()
    for mark in sorted(manual_shots_sec):
        best: tuple[float, dict] | None = None
        for ev in events:
            err = abs(float(ev.get("peak_sec", 0.0)) - mark)
            if err <= match_tolerance_sec and (best is None or err < best[0]):
                best = (err, ev)
        attack = _nearest_attack(times, flux, peak_idxs, mark)
        row = ManualShotReport(
            manual_sec=round(mark, 3),
            matched_event_id=None,
            matched_peak_sec=None,
            match_error_sec=None,
            detected_confidence=None,
            nearest_attack_sec=None if attack is None else round(attack, 3),
            attack_offset_sec=None if attack is None else round(attack - mark, 3),
            flux_rel_shot_level=_rel(mark if attack is None else attack),
            prominence=round(
                local_flux_ratio(times, flux, mark if attack is None else attack), 2
            ),
            status="miss",
        )
        if best is not None:
            err, ev = best
            row.matched_event_id = str(ev.get("event_id"))
            row.matched_peak_sec = round(float(ev.get("peak_sec", 0.0)), 3)
            row.match_error_sec = round(err, 3)
            row.detected_confidence = float(ev.get("confidence", 0.0))
            row.status = "hit"
            used.add(row.matched_event_id)
        manual_rows.append(row)

    detected_rows: list[DetectedShotReport] = []
    for ev in events:
        peak = float(ev.get("peak_sec", 0.0))
        nearest = (
            min(manual_shots_sec, key=lambda m: abs(m - peak))
            if manual_shots_sec
            else None
        )
        eid = str(ev.get("event_id"))
        detected_rows.append(
            DetectedShotReport(
                event_id=eid,
                peak_sec=round(peak, 3),
                confidence=float(ev.get("confidence", 0.0)),
                audio_score=ev.get("audio_score"),
                nearest_manual_sec=None if nearest is None else round(nearest, 3),
                distance_sec=None if nearest is None else round(abs(nearest - peak), 3),
                flux_rel_shot_level=_rel(peak),
                prominence=round(local_flux_ratio(times, flux, peak), 2),
                label="tp" if eid in used else "fp",
            )
        )

    hits = sum(1 for r in manual_rows if r.status == "hit")
    return {
        "summary": {
            "manual_count": len(manual_rows),
            "hits": hits,
            "misses": len(manual_rows) - hits,
            "detected_count": len(detected_rows),
            "false_positives": sum(1 for r in detected_rows if r.label == "fp"),
            "shot_level_flux": round(shot_level, 4),
            "shot_level_from_events": confirmed,
            "match_tolerance_sec": match_tolerance_sec,
        },
        "manual_shots": [asdict(r) for r in manual_rows],
        "detected_shots": [asdict(r) for r in detected_rows],
    }


def scan_shot_attacks(
    source_wav: str | Path,
    *,
    start_sec: float = 0.0,
    end_sec: float | None = None,
    top_n: int = 40,
    taxonomy: Taxonomy | None = None,
) -> dict:
    \"\"\"List the strongest flux attacks in a range — the clip's actual transients.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    source_wav = Path(source_wav)
    times, flux = _flux_profile(source_wav, taxonomy)
    if flux.size == 0:
        return {"summary": {"count": 0}, "attacks": []}

    hop_sec = taxonomy.onset_flux_hop_ms / 1000.0
    min_frames = max(1, int(round(taxonomy.impulsive_min_peak_distance_sec / hop_sec)))
    flux_peak = float(np.max(flux))
    idxs = _local_peak_indices(
        flux,
        min_score=flux_peak * 0.05,
        min_distance_frames=min_frames,
    )
    end_sec = float(times[-1]) if end_sec is None else end_sec

    rows = []
    for i in idxs:
        t = float(times[i])
        if t < start_sec or t > end_sec:
            continue
        rows.append(
            {
                "t_sec": round(t, 3),
                "flux": round(float(flux[i]), 4),
                "flux_rel_max": round(float(flux[i]) / flux_peak, 3),
                "prominence": round(local_flux_ratio(times, flux, t), 2),
            }
        )
    rows.sort(key=lambda r: r["flux"], reverse=True)
    rows = rows[:top_n]
    rows.sort(key=lambda r: r["t_sec"])
    return {
        "summary": {
            "count": len(rows),
            "flux_max": round(flux_peak, 4),
            "start_sec": start_sec,
            "end_sec": round(end_sec, 3),
        },
        "attacks": rows,
    }


def _num(value, digits: int = 3, width: int = 8, signed: bool = False) -> str:
    \"\"\"Fixed-width number or dash. Kept quote-free for Python 3.10 f-strings.\"\"\"
    if value is None:
        return "-".rjust(width)
    sign = "+" if signed else ""
    return format(float(value), sign + "." + str(digits) + "f").rjust(width)


def format_shot_calibration_table(report: dict) -> str:
    s = report["summary"]
    level_src = (
        "from events"
        if s["shot_level_from_events"]
        else "(fallback: 35% of clip max)"
    )
    hits = s["hits"]
    total = s["manual_count"]
    lines = [
        "Shots: {0}/{1} hits, {2} misses, {3} false positives (tolerance {4}s)".format(
            hits, total, s["misses"], s["false_positives"], s["match_tolerance_sec"]
        ),
        "Shot level (flux of confident blasts): {0} {1}".format(
            s["shot_level_flux"], level_src
        ),
        "",
        "   mark status      event     peak     err   attack  d(att)  flux/shot  promin",
    ]
    for r in report["manual_shots"]:
        lines.append(
            _num(r["manual_sec"], 2, 7)
            + r["status"].rjust(7)
            + (r["matched_event_id"] or "-").rjust(11)
            + _num(r["matched_peak_sec"], 3, 9)
            + _num(r["match_error_sec"], 3, 8)
            + _num(r["nearest_attack_sec"], 3, 9)
            + _num(r["attack_offset_sec"], 3, 8, signed=True)
            + _num(r["flux_rel_shot_level"], 3, 11)
            + _num(r["prominence"], 2, 8)
        )

    lines += [
        "",
        "     event     peak   conf  label  nearest    dist  flux/shot  promin",
    ]
    for r in report["detected_shots"]:
        lines.append(
            r["event_id"].rjust(10)
            + _num(r["peak_sec"], 3, 9)
            + _num(r["confidence"], 2, 7)
            + r["label"].rjust(7)
            + _num(r["nearest_manual_sec"], 2, 9)
            + _num(r["distance_sec"], 3, 8)
            + _num(r["flux_rel_shot_level"], 3, 11)
            + _num(r["prominence"], 2, 8)
        )
    return "\\n".join(lines)


def format_shot_scan(scan: dict) -> str:
    s = scan["summary"]
    lines = [
        "Strongest attacks {0}-{1}s (flux max {2}), {3} rows".format(
            s["start_sec"], s["end_sec"], s["flux_max"], s["count"]
        ),
        "   t_sec       flux  rel_max  promin",
    ]
    for r in scan["attacks"]:
        lines.append(
            _num(r["t_sec"], 3, 8)
            + _num(r["flux"], 4, 11)
            + _num(r["flux_rel_max"], 3, 9)
            + _num(r["prominence"], 2, 8)
        )
    return "\\n".join(lines)
""",
    "context/sustained_merge.py": """\"\"\"Merge nearby sustained events into longer rumble spans.\"\"\"

from __future__ import annotations

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def merge_sustained_events(
    events: list[DetectedEvent],
    taxonomy: Taxonomy | None = None,
    *,
    gap_sec: float | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Collapse fragmented sustained detections (vehicle chips) into fewer spans.

    Impulsive events are left unchanged. Sustained events of the same category
    whose windows are within ``gap_sec`` are merged; peak/confidence follow the
    strongest member.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    gap = gap_sec if gap_sec is not None else taxonomy.sustained_merge_gap_sec

    impulsive: list[DetectedEvent] = []
    by_cat: dict[str, list[DetectedEvent]] = {}
    for ev in events:
        cat = taxonomy.categories.get(ev.category)
        if cat is not None and cat.impulsive:
            impulsive.append(ev)
            continue
        by_cat.setdefault(ev.category, []).append(ev)

    merged: list[DetectedEvent] = list(impulsive)
    for cat_name, group in by_cat.items():
        ordered = sorted(group, key=lambda e: e.start_sec)
        if not ordered:
            continue
        cur = ordered[0]
        for ev in ordered[1:]:
            if ev.start_sec <= cur.end_sec + gap:
                best = ev if ev.confidence >= cur.confidence else cur
                cur = DetectedEvent(
                    category=cur.category,
                    label=best.label,
                    start_sec=min(cur.start_sec, ev.start_sec),
                    peak_sec=best.peak_sec,
                    end_sec=max(cur.end_sec, ev.end_sec),
                    confidence=max(cur.confidence, ev.confidence),
                    context_token=cur.context_token or ev.context_token,
                    audio_score=_max_opt(cur.audio_score, ev.audio_score),
                    video_score=_max_opt(cur.video_score, ev.video_score),
                    sources=sorted(set(cur.sources) | set(ev.sources)),
                )
            else:
                merged.append(cur)
                cur = ev
        merged.append(cur)

    merged.sort(key=lambda e: e.start_sec)
    return merged


def _max_opt(a: float | None, b: float | None) -> float | None:
    if a is None:
        return b
    if b is None:
        return a
    return max(a, b)


def sustained_coverage_sec(events: list[DetectedEvent], taxonomy: Taxonomy) -> float:
    \"\"\"Union duration of non-impulsive event windows.\"\"\"
    intervals = [
        (e.start_sec, e.end_sec)
        for e in events
        if (cat := taxonomy.categories.get(e.category)) is not None and not cat.impulsive
    ]
    if not intervals:
        return 0.0
    intervals.sort()
    total = 0.0
    cur_s, cur_e = intervals[0]
    for s, e in intervals[1:]:
        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            total += max(0.0, cur_e - cur_s)
            cur_s, cur_e = s, e
    total += max(0.0, cur_e - cur_s)
    return total
""",
    "context/sustained_salience.py": """\"\"\"Vehicle rumble from absolute loudness (local RMS), not RMS rise.

AST spans are only a category prior. Haptic windows are the loud RMS islands
so quiet gaps stay quiet (a 10 s AST slab must not vibrate through a dip).

The loudness floor is measured per scene, not per clip. One clip can hold a
distant tank drive and a close car rumble twice its level; a single clip-wide
floor is then above the drive and below the car's idle bed, which loses the
drive completely and merges every car burst into one long buzz.
\"\"\"

from __future__ import annotations

import numpy as np
from scipy.ndimage import median_filter

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def local_rms(
    times,
    env,
    center_sec: float,
    *,
    half_win_sec: float = 0.15,
) -> float:
    \"\"\"Peak envelope in a short window around ``center_sec``.\"\"\"
    if env.size == 0 or times.size == 0:
        return 0.0
    mask = (times >= center_sec - half_win_sec) & (times <= center_sec + half_win_sec)
    if not np.any(mask):
        idx = int(np.argmin(np.abs(times - center_sec)))
        return float(env[idx])
    return float(np.max(env[mask]))


def _median_smooth(env: np.ndarray, k: int = 15) -> np.ndarray:
    \"\"\"Median filter (~75 ms at 5 ms hop) so 5 ms spikes cannot bridge islands.\"\"\"
    if env.size == 0 or k <= 1:
        return env
    return median_filter(np.asarray(env, dtype=np.float64), size=int(k) | 1, mode="nearest")


def _frame_dt(times, default: float = 0.005) -> float:
    if times is None or len(times) < 2:
        return default
    dt = float(np.median(np.diff(times)))
    return dt if np.isfinite(dt) and dt > 0 else default


def _smoothing_frames(times, default: int = 15) -> int:
    \"\"\"Median window in frames, ~75 ms — longer than one cycle of a 40 Hz rumble.\"\"\"
    if times is None or len(times) < 2:
        return default
    dt = float(np.median(np.diff(times)))
    if not np.isfinite(dt) or dt <= 0:
        return default
    return max(5, int(round(0.075 / dt)) | 1)


def _active_frames(values: np.ndarray, taxonomy: Taxonomy) -> np.ndarray:
    \"\"\"Drop silence before taking percentiles.

    In a region that is half silence, p50 lands inside the rumble and the gate
    ends up above the very thing it should keep.
    \"\"\"
    if values.size == 0:
        return values
    silence_floor = (
        taxonomy.sustained_salience_silence_frac * float(np.percentile(values, 99))
    )
    active = values[values >= silence_floor]
    return active if active.size >= 8 else values


def _runs(flags: np.ndarray, dt: float) -> list[float]:
    \"\"\"Lengths in seconds of each run of True in ``flags``.\"\"\"
    out: list[float] = []
    run = 0
    for on in flags:
        if on:
            run += 1
        elif run:
            out.append(run * dt)
            run = 0
    if run:
        out.append(run * dt)
    return out


def _is_rhythmic(series: np.ndarray, threshold: float, dt: float, taxonomy: Taxonomy) -> bool:
    \"\"\"True when the scene alternates between loud bursts and quiet gaps.

    Only the stretch between the first and last loud frame counts: the silence
    before a rumble starts is not a gap in it.

    Both sides have to be substantial. Gaps must be long enough that the islands
    would not bridge them anyway, and bursts at least as long as the shortest
    rumble -- otherwise track clanks over a steady drive read as a rhythm and the
    drive gets chopped into the spaces between its own clanks.
    \"\"\"
    if series.size == 0 or dt <= 0:
        return False
    loud = np.where(series >= threshold)[0]
    if loud.size < 2:
        return False
    inside = series[loud[0] : loud[-1] + 1]
    gaps = [g for g in _runs(inside < threshold, dt) if g >= taxonomy.sustained_salience_gap_sec]
    bursts = _runs(inside >= threshold, dt)
    if len(gaps) < taxonomy.sustained_rhythm_min_gaps or not bursts:
        return False
    return (
        float(np.median(gaps)) <= taxonomy.sustained_rhythm_gap_max_sec
        and float(np.median(bursts)) >= taxonomy.sustained_salience_min_sec
    )


def _region_threshold(
    series: np.ndarray,
    taxonomy: Taxonomy,
    *,
    dt: float = 0.005,
) -> dict[str, float | str]:
    \"\"\"Loudness floor for one scene, from the shape and the rhythm of its frames.

    A scene with two loudness modes is split between them, so gaps between bursts
    stay quiet. A scene with one mode is kept whole -- unless its quiet stretches
    repeat, which is a rumble easing off and coming back rather than a steady one.

    Spread is p90/p50, not p99/p50: the top percentile is a handful of frames, so
    track clanks over a steady drive read as a second loudness mode and the drive
    gets chopped into the gaps between its own clanks.
    \"\"\"
    active = _active_frames(series, taxonomy)
    if active.size == 0:
        return {"threshold": 0.0, "spread": 1.0, "p_mid": 0.0, "p_loud": 0.0, "mode": "keep"}

    p_mid = float(np.percentile(active, taxonomy.sustained_salience_mid_pct))
    p_loud = float(np.percentile(active, 90))
    p_quiet = float(np.percentile(active, 10))
    spread = p_loud / max(p_mid, 1e-12)

    # Measured from the quiet level, not the median: once bursts fill more than
    # half the scene the median sits inside them and a median-based split lands on
    # top of the bursts it is supposed to keep.
    split = p_quiet + taxonomy.sustained_salience_mix * (p_loud - p_quiet)
    # Keeping all of it: half the median is not low enough -- a cut to a wider
    # camera angle halves the level while the tank keeps rolling -- so sit under
    # the quietest sustained part, with a floor so the gate stays meaningful when
    # that part is barely above silence.
    keep = max(0.6 * p_quiet, 0.30 * p_mid)

    bimodal = spread >= taxonomy.sustained_salience_min_sep
    # A rhythm needs two levels to alternate between; without this check the ripple
    # of a steady idle crosses its own split threshold and reads as a rhythm.
    two_level = p_loud >= taxonomy.sustained_rhythm_level_ratio * max(p_quiet, 1e-12)
    rhythmic = two_level and _is_rhythmic(series, split, dt, taxonomy)

    mode = "split" if (bimodal or rhythmic) else "keep"
    return {
        "threshold": float(split if mode == "split" else keep),
        "spread": float(spread),
        "p_mid": p_mid,
        "p_loud": p_loud,
        "mode": mode,
    }


def scene_spans(times, env, taxonomy: Taxonomy) -> list[tuple[float, float]]:
    \"\"\"Split a clip where its sustained level steps to a new value for seconds.

    Clips cut together from several sources hold one recording level per source.
    Each is its own scene and gets its own loudness floor. A brief loud passage is
    not a scene -- a rumble burst over an engine bed must keep being measured
    against that bed, not against itself.
    \"\"\"
    times = np.asarray(times, dtype=np.float64)
    if times.size < 2:
        return [(float(times[0]), float(times[-1]))] if times.size else []

    env_s = _median_smooth(np.asarray(env, dtype=np.float64), _smoothing_frames(times))
    start, end = float(times[0]), float(times[-1])
    block = max(taxonomy.sustained_scene_block_sec, _frame_dt(times) * 4)
    min_sec = taxonomy.sustained_scene_min_sec
    if end - start < 2 * min_sec:
        return [(start, end)]

    edges = np.arange(start, end, block)
    # How loud each block gets, not its median: a scene can be dynamic inside, and
    # a rumble that eases off between bursts must not read as a new scene.
    levels = np.array(
        [
            float(np.percentile(env_s[m], 90)) if np.any(m := ((times >= a) & (times < a + block))) else 0.0
            for a in edges
        ]
    )
    context = max(1, int(round(min_sec / block)))
    stable = taxonomy.sustained_scene_stable_ratio

    def _steady(chunk: np.ndarray) -> bool:
        if chunk.size == 0:
            return False
        return float(np.max(chunk)) <= stable * max(float(np.min(chunk)), 1e-9)

    candidates: list[tuple[float, float]] = []
    for i in range(1, len(levels)):
        left, right = levels[max(0, i - context) : i], levels[i : i + context]
        if left.size < context or right.size < context:
            continue
        # Both sides must hold their own level, otherwise the "step" is a passage
        # inside one scene -- a rumble burst over a bed, not a change of source.
        if not (_steady(left) and _steady(right)):
            continue
        hi, lo = float(np.median(left)), float(np.median(right))
        if hi < lo:
            hi, lo = lo, hi
        ratio = hi / max(lo, 1e-9)
        if ratio >= taxonomy.sustained_scene_level_ratio:
            candidates.append((ratio, float(edges[i])))

    cuts: list[float] = []
    for _ratio, t in sorted(candidates, reverse=True):
        pieces = sorted([start, end, t, *cuts])
        if all(b - a >= min_sec for a, b in zip(pieces, pieces[1:], strict=False)):
            cuts.append(t)
    cuts.sort()

    bounds = [start, *cuts, end]
    return list(zip(bounds, bounds[1:], strict=False))


def _gate_source(
    env_s: np.ndarray,
    times,
    exclude_peaks: list[float] | None,
    exclude_radius_sec: float,
) -> np.ndarray:
    \"\"\"Frames the loudness floor is measured on: blasts masked out, NaNs dropped.\"\"\"
    keep = np.isfinite(env_s)
    if times is not None and exclude_peaks:
        mask = np.ones(len(env_s), dtype=bool)
        for peak in exclude_peaks:
            mask &= (times < peak - exclude_radius_sec) | (times > peak + exclude_radius_sec)
        if np.any(mask & keep):
            keep &= mask
    return env_s[keep]


def salience_stats(
    env,
    taxonomy: Taxonomy,
    *,
    times=None,
    exclude_peaks: list[float] | None = None,
    exclude_radius_sec: float = 0.7,
) -> dict[str, float | bool]:
    \"\"\"Clip-wide loudness floor. Unimodal clips (steady idle) are not split.\"\"\"
    # Smooth first, with the same window the islands use: on a raw 5 ms envelope a
    # low-frequency rumble swings within each cycle, which fakes a wide spread.
    env_s = _median_smooth(np.asarray(env, dtype=np.float64), _smoothing_frames(times))
    v = env_s[np.isfinite(env_s)]
    if v.size == 0:
        return {
            "threshold": 0.0,
            "sep": 1.0,
            "unimodal": True,
            "p_mid": 0.0,
            "p_loud": 0.0,
        }
    p_mid_raw = float(np.percentile(v, taxonomy.sustained_salience_mid_pct))
    p_loud_raw = float(np.percentile(v, taxonomy.sustained_salience_loud_pct))

    gate_src = _gate_source(env_s, times, exclude_peaks, exclude_radius_sec)
    out = _region_threshold(gate_src, taxonomy, dt=_frame_dt(times))

    sep = p_loud_raw / max(p_mid_raw, 1e-8)
    return {
        "threshold": out["threshold"],
        "sep": float(sep),
        "sep_active": out["spread"],
        # Callers use this to leave the classifier's spans alone. A rumble whose
        # quiet stretches repeat has gaps to keep quiet, so it is not one mode even
        # when its percentiles say so.
        "unimodal": bool(sep < taxonomy.sustained_salience_min_sep)
        and out["mode"] == "keep",
        "p_mid": out["p_mid"],
        "p_loud": out["p_loud"],
    }


def salience_threshold_curve(
    times,
    env,
    taxonomy: Taxonomy,
    *,
    exclude_peaks: list[float] | None = None,
    exclude_radius_sec: float = 0.7,
) -> np.ndarray:
    \"\"\"Per-frame loudness floor: one level per scene.

    A clip with a single scene gets a single floor, the clip-wide one. Clips cut
    together from several sources get one floor per source, so a distant tank
    drive is not measured against a car rumble recorded twice as close.
    \"\"\"
    times = np.asarray(times, dtype=np.float64)
    env_s = _median_smooth(np.asarray(env, dtype=np.float64), _smoothing_frames(times))
    if env_s.size == 0:
        return np.zeros(0, dtype=np.float64)

    global_thr = float(
        salience_stats(
            env,
            taxonomy,
            times=times,
            exclude_peaks=exclude_peaks,
            exclude_radius_sec=exclude_radius_sec,
        )["threshold"]
    )
    scenes = scene_spans(times, env, taxonomy)
    if len(scenes) <= 1:
        return np.full(env_s.size, global_thr, dtype=np.float64)

    keep = np.isfinite(env_s)
    if exclude_peaks:
        for peak in exclude_peaks:
            keep &= (times < peak - exclude_radius_sec) | (times > peak + exclude_radius_sec)
        if not np.any(keep):
            keep = np.isfinite(env_s)

    # A scene of near-silence has percentiles of near-silence, and its own floor
    # would gate room tone on as rumble. Nothing quieter than this is a vehicle.
    clip_floor = taxonomy.sustained_salience_scene_floor_frac * float(
        np.percentile(env_s[np.isfinite(env_s)], 99)
    )

    curve = np.full(env_s.size, global_thr, dtype=np.float64)
    for lo, hi in scenes:
        span = (times >= lo) & (times <= hi)
        frames = env_s[span & keep]
        if frames.size < 8:
            continue
        thr = float(_region_threshold(frames, taxonomy, dt=_frame_dt(times))["threshold"])
        curve[span] = max(thr, clip_floor)
    return curve


def rms_islands(
    times,
    env,
    threshold: float | np.ndarray,
    *,
    min_sec: float,
    gap_sec: float,
    min_duty: float = 0.55,
    edge_frac: float = 0.75,
    max_extend_sec: float = 0.4,
) -> list[tuple[float, float, float]]:
    \"\"\"Contiguous high-RMS runs as (start, peak, end). Drops spiky low-duty blobs.

    ``threshold`` is a level or a per-frame floor (see
    :func:`salience_threshold_curve`).

    Edges are then walked outward down to ``edge_frac`` of the threshold: a rumble
    ramps up before it crosses the gate, and starting the buzz a third of a second
    into the burst feels late. The walk is capped so a gap cannot be swallowed.
    \"\"\"
    if env.size == 0 or times.size == 0:
        return []
    dt = float(np.median(np.diff(times))) if times.size > 1 else 0.005
    dt = max(dt, 1e-4)
    smooth_k = max(5, int(round(0.075 / dt)) | 1)
    smooth = _median_smooth(env, k=smooth_k)
    thr = np.asarray(threshold, dtype=np.float64)
    if thr.ndim == 0:
        thr = np.full(smooth.size, float(thr))
    mask = smooth >= thr
    gap_frames = max(1, int(round(gap_sec / dt)))
    min_frames = max(1, int(round(min_sec / dt)))

    closed = mask.copy()
    i = 0
    n = len(closed)
    while i < n:
        if closed[i]:
            i += 1
            continue
        j = i
        while j < n and not mask[j]:
            j += 1
        if i > 0 and j < n and (j - i) <= gap_frames:
            closed[i:j] = True
        i = j

    edge_thr = edge_frac * thr
    max_extend_frames = max(0, int(round(max_extend_sec / dt)))

    spans: list[tuple[int, int, int]] = []
    i = 0
    while i < n:
        if not closed[i]:
            i += 1
            continue
        j = i
        while j < n and closed[j]:
            j += 1
        if (j - i) >= min_frames:
            sl = slice(i, j)
            duty = float(np.mean(mask[sl]))
            if duty >= min_duty:
                peak_i = i + int(np.argmax(env[sl]))
                lo = i
                limit = max(0, i - max_extend_frames)
                while lo > limit and smooth[lo - 1] >= edge_thr[lo - 1]:
                    lo -= 1
                hi = j - 1
                limit = min(n - 1, (j - 1) + max_extend_frames)
                while hi < limit and smooth[hi + 1] >= edge_thr[hi + 1]:
                    hi += 1
                spans.append((lo, peak_i, hi))
        i = j

    # Two bursts a short gap apart can each walk outward into the other; emit one
    merged: list[tuple[int, int, int]] = []
    for lo, peak_i, hi in spans:
        if merged and lo <= merged[-1][2]:
            p_lo, _, p_hi = merged[-1]
            hi = max(p_hi, hi)
            merged[-1] = (p_lo, p_lo + int(np.argmax(env[p_lo : hi + 1])), hi)
            continue
        merged.append((lo, peak_i, hi))

    return [
        (float(times[lo]), float(times[peak_i]), float(times[hi]))
        for lo, peak_i, hi in merged
    ]


def curve_value(times, values: np.ndarray, t_sec: float) -> float:
    \"\"\"Value of a per-frame curve at ``t_sec``.\"\"\"
    if values.size == 0:
        return 0.0
    return float(values[int(np.argmin(np.abs(np.asarray(times) - t_sec)))])


def crop_to_loud(
    times,
    env,
    center_sec: float,
    threshold: float,
    *,
    max_sec: float = 1.2,
    min_sec: float = 0.2,
) -> tuple[float, float, float]:
    \"\"\"Tight start/peak/end around a loud peak; never keep a multi-second AST slab.\"\"\"
    if env.size == 0 or times.size == 0:
        return center_sec, center_sec, center_sec + min_sec
    idx = int(np.argmin(np.abs(times - center_sec)))
    n = len(env)
    lo = idx
    while lo > 0 and env[lo - 1] >= threshold and (times[idx] - times[lo - 1]) <= max_sec / 2:
        lo -= 1
    hi = idx
    while hi + 1 < n and env[hi + 1] >= threshold and (times[hi + 1] - times[idx]) <= max_sec / 2:
        hi += 1
    start = float(times[lo])
    end = float(times[hi])
    if end - start < min_sec:
        pad = (min_sec - (end - start)) / 2
        start -= pad
        end += pad
    peak_i = lo + int(np.argmax(env[lo : hi + 1]))
    return start, float(times[peak_i]), end


def _overlaps(start: float, end: float, island: tuple[float, float, float], pad: float) -> bool:
    return start - pad <= island[2] and end + pad >= island[0]


def apply_vehicle_salience(
    events: list[DetectedEvent],
    times,
    env,
    taxonomy: Taxonomy | None = None,
    *,
    duration_sec: float | None = None,
    report: dict | None = None,
) -> list[DetectedEvent]:
    \"\"\"
    Replace vehicle AST spans with loud RMS islands.

    One long classifier event may cover several rumbles; each island becomes
    its own haptic window. Quiet gaps are not filled. Short loud onsets with
    no island are cropped to ~1 s, never the original AST start/end.

    Pass ``report`` to record what the gate decided (floor per scene, which
    proposals it dropped and how far below the floor they were). Without it a
    missing rumble is indistinguishable from one the classifier never proposed.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    passthrough: list[DetectedEvent] = []
    vehicles: list[DetectedEvent] = []
    for ev in events:
        if ev.category == "vehicle":
            vehicles.append(ev)
        else:
            passthrough.append(ev)

    if report is not None:
        report.update({"proposals": len(vehicles), "kept": 0, "dropped": []})

    if not vehicles:
        return list(events)

    imp_peaks = [
        e.peak_sec
        for e in passthrough
        if (c := taxonomy.categories.get(e.category)) is not None and c.impulsive
    ]
    stats = salience_stats(
        env, taxonomy, times=times, exclude_peaks=imp_peaks, exclude_radius_sec=0.7
    )
    thr_curve = salience_threshold_curve(
        times, env, taxonomy, exclude_peaks=imp_peaks, exclude_radius_sec=0.7
    )
    peak_win = taxonomy.sustained_salience_peak_win_sec

    if report is not None:
        scenes = scene_spans(times, env, taxonomy)
        report.update(
            {
                "clip_threshold": round(float(stats["threshold"]), 4),
                "clip_unimodal": bool(stats["unimodal"]),
                "scenes": [
                    {
                        "start_sec": round(lo, 2),
                        "end_sec": round(hi, 2),
                        "threshold": round(curve_value(times, thr_curve, (lo + hi) / 2), 4),
                    }
                    for lo, hi in scenes
                ],
            }
        )

    if stats["unimodal"]:
        if report is not None:
            report["kept"] = len(vehicles)
        return list(events)

    islands = rms_islands(
        times,
        env,
        thr_curve,
        min_sec=taxonomy.sustained_salience_min_sec,
        gap_sec=taxonomy.sustained_salience_gap_sec,
        min_duty=taxonomy.sustained_salience_min_duty,
        edge_frac=taxonomy.sustained_island_edge_frac,
        max_extend_sec=taxonomy.sustained_island_max_extend_sec,
    )

    kept: list[DetectedEvent] = list(passthrough)
    template = max(vehicles, key=lambda e: e.confidence)

    def _emit(ev: DetectedEvent, start: float, peak: float, end: float, sources: list[str]):
        if duration_sec is not None:
            end = min(end, duration_sec)
            start = min(max(0.0, start), end)
        kept.append(
            DetectedEvent(
                category="vehicle",
                label=ev.label or "Vehicle",
                start_sec=max(0.0, start),
                peak_sec=min(max(peak, start), max(end, start + 0.2)),
                end_sec=max(end, start + 0.2),
                confidence=ev.confidence,
                context_token=ev.context_token,
                audio_score=ev.audio_score,
                video_score=ev.video_score,
                sources=sources,
            )
        )

    for i, isl in enumerate(islands):
        # Skip only if this island's peak IS the blast, not the next volley ~0.5 s later
        if any(abs(isl[1] - p) <= 0.45 for p in imp_peaks):
            continue
        # A short island right after a blast is its decay, not engine rumble
        if isl[2] - isl[0] < 1.5 and any(
            isl[0] - 0.7 <= p <= isl[2] for p in imp_peaks
        ):
            continue
        best: DetectedEvent | None = None
        for ev in vehicles:
            if _overlaps(ev.start_sec, ev.end_sec, isl, 0.25) or (
                isl[0] - 0.15 <= ev.peak_sec <= isl[2] + 0.15
            ):
                if best is None or ev.confidence > best.confidence:
                    best = ev
        src_ev = best or template
        src = list(src_ev.sources) if src_ev.sources else ["audio"]
        if best is None and "salience" not in src:
            src = src + ["salience"]
        start, peak, end = isl
        _emit(src_ev, start, peak, end, src)

    max_orphan = taxonomy.sustained_salience_max_orphan_sec
    for ev in vehicles:
        if any(
            _overlaps(ev.start_sec, ev.end_sec, isl, 0.15)
            or (isl[0] - 0.15 <= ev.peak_sec <= isl[2] + 0.15)
            for isl in islands
        ):
            continue
        thr = curve_value(times, thr_curve, ev.peak_sec)
        loudness = local_rms(times, env, ev.peak_sec, half_win_sec=peak_win)
        if loudness < thr:
            if report is not None:
                report["dropped"].append(
                    {
                        "start_sec": round(ev.start_sec, 3),
                        "end_sec": round(ev.end_sec, 3),
                        "local_rms": round(loudness, 4),
                        "scene_threshold": round(thr, 4),
                        "reason": "below scene loudness floor",
                    }
                )
            continue
        start, peak, end = crop_to_loud(
            times, env, ev.peak_sec, thr, max_sec=max_orphan, min_sec=0.0
        )
        if end - start < taxonomy.sustained_salience_min_sec:
            if report is not None:
                report["dropped"].append(
                    {
                        "start_sec": round(ev.start_sec, 3),
                        "end_sec": round(ev.end_sec, 3),
                        "local_rms": round(loudness, 4),
                        "scene_threshold": round(thr, 4),
                        "reason": "loud span shorter than the minimum rumble",
                    }
                )
            continue
        src = list(ev.sources) if ev.sources else ["audio"]
        _emit(ev, start, peak, end, src)

    kept.sort(key=lambda e: e.start_sec)
    if report is not None:
        report["kept"] = sum(1 for e in kept if e.category == "vehicle")
    return kept
""",
    "context/taxonomy.py": """\"\"\"Load and query the predefined event taxonomy.\"\"\"

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import yaml

DEFAULT_TAXONOMY_PATH = Path(__file__).with_name("taxonomy.yaml")


@dataclass
class CategoryConfig:
    name: str
    audioset_labels: list[str] = field(default_factory=list)
    kinetics_labels: list[str] = field(default_factory=list)
    context_token_labels: list[str] = field(default_factory=list)
    audio_weight: float = 0.5
    video_weight: float = 0.5
    require_context_or_both: bool = False
    impulsive: bool = False
    include_in_haptic_gate: bool = False
    # Optional per-category SED hysteresis overrides
    sed_high: float | None = None
    sed_low: float | None = None


@dataclass
class Taxonomy:
    timeline_hz: int = 100
    context_detector_threshold: float = 0.85
    encoder_threshold: float = 0.35
    impulsive_encoder_threshold: float = 0.30
    sustained_encoder_threshold: float = 0.25
    impulsive_min_peak_distance_sec: float = 0.45
    impulsive_event_half_width_sec: float = 0.45
    impulsive_onset_search_radius_sec: float = 0.75
    impulsive_onset_back_sec: float = 0.35
    impulsive_onset_forward_sec: float = 1.2
    impulsive_short_clip_sec: float = 8.0
    impulsive_decay_tail_sec: float = 0.85
    impulsive_decay_threshold: float = 0.12
    impulsive_pre_roll_sec: float = 0.08
    sustained_max_gate_sec: float = 2.5
    sustained_merge_gap_sec: float = 0.35
    sustained_burst_rise_ratio: float = 1.10
    sustained_burst_pre_sec: float = 0.7
    sustained_burst_post_sec: float = 1.5
    sustained_burst_max_sec: float = 2.5
    sustained_salience_mid_pct: float = 50.0
    sustained_salience_loud_pct: float = 99.0
    sustained_salience_mix: float = 0.40
    sustained_salience_min_sep: float = 1.45
    sustained_salience_silence_frac: float = 0.05
    sustained_rhythm_min_gaps: int = 2
    sustained_rhythm_gap_max_sec: float = 2.0
    sustained_rhythm_level_ratio: float = 2.0
    sustained_scene_level_ratio: float = 2.5
    sustained_scene_min_sec: float = 3.5
    sustained_scene_block_sec: float = 2.0
    sustained_scene_stable_ratio: float = 2.0
    sustained_salience_scene_floor_frac: float = 0.16
    sustained_island_edge_frac: float = 0.75
    sustained_island_max_extend_sec: float = 0.4
    sustained_salience_min_sec: float = 0.50
    sustained_salience_gap_sec: float = 0.18
    sustained_salience_peak_win_sec: float = 0.15
    sustained_salience_min_duty: float = 0.55
    sustained_salience_max_orphan_sec: float = 1.2
    proposal_rms_hop_ms: float = 5.0
    proposal_threshold_ratio: float = 0.25
    proposal_search_pad_sec: float = 0.75
    proposal_window_sec: float = 1.0
    proposal_sustained_hop_sec: float = 1.0
    onset_flux_hop_ms: float = 5.0
    onset_flux_min_ratio: float = 0.45
    onset_flux_early_rel: float = 0.55
    onset_flux_proposal_ratio: float = 0.16
    # Intermittent rumble: mask continuous bed by sustained spans
    sustained_mask_min_events: int = 3
    sustained_mask_min_coverage: float = 0.15
    # Frame-level SED decoding
    sed_enabled: bool = True
    sed_backend: str = "auto"
    sed_window_sec: float = 1.0
    sed_hop_sec: float = 0.1
    sed_median_impulsive_sec: float = 0.15
    sed_median_sustained_sec: float = 0.45
    sed_onset_high: float = 0.30
    sed_onset_low: float = 0.15
    sed_min_event_sec_impulsive: float = 0.10
    sed_min_event_sec_sustained: float = 0.40
    sed_merge_gap_sec: float = 0.20
    sed_peak_rel: float = 0.60
    # Orange/fireball onsets from pixels (not ViViT). Snaps impulsive peaks
    # onto the picture so a montage cut or late boom does not fire off-screen.
    visual_flash_enabled: bool = True
    visual_flash_min_d_warm: float = 0.025
    visual_flash_min_warm: float = 0.025
    visual_flash_min_d_hot: float = -0.005
    visual_flash_match_sec: float = 0.50
    visual_flash_min_sep_sec: float = 0.45
    # Video fusion is off; see taxonomy.yaml for why
    use_video: bool = False
    categories: dict[str, CategoryConfig] = field(default_factory=dict)

    def all_audioset_labels(self) -> set[str]:
        out: set[str] = set()
        for cat in self.categories.values():
            out.update(cat.audioset_labels)
        return out

    def all_kinetics_labels(self) -> set[str]:
        out: set[str] = set()
        for cat in self.categories.values():
            out.update(cat.kinetics_labels)
        return out


def _normalize(label: str) -> str:
    return " ".join(label.lower().strip().split())


def load_taxonomy(path: str | Path | None = None) -> Taxonomy:
    path = Path(path) if path else DEFAULT_TAXONOMY_PATH
    raw = yaml.safe_load(path.read_text(encoding="utf-8"))
    categories: dict[str, CategoryConfig] = {}
    for name, cfg in raw.get("categories", {}).items():
        categories[name] = CategoryConfig(
            name=name,
            audioset_labels=list(cfg.get("audioset_labels", [])),
            kinetics_labels=list(cfg.get("kinetics_labels", [])),
            context_token_labels=list(cfg.get("context_token_labels", [])),
            audio_weight=float(cfg.get("audio_weight", 0.5)),
            video_weight=float(cfg.get("video_weight", 0.5)),
            require_context_or_both=bool(cfg.get("require_context_or_both", False)),
            impulsive=bool(cfg.get("impulsive", False)),
            include_in_haptic_gate=bool(
                cfg.get("include_in_haptic_gate", bool(cfg.get("impulsive", False)))
            ),
            sed_high=(
                float(cfg["sed_high"]) if cfg.get("sed_high") is not None else None
            ),
            sed_low=(float(cfg["sed_low"]) if cfg.get("sed_low") is not None else None),
        )
    return Taxonomy(
        timeline_hz=int(raw.get("timeline_hz", 100)),
        context_detector_threshold=float(raw.get("context_detector_threshold", 0.85)),
        encoder_threshold=float(raw.get("encoder_threshold", 0.35)),
        impulsive_encoder_threshold=float(raw.get("impulsive_encoder_threshold", 0.30)),
        sustained_encoder_threshold=float(raw.get("sustained_encoder_threshold", 0.28)),
        impulsive_min_peak_distance_sec=float(
            raw.get("impulsive_min_peak_distance_sec", 0.45)
        ),
        impulsive_event_half_width_sec=float(
            raw.get("impulsive_event_half_width_sec", 0.45)
        ),
        impulsive_onset_search_radius_sec=float(
            raw.get("impulsive_onset_search_radius_sec", 0.75)
        ),
        impulsive_onset_back_sec=float(raw.get("impulsive_onset_back_sec", 0.35)),
        impulsive_onset_forward_sec=float(raw.get("impulsive_onset_forward_sec", 1.2)),
        impulsive_short_clip_sec=float(raw.get("impulsive_short_clip_sec", 8.0)),
        impulsive_decay_tail_sec=float(raw.get("impulsive_decay_tail_sec", 0.85)),
        impulsive_decay_threshold=float(raw.get("impulsive_decay_threshold", 0.12)),
        impulsive_pre_roll_sec=float(raw.get("impulsive_pre_roll_sec", 0.08)),
        sustained_max_gate_sec=float(raw.get("sustained_max_gate_sec", 2.5)),
        sustained_merge_gap_sec=float(raw.get("sustained_merge_gap_sec", 0.35)),
        sustained_burst_rise_ratio=float(raw.get("sustained_burst_rise_ratio", 1.10)),
        sustained_burst_pre_sec=float(raw.get("sustained_burst_pre_sec", 0.7)),
        sustained_burst_post_sec=float(raw.get("sustained_burst_post_sec", 1.5)),
        sustained_burst_max_sec=float(raw.get("sustained_burst_max_sec", 2.5)),
        sustained_salience_mid_pct=float(raw.get("sustained_salience_mid_pct", 50.0)),
        sustained_salience_loud_pct=float(raw.get("sustained_salience_loud_pct", 99.0)),
        sustained_salience_mix=float(raw.get("sustained_salience_mix", 0.40)),
        sustained_salience_min_sep=float(raw.get("sustained_salience_min_sep", 1.45)),
        sustained_salience_silence_frac=float(
            raw.get("sustained_salience_silence_frac", 0.05)
        ),
        sustained_rhythm_min_gaps=int(raw.get("sustained_rhythm_min_gaps", 2)),
        sustained_rhythm_gap_max_sec=float(
            raw.get("sustained_rhythm_gap_max_sec", 2.0)
        ),
        sustained_rhythm_level_ratio=float(
            raw.get("sustained_rhythm_level_ratio", 2.0)
        ),
        sustained_scene_level_ratio=float(raw.get("sustained_scene_level_ratio", 2.5)),
        sustained_scene_min_sec=float(raw.get("sustained_scene_min_sec", 3.5)),
        sustained_scene_block_sec=float(raw.get("sustained_scene_block_sec", 2.0)),
        sustained_scene_stable_ratio=float(
            raw.get("sustained_scene_stable_ratio", 2.0)
        ),
        sustained_salience_scene_floor_frac=float(
            raw.get("sustained_salience_scene_floor_frac", 0.16)
        ),
        sustained_island_edge_frac=float(raw.get("sustained_island_edge_frac", 0.75)),
        sustained_island_max_extend_sec=float(
            raw.get("sustained_island_max_extend_sec", 0.4)
        ),
        sustained_salience_min_sec=float(raw.get("sustained_salience_min_sec", 0.50)),
        sustained_salience_gap_sec=float(raw.get("sustained_salience_gap_sec", 0.18)),
        sustained_salience_peak_win_sec=float(raw.get("sustained_salience_peak_win_sec", 0.15)),
        sustained_salience_min_duty=float(raw.get("sustained_salience_min_duty", 0.55)),
        sustained_salience_max_orphan_sec=float(
            raw.get("sustained_salience_max_orphan_sec", 1.2)
        ),
        proposal_rms_hop_ms=float(raw.get("proposal_rms_hop_ms", 5.0)),
        proposal_threshold_ratio=float(raw.get("proposal_threshold_ratio", 0.25)),
        proposal_search_pad_sec=float(raw.get("proposal_search_pad_sec", 0.75)),
        proposal_window_sec=float(raw.get("proposal_window_sec", 1.0)),
        proposal_sustained_hop_sec=float(raw.get("proposal_sustained_hop_sec", 1.0)),
        onset_flux_hop_ms=float(raw.get("onset_flux_hop_ms", 5.0)),
        onset_flux_min_ratio=float(raw.get("onset_flux_min_ratio", 0.45)),
        onset_flux_early_rel=float(raw.get("onset_flux_early_rel", 0.55)),
        onset_flux_proposal_ratio=float(raw.get("onset_flux_proposal_ratio", 0.16)),
        sustained_mask_min_events=int(raw.get("sustained_mask_min_events", 3)),
        sustained_mask_min_coverage=float(raw.get("sustained_mask_min_coverage", 0.15)),
        sed_enabled=bool(raw.get("sed_enabled", True)),
        sed_backend=str(raw.get("sed_backend", "auto")),
        sed_window_sec=float(raw.get("sed_window_sec", 1.0)),
        sed_hop_sec=float(raw.get("sed_hop_sec", 0.1)),
        sed_median_impulsive_sec=float(raw.get("sed_median_impulsive_sec", 0.15)),
        sed_median_sustained_sec=float(raw.get("sed_median_sustained_sec", 0.45)),
        sed_onset_high=float(raw.get("sed_onset_high", 0.30)),
        sed_onset_low=float(raw.get("sed_onset_low", 0.15)),
        sed_min_event_sec_impulsive=float(raw.get("sed_min_event_sec_impulsive", 0.10)),
        sed_min_event_sec_sustained=float(raw.get("sed_min_event_sec_sustained", 0.40)),
        sed_merge_gap_sec=float(raw.get("sed_merge_gap_sec", 0.20)),
        sed_peak_rel=float(raw.get("sed_peak_rel", 0.60)),
        visual_flash_enabled=bool(raw.get("visual_flash_enabled", True)),
        visual_flash_min_d_warm=float(raw.get("visual_flash_min_d_warm", 0.025)),
        visual_flash_min_warm=float(raw.get("visual_flash_min_warm", 0.025)),
        visual_flash_min_d_hot=float(raw.get("visual_flash_min_d_hot", -0.005)),
        visual_flash_match_sec=float(raw.get("visual_flash_match_sec", 0.50)),
        visual_flash_min_sep_sec=float(raw.get("visual_flash_min_sep_sec", 0.45)),
        use_video=bool(raw.get("use_video", False)),
        categories=categories,
    )


def match_label_to_category(taxonomy: Taxonomy, label: str, source: str) -> str | None:
    \"\"\"Map a raw model label to a taxonomy category name, or None.\"\"\"
    norm = _normalize(label)
    for cat_name, cat in taxonomy.categories.items():
        if source in ("audio", "context") or source == "audioset":
            pool = cat.audioset_labels + cat.context_token_labels
        elif source in ("video", "kinetics"):
            pool = cat.kinetics_labels
        elif source == "context_token":
            pool = cat.context_token_labels
        else:
            pool = (
                cat.audioset_labels
                + cat.kinetics_labels
                + cat.context_token_labels
            )
        for candidate in pool:
            c_norm = _normalize(candidate)
            if norm == c_norm or c_norm in norm or norm in c_norm:
                return cat_name
    return None
""",
    "context/tokenization.py": """\"\"\"Phase 1 tokenization: 100Hz timeline and 16kHz audio.\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

MEL_SR = 16_000
TIMELINE_HZ = 100
FRAME_SIZE = 224


@dataclass
class TimelineTokens:
    \"\"\"Aligned multimodal tokens on a fixed-rate timeline.\"\"\"

    duration_sec: float
    timeline_hz: int
    video_frames: np.ndarray  # [T, 3, H, W] float32 in [0, 1]
    audio_16k: np.ndarray  # 1D float32 @ 16kHz
    source_fps: float


def _decode_video_frames(video_path: Path, n_bins: int, duration_sec: float) -> tuple[np.ndarray, float]:
    try:
        import cv2
    except ImportError as exc:
        raise RuntimeError(
            "opencv-python is required for video tokenization. "
            "Install with: pip install opencv-python"
        ) from exc

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    raw_frames: list[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (FRAME_SIZE, FRAME_SIZE), interpolation=cv2.INTER_AREA)
        raw_frames.append(resized.astype(np.float32) / 255.0)

    cap.release()
    if not raw_frames:
        raise RuntimeError(f"No frames decoded from video: {video_path}")

    if duration_sec <= 0:
        duration_sec = len(raw_frames) / fps

    aligned = np.zeros((n_bins, FRAME_SIZE, FRAME_SIZE, 3), dtype=np.float32)
    for i in range(n_bins):
        t = i / TIMELINE_HZ
        src_idx = int(round(t * fps))
        src_idx = min(max(src_idx, 0), len(raw_frames) - 1)
        aligned[i] = raw_frames[src_idx]

    video = np.transpose(aligned, (0, 3, 1, 2))
    return video, float(fps)


def _load_audio_16k(source_wav: Path) -> np.ndarray:
    audio, sr = sf.read(source_wav, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != MEL_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=MEL_SR)
    return audio


def tokenize_video_audio(
    video_path: str | Path,
    source_wav: str | Path,
    *,
    timeline_hz: int = TIMELINE_HZ,
) -> TimelineTokens:
    \"\"\"Build 100Hz-aligned video frames and 16kHz mono audio.\"\"\"
    video_path = Path(video_path)
    source_wav = Path(source_wav)
    audio_16k = _load_audio_16k(source_wav)
    duration_sec = len(audio_16k) / MEL_SR
    n_bins = max(1, int(round(duration_sec * timeline_hz)))

    video_frames, source_fps = _decode_video_frames(video_path, n_bins, duration_sec)

    return TimelineTokens(
        duration_sec=duration_sec,
        timeline_hz=timeline_hz,
        video_frames=video_frames,
        audio_16k=audio_16k,
        source_fps=source_fps,
    )
""",
    "context/visual_flash.py": """\"\"\"Snap impulsive events onto visible fireballs / muzzle flashes.

Audio-only SED follows the soundtrack. Montage clips (this tank cut) put the
boom, the cut, and the orange flash on different frames, so a peak that is
correct in the WAV still looks late or on the wrong shot. This pass is *not*
ViViT: it finds frames where orange/fire pixels suddenly appear.
\"\"\"

from __future__ import annotations

from pathlib import Path

import numpy as np

from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy

_IMPULSIVE = frozenset({"explosion", "gunshot"})


def flashes_from_frame_metrics(
    times: np.ndarray,
    warm: np.ndarray,
    hot: np.ndarray,
    *,
    min_d_warm: float,
    min_warm: float,
    min_d_hot: float,
    min_sep_sec: float,
) -> list[float]:
    \"\"\"Return onset times of orange flashes, strongest first then spaced.\"\"\"
    if times.size == 0:
        return []
    d_warm = np.diff(warm, prepend=float(warm[0]))
    d_hot = np.diff(hot, prepend=float(hot[0]))
    idxs = np.flatnonzero(
        (d_warm >= min_d_warm) & (warm >= min_warm) & (d_hot >= min_d_hot)
    )
    ranked = sorted(
        ((float(d_warm[i]), float(times[i])) for i in idxs),
        reverse=True,
    )
    kept: list[float] = []
    for _, t in ranked:
        if any(abs(t - k) < min_sep_sec for k in kept):
            continue
        kept.append(t)
    kept.sort()
    return kept


def detect_visual_flashes(
    video_path: str | Path,
    taxonomy: Taxonomy | None = None,
) -> list[float]:
    \"\"\"Scan ``video_path`` for orange-pixel onsets.\"\"\"
    taxonomy = taxonomy or load_taxonomy()
    try:
        import cv2
    except ImportError:
        return []

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0) or 30.0
    times: list[float] = []
    warm: list[float] = []
    hot: list[float] = []
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        small = cv2.resize(frame, (160, 90), interpolation=cv2.INTER_AREA)
        b, g, r = cv2.split(small)
        luma = 0.114 * b + 0.587 * g + 0.299 * r
        times.append(i / fps)
        warm.append(
            float(((r > 160) & (r > g + 15) & (r > b + 20)).mean())
        )
        hot.append(float((luma > 200).mean()))
        i += 1
    cap.release()
    if not times:
        return []
    return flashes_from_frame_metrics(
        np.asarray(times, dtype=np.float64),
        np.asarray(warm, dtype=np.float64),
        np.asarray(hot, dtype=np.float64),
        min_d_warm=taxonomy.visual_flash_min_d_warm,
        min_warm=taxonomy.visual_flash_min_warm,
        min_d_hot=taxonomy.visual_flash_min_d_hot,
        min_sep_sec=taxonomy.visual_flash_min_sep_sec,
    )


def align_impulsive_events_to_flashes(
    events: list[DetectedEvent],
    video_path: str | Path | None = None,
    taxonomy: Taxonomy | None = None,
    *,
    flashes: list[float] | None = None,
) -> list[DetectedEvent]:
    \"\"\"Snap / add explosion peaks onto visible flashes; never drop audio fires.

    Matched peaks move onto the fireball frame. Unmatched flashes become new
    accents. Audio/SED bangs with no nearby flash are kept as-is (off-screen
    or low-orange booms still need a haptic hit). Vehicle/weather spans are
    left alone.
    \"\"\"
    taxonomy = taxonomy or load_taxonomy()
    if not taxonomy.visual_flash_enabled:
        return list(events)

    if flashes is None:
        if video_path is None:
            return list(events)
        flashes = detect_visual_flashes(video_path, taxonomy)
    if not flashes:
        return list(events)

    radius = taxonomy.visual_flash_match_sec
    pre = taxonomy.impulsive_pre_roll_sec
    half = taxonomy.impulsive_event_half_width_sec

    impulsive = [e for e in events if e.category in _IMPULSIVE]
    other = [e for e in events if e.category not in _IMPULSIVE]
    used: set[int] = set()
    out: list[DetectedEvent] = list(other)

    for flash_t in flashes:
        best_i = None
        best_dist = radius
        for i, ev in enumerate(impulsive):
            if i in used:
                continue
            dist = abs(ev.peak_sec - flash_t)
            if dist <= best_dist:
                best_dist = dist
                best_i = i
        if best_i is None:
            out.append(
                DetectedEvent(
                    category="explosion",
                    label="Explosion",
                    start_sec=max(0.0, flash_t - pre),
                    peak_sec=flash_t,
                    end_sec=flash_t + half,
                    confidence=0.55,
                    sources=["visual_flash"],
                    video_score=1.0,
                )
            )
            continue
        used.add(best_i)
        ev = impulsive[best_i]
        sources = list(ev.sources)
        if "visual_flash" not in sources:
            sources.append("visual_flash")
        start = max(0.0, flash_t - pre)
        end = max(ev.end_sec, flash_t + half)
        out.append(
            DetectedEvent(
                category=ev.category,
                label=ev.label,
                start_sec=start,
                peak_sec=flash_t,
                end_sec=max(end, start + 0.2),
                confidence=ev.confidence,
                context_token=ev.context_token,
                audio_score=ev.audio_score,
                video_score=ev.video_score,
                sources=sources,
                attack_rel_max=ev.attack_rel_max,
                attack_prominence=ev.attack_prominence,
            )
        )

    # Keep bangs the picture missed (no orange jump / off-screen boom).
    for i, ev in enumerate(impulsive):
        if i not in used:
            out.append(ev)

    out.sort(key=lambda e: e.start_sec)
    return out
""",
    "eval/__init__.py": """\"\"\"Evaluation metrics for detected events (DCASE-style).\"\"\"
""",
    "eval/sed_metrics.py": """\"\"\"Event-based and segment-based F1, following the DCASE / sed_eval conventions.

Two standard views of the same detections:

Event-based
    A detection matches a reference event when their onsets fall within a collar
    (DCASE Task 4 uses 200 ms) and, optionally, when the offsets agree within
    ``offset_collar_sec`` or a percentage of the reference length. Matching is
    one-to-one, so duplicate detections on one reference count as insertions.

Segment-based
    The timeline is cut into fixed segments (1 s by default) and each segment is
    scored for presence/absence per category. Insensitive to onset jitter, so it
    answers "did we find the event at all" rather than "is the timing tight".

Reference events may be given as full spans, or as onset-only marks (a hand-made
list of times), which is what listening by ear produces.
\"\"\"

from __future__ import annotations

from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class RefEvent:
    category: str
    onset_sec: float
    offset_sec: float | None = None


@dataclass
class PRF:
    precision: float
    recall: float
    f1: float
    n_ref: int
    n_det: int
    tp: int
    fp: int
    fn: int


def _prf(tp: int, fp: int, fn: int, n_ref: int, n_det: int) -> PRF:
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    )
    return PRF(
        precision=round(precision, 4),
        recall=round(recall, 4),
        f1=round(f1, 4),
        n_ref=n_ref,
        n_det=n_det,
        tp=tp,
        fp=fp,
        fn=fn,
    )


def _as_ref_events(
    reference: list[RefEvent] | list[dict] | dict[str, list[float]],
) -> list[RefEvent]:
    \"\"\"Accept RefEvent list, events.json-style dicts, or {category: [onsets]}.\"\"\"
    if isinstance(reference, dict):
        out: list[RefEvent] = []
        for cat, times in reference.items():
            for t in times:
                out.append(RefEvent(category=cat, onset_sec=float(t)))
        return sorted(out, key=lambda r: (r.category, r.onset_sec))
    out = []
    for item in reference:
        if isinstance(item, RefEvent):
            out.append(item)
            continue
        onset = item.get("onset_sec", item.get("start_sec", item.get("peak_sec")))
        offset = item.get("offset_sec", item.get("end_sec"))
        out.append(
            RefEvent(
                category=str(item.get("category", "")),
                onset_sec=float(onset),
                offset_sec=None if offset is None else float(offset),
            )
        )
    return sorted(out, key=lambda r: (r.category, r.onset_sec))


def _detection_onsets(
    detections: list[dict],
    *,
    onset_field: str,
) -> list[tuple[str, float, float | None]]:
    rows: list[tuple[str, float, float | None]] = []
    for d in detections:
        cat = str(d.get("category", ""))
        onset = d.get(onset_field)
        if onset is None:
            onset = d.get("start_sec", d.get("peak_sec", 0.0))
        end = d.get("end_sec")
        rows.append((cat, float(onset), None if end is None else float(end)))
    rows.sort(key=lambda r: (r[0], r[1]))
    return rows


def event_based_prf(
    reference,
    detections: list[dict],
    *,
    onset_collar_sec: float = 0.2,
    offset_collar_sec: float | None = None,
    offset_collar_rate: float = 0.2,
    categories: list[str] | None = None,
    onset_field: str = "peak_sec",
) -> dict:
    \"\"\"One-to-one onset matching within a collar, per category and overall.

    ``offset_collar_sec`` None disables offset checking (onset-only scoring),
    which is the right mode for hand-marked onsets.
    \"\"\"
    refs = _as_ref_events(reference)
    dets = _detection_onsets(detections, onset_field=onset_field)
    cats = categories or sorted({r.category for r in refs} | {d[0] for d in dets})

    per_cat: dict[str, PRF] = {}
    matches: list[dict] = []
    tot_tp = tot_fp = tot_fn = 0
    tot_ref = tot_det = 0

    for cat in cats:
        cat_refs = [r for r in refs if r.category == cat]
        cat_dets = [d for d in dets if d[0] == cat]
        used_det: set[int] = set()
        tp = 0
        for ref in cat_refs:
            best: tuple[float, int] | None = None
            for j, (_, onset, end) in enumerate(cat_dets):
                if j in used_det:
                    continue
                err = abs(onset - ref.onset_sec)
                if err > onset_collar_sec:
                    continue
                if (
                    offset_collar_sec is not None
                    and ref.offset_sec is not None
                    and end is not None
                ):
                    tol = max(
                        offset_collar_sec,
                        offset_collar_rate * (ref.offset_sec - ref.onset_sec),
                    )
                    if abs(end - ref.offset_sec) > tol:
                        continue
                if best is None or err < best[0]:
                    best = (err, j)
            if best is None:
                matches.append(
                    {
                        "category": cat,
                        "ref_onset_sec": round(ref.onset_sec, 3),
                        "det_onset_sec": None,
                        "error_sec": None,
                        "status": "miss",
                    }
                )
                continue
            err, j = best
            used_det.add(j)
            tp += 1
            matches.append(
                {
                    "category": cat,
                    "ref_onset_sec": round(ref.onset_sec, 3),
                    "det_onset_sec": round(cat_dets[j][1], 3),
                    "error_sec": round(cat_dets[j][1] - ref.onset_sec, 3),
                    "status": "hit",
                }
            )
        for j, (_, onset, _end) in enumerate(cat_dets):
            if j in used_det:
                continue
            matches.append(
                {
                    "category": cat,
                    "ref_onset_sec": None,
                    "det_onset_sec": round(onset, 3),
                    "error_sec": None,
                    "status": "false_positive",
                }
            )
        fp = len(cat_dets) - tp
        fn = len(cat_refs) - tp
        per_cat[cat] = _prf(tp, fp, fn, len(cat_refs), len(cat_dets))
        tot_tp += tp
        tot_fp += fp
        tot_fn += fn
        tot_ref += len(cat_refs)
        tot_det += len(cat_dets)

    micro = _prf(tot_tp, tot_fp, tot_fn, tot_ref, tot_det)
    macro_f1 = (
        round(sum(v.f1 for v in per_cat.values()) / len(per_cat), 4) if per_cat else 0.0
    )
    errors = [abs(m["error_sec"]) for m in matches if m["error_sec"] is not None]
    return {
        "mode": "event_based",
        "onset_collar_sec": onset_collar_sec,
        "offset_collar_sec": offset_collar_sec,
        "micro": asdict(micro),
        "macro_f1": macro_f1,
        "per_category": {k: asdict(v) for k, v in per_cat.items()},
        "mean_abs_onset_error_sec": (
            round(sum(errors) / len(errors), 4) if errors else None
        ),
        "matches": sorted(
            matches, key=lambda m: (m["ref_onset_sec"] or m["det_onset_sec"] or 0.0)
        ),
    }


def segment_based_prf(
    reference,
    detections: list[dict],
    *,
    duration_sec: float,
    segment_sec: float = 1.0,
    default_ref_length_sec: float = 0.5,
    categories: list[str] | None = None,
) -> dict:
    \"\"\"Presence/absence per fixed segment, per category and overall.

    Onset-only references are given ``default_ref_length_sec`` so they occupy a
    segment at all.
    \"\"\"
    refs = _as_ref_events(reference)
    cats = categories or sorted(
        {r.category for r in refs} | {str(d.get("category", "")) for d in detections}
    )
    n_seg = max(1, int(round(duration_sec / segment_sec)))

    def _active(spans: list[tuple[float, float]], seg: int) -> bool:
        s0 = seg * segment_sec
        s1 = s0 + segment_sec
        return any(start < s1 and end > s0 for start, end in spans)

    per_cat: dict[str, PRF] = {}
    tot_tp = tot_fp = tot_fn = 0
    for cat in cats:
        ref_spans = [
            (
                r.onset_sec,
                r.offset_sec
                if r.offset_sec is not None
                else r.onset_sec + default_ref_length_sec,
            )
            for r in refs
            if r.category == cat
        ]
        det_spans = [
            (
                float(d.get("start_sec", d.get("peak_sec", 0.0))),
                float(d.get("end_sec", d.get("peak_sec", 0.0))),
            )
            for d in detections
            if str(d.get("category", "")) == cat
        ]
        tp = fp = fn = 0
        for seg in range(n_seg):
            r_on = _active(ref_spans, seg)
            d_on = _active(det_spans, seg)
            if r_on and d_on:
                tp += 1
            elif d_on:
                fp += 1
            elif r_on:
                fn += 1
        per_cat[cat] = _prf(tp, fp, fn, tp + fn, tp + fp)
        tot_tp += tp
        tot_fp += fp
        tot_fn += fn

    micro = _prf(tot_tp, tot_fp, tot_fn, tot_tp + tot_fn, tot_tp + tot_fp)
    macro_f1 = (
        round(sum(v.f1 for v in per_cat.values()) / len(per_cat), 4) if per_cat else 0.0
    )
    return {
        "mode": "segment_based",
        "segment_sec": segment_sec,
        "micro": asdict(micro),
        "macro_f1": macro_f1,
        "per_category": {k: asdict(v) for k, v in per_cat.items()},
    }


def coverage_report(
    reference,
    detections: list[dict],
    *,
    categories: list[str],
    duration_sec: float | None = None,
    tolerance_sec: float = 0.35,
) -> dict:
    \"\"\"Recall-only view: does a detected span contain each mark?

    Sustained rumble cannot be scored by onset collar or by segment presence when
    the reference is a handful of "I feel it here" marks: the marks are moments
    inside a burst, not onsets, and they say nothing about the seconds between
    them. Precision needs full reference spans, so only recall is reported, next
    to how much of the clip was marked active so over-triggering stays visible.

    ``tolerance_sec`` widens each span, because a mark typed as "6" is a rounded
    second and the burst it refers to may start at 6.3. Uncovered marks carry
    their distance to the nearest span so a boundary rounding is not mistaken for
    a silent stretch.
    \"\"\"
    refs = _as_ref_events(reference)
    per_cat: dict[str, dict] = {}
    for cat in categories:
        marks = [r.onset_sec for r in refs if r.category == cat]
        spans = [
            (
                float(d.get("start_sec", d.get("peak_sec", 0.0))),
                float(d.get("end_sec", d.get("peak_sec", 0.0))),
            )
            for d in detections
            if str(d.get("category", "")) == cat
        ]

        def _distance(m: float) -> float:
            if not spans:
                return float("inf")
            return min(max(s - m, m - e, 0.0) for s, e in spans)

        distances = {m: _distance(m) for m in marks}
        covered = sum(1 for m in marks if distances[m] <= tolerance_sec)
        active = sum(max(0.0, e - s) for s, e in spans)
        per_cat[cat] = {
            "n_marks": len(marks),
            "covered": covered,
            "recall": round(covered / len(marks), 4) if marks else 0.0,
            "n_spans": len(spans),
            "active_sec": round(active, 3),
            "active_fraction": (
                round(active / duration_sec, 4)
                if duration_sec and duration_sec > 0
                else None
            ),
            "tolerance_sec": tolerance_sec,
            "uncovered_marks": [
                {
                    "mark_sec": round(m, 3),
                    "distance_to_span_sec": (
                        None if distances[m] == float("inf") else round(distances[m], 3)
                    ),
                }
                for m in marks
                if distances[m] > tolerance_sec
            ],
        }
    return {"mode": "coverage", "per_category": per_cat}


def evaluate_events(
    reference,
    events_json: dict | list[dict],
    *,
    duration_sec: float,
    collars_sec: tuple[float, ...] = (0.2, 0.5),
    segment_sec: float = 1.0,
    onset_field: str = "peak_sec",
    sustained_categories: tuple[str, ...] = (),
    sustained_tolerance_sec: float = 0.35,
) -> dict:
    \"\"\"Event-based F1 at several collars, segment-based F1, sustained coverage.

    Categories in ``sustained_categories`` are held out of the F1 blocks and get
    the coverage view instead; see ``coverage_report``.
    \"\"\"
    detections = (
        events_json.get("events", [])
        if isinstance(events_json, dict)
        else list(events_json)
    )
    sustained = set(sustained_categories)
    refs = _as_ref_events(reference)
    onset_refs = [r for r in refs if r.category not in sustained]
    onset_dets = [d for d in detections if str(d.get("category", "")) not in sustained]

    report = {
        "event_based": [
            event_based_prf(
                onset_refs,
                onset_dets,
                onset_collar_sec=c,
                onset_field=onset_field,
            )
            for c in collars_sec
        ],
        "segment_based": segment_based_prf(
            onset_refs,
            onset_dets,
            duration_sec=duration_sec,
            segment_sec=segment_sec,
        ),
    }
    if sustained:
        report["coverage"] = coverage_report(
            refs,
            detections,
            categories=sorted(sustained & {r.category for r in refs}),
            duration_sec=duration_sec,
            tolerance_sec=sustained_tolerance_sec,
        )
    return report


def _fmt_prf(name: str, prf: dict) -> str:
    return (
        name.rjust(16)
        + format(prf["precision"], ".3f").rjust(11)
        + format(prf["recall"], ".3f").rjust(9)
        + format(prf["f1"], ".3f").rjust(8)
        + str(prf["n_ref"]).rjust(7)
        + str(prf["n_det"]).rjust(7)
        + str(prf["tp"]).rjust(5)
        + str(prf["fp"]).rjust(5)
        + str(prf["fn"]).rjust(5)
    )


def format_evaluation(report: dict) -> str:
    header = (
        "        category  precision   recall      F1  n_ref  n_det   TP   FP   FN"
    )
    lines: list[str] = []
    scored = any(
        block["micro"]["n_ref"] or block["micro"]["n_det"]
        for block in report["event_based"]
    )
    if not scored:
        lines.append(
            "Event-based / segment-based F1: no onset references or detections in "
            "this clip, nothing to score (zeros below would be meaningless)."
        )
    for block in report["event_based"] if scored else []:
        collar = block["onset_collar_sec"]
        mean_err = block["mean_abs_onset_error_sec"]
        lines.append(
            "Event-based F1 (onset collar {0} s){1}".format(
                collar,
                ""
                if mean_err is None
                else "  |  mean |onset error| = {0:.3f} s".format(mean_err),
            )
        )
        lines.append(header)
        for cat, prf in block["per_category"].items():
            lines.append(_fmt_prf(cat, prf))
        lines.append(_fmt_prf("MICRO", block["micro"]))
        lines.append("      macro F1: {0:.3f}".format(block["macro_f1"]))
        lines.append("")

    if scored:
        seg = report["segment_based"]
        lines.append("Segment-based F1 ({0} s segments)".format(seg["segment_sec"]))
        lines.append(header)
        for cat, prf in seg["per_category"].items():
            lines.append(_fmt_prf(cat, prf))
        lines.append(_fmt_prf("MICRO", seg["micro"]))
        lines.append("      macro F1: {0:.3f}".format(seg["macro_f1"]))

    cov = report.get("coverage")
    if cov and cov["per_category"]:
        lines.append("")
        lines.append("Sustained coverage (recall only; sparse marks cannot score precision)")
        lines.append("        category   covered   recall  spans  active_s  active_%")
        for cat, row in cov["per_category"].items():
            frac = row["active_fraction"]
            lines.append(
                cat.rjust(16)
                + "{0}/{1}".format(row["covered"], row["n_marks"]).rjust(10)
                + format(row["recall"], ".3f").rjust(9)
                + str(row["n_spans"]).rjust(7)
                + format(row["active_sec"], ".1f").rjust(10)
                + ("-" if frac is None else format(100.0 * frac, ".1f")).rjust(10)
            )
            if row["uncovered_marks"]:
                lines.append(
                    "                  missed marks: "
                    + ", ".join(
                        "{0:.2f} ({1})".format(
                            m["mark_sec"],
                            "no span"
                            if m["distance_to_span_sec"] is None
                            else "{0:.2f}s away".format(m["distance_to_span_sec"]),
                        )
                        for m in row["uncovered_marks"]
                    )
                )

    misses = [
        m
        for block in report["event_based"][:1]
        for m in block["matches"]
        if m["status"] != "hit"
    ]
    if misses:
        lines.append("")
        lines.append("Unmatched at tightest collar:")
        for m in misses:
            if m["status"] == "miss":
                lines.append(
                    "  miss  {0:>12}  ref {1:.3f}".format(
                        m["category"], m["ref_onset_sec"]
                    )
                )
            else:
                lines.append(
                    "  FP    {0:>12}  det {1:.3f}".format(
                        m["category"], m["det_onset_sec"]
                    )
                )
    return "\\n".join(lines)
""",
    "haptic_synthesis.py": """\"\"\"Stitch per-event haptic segments onto a full-length timeline.\"\"\"

from __future__ import annotations

import tempfile
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.ndimage import maximum_filter1d

from haptic_gt.audio_io import INPUT_SR, VIB_SR
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.mask import build_event_mask
from haptic_gt.context.sustained_merge import sustained_coverage_sec
from haptic_gt.context.taxonomy import Taxonomy, load_taxonomy


def _impulsive_accent_events(
    events: list[DetectedEvent],
    taxonomy: Taxonomy,
) -> list[DetectedEvent]:
    \"\"\"Only impulsive categories get bang-style accents (gunshot/explosion/thunder).\"\"\"
    out: list[DetectedEvent] = []
    for ev in events:
        cat = taxonomy.categories.get(ev.category)
        if cat is not None and cat.impulsive:
            out.append(ev)
    return out


def _sustained_events(
    events: list[DetectedEvent],
    taxonomy: Taxonomy,
) -> list[DetectedEvent]:
    out: list[DetectedEvent] = []
    for ev in events:
        cat = taxonomy.categories.get(ev.category)
        if cat is not None and not cat.impulsive:
            out.append(ev)
    return out


def _use_sustained_bed_mask(
    sustained: list[DetectedEvent],
    duration_sec: float,
    taxonomy: Taxonomy,
) -> bool:
    \"\"\"True for intermittent rumble clips; false for a single short vehicle chip.\"\"\"
    if not sustained or duration_sec <= 0:
        return False
    n = len(sustained)
    if n >= taxonomy.sustained_mask_min_events:
        return True
    # One sparse detection (e.g. short tank idle chip) keeps the full bed
    if n < 2:
        return False
    coverage = sustained_coverage_sec(sustained, taxonomy) / duration_sec
    return coverage >= taxonomy.sustained_mask_min_coverage


@dataclass(frozen=True)
class ContinuousProfile:
    \"\"\"
    Shape of the continuous haptic layer rendered between detected events.

    Defaults are tuned against the rule-based reference map
    (`sample/videoplayback_output_haptic_map[1].json`), which keeps ~71% of its
    40 ms windows active at a median intensity of 50/255 while reserving the top
    of the range for event peaks.
    \"\"\"

    enabled: bool = True

    #: Level the continuous layer's loud passages sit at, measured as the 95th
    #: percentile of its envelope rather than its absolute peak so the bed has a
    #: predictable strength regardless of how spiky the algorithm output is.
    base_gain: float = 0.24
    #: Envelope percentile that `base_gain` refers to.
    base_level_pct: float = 0.95

    # -- Input levelling -----------------------------------------------------
    # Algorithms A-D peak-normalize whatever they are handed and apply their own
    # perceptual thresholds, so a quiet passage inside a loud clip renders as
    # silence (A returns literal zeros below its detection threshold). Levelling
    # the audio first keeps every active passage above those thresholds.
    #: Exponent applied to the source envelope when levelling. Lower is flatter.
    agc_strength: float = 0.05
    #: Window used when measuring the source envelope.
    agc_window_ms: float = 50.0
    #: Peak-hold applied to the envelope before deriving gain, so transients are
    #: not amplified along with the quiet material around them.
    agc_hold_ms: float = 150.0
    #: Ceiling on levelling gain, in dB.
    agc_max_gain_db: float = 40.0

    # -- Output dynamics -----------------------------------------------------
    #: Exponent re-imposing macro dynamics on the levelled output, so louder
    #: passages still feel stronger. 1.0 restores the source's full range.
    #: A rumble is one long span whose bursts and lulls are all the haptic has to
    #: convey its rhythm with, so this has to stay well clear of a flat bed: at
    #: 0.30 a burst measured 2.9x its gap came out 0.98x, i.e. no rhythm at all.
    dynamics_strength: float = 0.65
    #: Fraction of the quietest frames muted, so real silence stays silent.
    #: The reference map leaves 29% of its windows empty.
    silence_pct: float = 0.28
    #: Frames louder than this fraction of the peak are never muted, so a clip
    #: that is active end to end is not silenced just to hit `silence_pct`.
    max_floor_ratio: float = 0.15

    # -- Event accents -------------------------------------------------------
    #: Continuous layer is attenuated to this fraction underneath an event so
    #: the accent keeps its full dynamic punch without clipping the sum.
    event_duck: float = 0.28
    #: Ramp applied at the edges of each duck region.
    duck_ramp_ms: float = 30.0
    #: Peak amplitude each event segment is normalized to.
    event_gain: float = 0.92
    #: Accents are placed this far ahead of the audio attack. Event peaks land
    #: within ~10 ms of the attack and the rendered accent within ~10 ms of the
    #: event peak, yet the vibration still feels late: a low-frequency actuator
    #: needs a few cycles to be felt, so a haptic aligned to the sample is felt
    #: after the sound. Leading it by a fraction of that rise time lines the two
    #: up perceptually.
    accent_lead_ms: float = 25.0
    #: Release applied where one accent would still be ringing under the next, so
    #: a volley reads as separate hits instead of one long buzz.
    accent_release_ms: float = 45.0
    #: Peak amplitude the loudest sustained rumble span is normalized to.
    sustained_soft_gain: float = 0.48
    #: Quieter rumble spans are scaled by their own loudness relative to the
    #: loudest one, so a distant drive does not hit as hard as a car alongside.
    #: 1.0 would be literal; some compression keeps the quiet scene perceptible.
    sustained_level_exp: float = 0.6
    #: Floor on that scaling, so the quietest rumble is still felt.
    sustained_level_floor: float = 0.35


def _read_mono(path: Path) -> tuple[np.ndarray, int]:
    audio, sr = sf.read(path, always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32), sr


def _write_wav(path: Path, audio: np.ndarray, sample_rate: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, audio, sample_rate, subtype="PCM_16")


def _extract_clip_wav(
    source_wav: Path,
    start_sec: float,
    end_sec: float,
    dest: Path,
    *,
    sample_rate: int = INPUT_SR,
) -> None:
    audio, sr = _read_mono(source_wav)
    if sr != sample_rate:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate

    s0 = max(0, int(start_sec * sr))
    s1 = min(len(audio), int(end_sec * sr))
    _write_wav(dest, audio[s0:s1], sr)


def _event_clip_bounds(
    ev: DetectedEvent,
    duration_sec: float,
    *,
    min_clip_sec: float = 0.6,
) -> tuple[float, float]:
    \"\"\"Bounds that include the onset peak with enough post-event context.\"\"\"
    start = max(0.0, min(ev.start_sec, ev.peak_sec - 0.05))
    end = max(ev.end_sec, ev.peak_sec + 0.35)
    if end - start < min_clip_sec:
        end = min(duration_sec, start + min_clip_sec)
        start = max(0.0, end - min_clip_sec)
    end = min(duration_sec, end)
    return start, end


def _haptic_onset_sec(segment: np.ndarray, sample_rate: int, *, ratio: float = 0.18) -> float:
    \"\"\"First time the haptic |signal| crosses a fraction of its peak (attack).\"\"\"
    if segment.size == 0:
        return 0.0
    abs_seg = np.abs(segment)
    peak = float(np.max(abs_seg))
    if peak < 1e-10:
        return 0.0
    thr = peak * ratio
    hits = np.where(abs_seg >= thr)[0]
    if hits.size == 0:
        return float(np.argmax(abs_seg) / sample_rate)
    return float(hits[0] / sample_rate)


def _normalize_segment(segment: np.ndarray, *, target: float = 0.85) -> np.ndarray:
    \"\"\"Scale segment so its peak lands at `target` (haptic actuator headroom).\"\"\"
    peak = float(np.max(np.abs(segment)))
    if peak < 1e-10:
        return segment
    return segment * (target / peak)


def _scale_by_energy(
    segment: np.ndarray,
    *,
    target: float,
    crest: float = 0.707,
    ceiling: float = 0.95,
) -> np.ndarray:
    \"\"\"Level a rumble span by its energy instead of its loudest spike.

    Peak normalizing a span makes one holding a sharp transient come out quiet
    and a smooth one come out loud, which puts the power on the wrong span: a
    distant drive rendered as a steady tone then feels stronger than a car
    alongside whose burst has an attack in it. ``crest`` is the peak/RMS ratio of
    a sine, so a smooth rumble still lands on ``target``; spikier material is
    only pulled back if it would run out of headroom.
    \"\"\"
    rms = float(np.sqrt(np.mean(np.square(segment))))
    peak = float(np.max(np.abs(segment)))
    if rms < 1e-10 or peak < 1e-10:
        return segment
    scale = crest * target / rms
    if peak * scale > ceiling:
        scale = ceiling / peak
    return segment * scale


def _release_tail(segment: np.ndarray, keep: int, release: int) -> np.ndarray:
    \"\"\"Cut a segment to ``keep`` samples with a cosine release, not a click.\"\"\"
    if keep <= 0 or segment.size <= keep:
        return segment
    out = segment[:keep].copy()
    r = min(release, keep)
    if r > 1:
        out[-r:] *= ((1.0 + np.cos(np.linspace(0.0, np.pi, r))) / 2.0).astype(out.dtype)
    return out


def _source_levels(
    audio: np.ndarray,
    sample_rate: int,
    events: list[DetectedEvent],
) -> list[float]:
    \"\"\"RMS of the source under each event span.\"\"\"
    levels: list[float] = []
    for ev in events:
        s0 = max(0, int(ev.start_sec * sample_rate))
        s1 = min(len(audio), int(ev.end_sec * sample_rate))
        seg = audio[s0:s1]
        levels.append(float(np.sqrt(np.mean(np.square(seg)))) if seg.size else 0.0)
    return levels


def _frame_envelope(
    signal: np.ndarray,
    sample_rate: int,
    window_ms: float,
    hold_ms: float = 0.0,
) -> tuple[np.ndarray, np.ndarray]:
    \"\"\"
    Short-time RMS envelope with frame centres in seconds.

    `hold_ms` widens each frame to the loudest value nearby. Deriving gain from
    a peak-held envelope keeps a transient from dragging up the gain applied to
    the quiet material next to it.
    \"\"\"
    frame = max(1, int(sample_rate * window_ms / 1000.0))
    n_frames = max(1, int(np.ceil(signal.size / frame)))
    padded = np.pad(signal, (0, n_frames * frame - signal.size))
    envelope = np.sqrt(np.mean(np.square(padded.reshape(n_frames, frame)), axis=1))

    hold_radius = int(round(hold_ms / max(window_ms, 1e-6)))
    if hold_radius > 0 and envelope.size > 1:
        envelope = maximum_filter1d(envelope, size=2 * hold_radius + 1, mode="nearest")

    centers_sec = (np.arange(n_frames, dtype=np.float64) * frame + frame / 2.0) / sample_rate
    return envelope, centers_sec


def _resample_curve(
    values: np.ndarray,
    centers_sec: np.ndarray,
    n_samples: int,
    sample_rate: int,
) -> np.ndarray:
    \"\"\"Interpolate a per-frame curve onto a sample timeline at `sample_rate`.\"\"\"
    if values.size == 1:
        return np.full(n_samples, values[0], dtype=np.float32)
    positions = np.arange(n_samples, dtype=np.float64) / sample_rate
    return np.interp(positions, centers_sec, values).astype(np.float32)


def _level_audio(audio: np.ndarray, sample_rate: int, profile: ContinuousProfile) -> np.ndarray:
    \"\"\"Flatten the source envelope so every active passage drives the algorithm.\"\"\"
    envelope, centers = _frame_envelope(
        audio, sample_rate, profile.agc_window_ms, profile.agc_hold_ms
    )
    peak = float(np.max(envelope))
    if peak < 1e-10:
        return audio

    normalized = np.maximum(envelope / peak, 1e-6)
    max_gain = float(10.0 ** (profile.agc_max_gain_db / 20.0))
    gain = np.clip(np.power(normalized, profile.agc_strength - 1.0), 0.0, max_gain)
    return audio * _resample_curve(gain, centers, audio.size, sample_rate)


def _dynamics_curve(
    source_audio: np.ndarray,
    source_sr: int,
    profile: ContinuousProfile,
) -> tuple[np.ndarray, np.ndarray]:
    \"\"\"
    Per-frame gain that re-imposes compressed source dynamics, plus a gate.

    Both are derived from the source audio rather than the algorithm output, so
    the continuous layer rises and falls with what is actually audible.
    \"\"\"
    envelope, centers = _frame_envelope(source_audio, source_sr, profile.agc_window_ms)
    peak = float(np.max(envelope))
    if peak < 1e-10:
        return np.zeros_like(envelope), centers

    normalized = envelope / peak
    threshold = min(
        float(np.quantile(normalized, np.clip(profile.silence_pct, 0.0, 0.9))),
        profile.max_floor_ratio,
    )
    gate = (normalized > threshold).astype(np.float64)
    shape = np.power(np.maximum(normalized, 1e-6), profile.dynamics_strength)
    return shape * gate, centers


def _duck_envelope(
    total_samples: int,
    spans: list[tuple[int, int]],
    sample_rate: int,
    *,
    duck: float,
    ramp_ms: float,
) -> np.ndarray:
    \"\"\"Gain envelope that dips to `duck` across each span with cosine edges.\"\"\"
    envelope = np.ones(total_samples, dtype=np.float32)
    if not spans:
        return envelope

    ramp_len = max(1, int(sample_rate * ramp_ms / 1000.0))
    ramp = (1.0 - np.cos(np.linspace(0.0, np.pi, ramp_len))) / 2.0

    for start, end in spans:
        start = max(0, start)
        end = min(total_samples, end)
        if end <= start:
            continue
        envelope[start:end] = np.minimum(envelope[start:end], duck)

        lead_start = max(0, start - ramp_len)
        lead_len = start - lead_start
        if lead_len > 0:
            fade = 1.0 - (1.0 - duck) * ramp[-lead_len:]
            envelope[lead_start:start] = np.minimum(envelope[lead_start:start], fade)

        tail_end = min(total_samples, end + ramp_len)
        tail_len = tail_end - end
        if tail_len > 0:
            fade = 1.0 - (1.0 - duck) * ramp[::-1][:tail_len]
            envelope[end:tail_end] = np.minimum(envelope[end:tail_end], fade)

    return envelope


def _run_algorithm(
    process_file: Callable[..., object],
    clip_in: Path,
    clip_out: Path,
    *,
    output_sr: int,
    process_kwargs: dict,
) -> np.ndarray:
    process_file(clip_in, clip_out, **process_kwargs)
    segment, seg_sr = _read_mono(clip_out)
    if seg_sr != output_sr:
        import librosa

        segment = librosa.resample(segment, orig_sr=seg_sr, target_sr=output_sr)
    return segment.astype(np.float32)


def _render_continuous_layer(
    source_audio: np.ndarray,
    source_sr: int,
    process_file: Callable[..., object],
    work_dir: Path,
    *,
    output_sr: int,
    total_samples: int,
    process_kwargs: dict,
    profile: ContinuousProfile,
    dynamics: np.ndarray,
) -> np.ndarray:
    \"\"\"Render the whole clip through an algorithm as a low-level bed.\"\"\"
    levelled = _level_audio(source_audio, source_sr, profile)
    levelled = _normalize_segment(levelled, target=0.95)

    clip_in = work_dir / "base_in.wav"
    clip_out = work_dir / "base_out.wav"
    _write_wav(clip_in, levelled, source_sr)
    base = _run_algorithm(
        process_file, clip_in, clip_out, output_sr=output_sr, process_kwargs=process_kwargs
    )

    if base.size < total_samples:
        base = np.pad(base, (0, total_samples - base.size))
    base = base[:total_samples]

    base = base * dynamics

    envelope, _ = _frame_envelope(base, output_sr, 20.0)
    active = envelope[envelope > 1e-6]
    if active.size == 0:
        return np.zeros_like(base)
    reference = float(np.quantile(active, profile.base_level_pct))
    if reference < 1e-9:
        return _normalize_segment(base, target=profile.base_gain)
    return np.clip(base * (profile.base_gain / reference), -1.0, 1.0)


def stitch_algorithm_output(
    source_wav: Path,
    events: list[DetectedEvent],
    output_path: Path,
    process_file: Callable[..., object],
    *,
    input_sr: int = INPUT_SR,
    output_sr: int = VIB_SR,
    process_kwargs: dict | None = None,
    continuous: ContinuousProfile | None = None,
    taxonomy: Taxonomy | None = None,
) -> None:
    \"\"\"
    Run an algorithm over the whole clip and mix per-event accents on top.

    Two layers are produced. The continuous layer renders the entire source
    audio so sustained content still produces motion. Impulsive events
    (gunshot / explosion / thunder) add stronger accents aligned to
    ``peak_sec``, with the bed ducked underneath.

    Sustained categories (e.g. vehicle):
    - Sparse / short detection → full continuous bed (steady rumble clips).
    - Many / high-coverage spans → bed masked to those spans + soft
      time-aligned rumble segments (intermittent car rumble), no bang duck.
    \"\"\"
    process_kwargs = process_kwargs or {}
    profile = continuous or ContinuousProfile()
    taxonomy = taxonomy or load_taxonomy()

    audio, sr = _read_mono(source_wav)
    if sr != input_sr:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=input_sr)
        sr = input_sr

    duration_sec = len(audio) / sr
    total_samples = max(1, int(round(duration_sec * output_sr)))
    timeline = np.zeros(total_samples, dtype=np.float32)

    sustained = _sustained_events(events, taxonomy)
    intermittent = profile.enabled and _use_sustained_bed_mask(
        sustained, duration_sec, taxonomy
    )

    # With continuous bed on: bang accents only for impulsive hits.
    # With continuous off: keep legacy behaviour (accent every gated event).
    if profile.enabled:
        accent_events = _impulsive_accent_events(events, taxonomy)
    else:
        accent_events = list(events)

    if not accent_events and not profile.enabled and not intermittent:
        _write_wav(output_path, timeline, output_sr)
        return

    # One dynamics curve for the whole clip, shared by the bed and the rumble
    # segments so both rise and fall with what is audible.
    dyn_values, dyn_centers = _dynamics_curve(audio, sr, profile)
    dynamics = _resample_curve(dyn_values, dyn_centers, total_samples, output_sr)

    soft_placements: list[tuple[int, np.ndarray]] = []
    placements: list[tuple[int, np.ndarray]] = []
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)

        for i, ev in enumerate(accent_events):
            clip_start, clip_end = _event_clip_bounds(ev, duration_sec)
            clip_in = tmp_path / f"event_{i}_in.wav"
            clip_out = tmp_path / f"event_{i}_out.wav"
            _extract_clip_wav(source_wav, clip_start, clip_end, clip_in, sample_rate=sr)
            segment = _run_algorithm(
                process_file,
                clip_in,
                clip_out,
                output_sr=output_sr,
                process_kwargs=process_kwargs,
            )
            segment = _normalize_segment(segment, target=profile.event_gain)

            onset_sec = _haptic_onset_sec(segment, output_sr)
            lead_sec = profile.accent_lead_ms / 1000.0
            start_idx = int(round((ev.peak_sec - onset_sec - lead_sec) * output_sr))
            if start_idx < 0:
                segment = segment[-start_idx:]
                start_idx = 0
            if start_idx >= total_samples or segment.size == 0:
                continue
            placements.append((start_idx, segment))

        # A bang still ringing when the next one lands buries its attack, which
        # reads as the next hit arriving late rather than as one loud volley.
        placements.sort(key=lambda p: p[0])
        release = max(1, int(output_sr * profile.accent_release_ms / 1000.0))
        for i in range(len(placements) - 1):
            start, segment = placements[i]
            keep = placements[i + 1][0] - start
            if 0 < keep < segment.size:
                placements[i] = (start, _release_tail(segment, keep, release))

        if intermittent:
            levels = _source_levels(audio, sr, sustained)
            loudest = max(levels) if levels else 0.0
            for i, ev in enumerate(sustained):
                span = max(0.05, ev.end_sec - ev.start_sec)
                clip_in = tmp_path / f"sustained_{i}_in.wav"
                clip_out = tmp_path / f"sustained_{i}_out.wav"
                _extract_clip_wav(
                    source_wav, ev.start_sec, ev.end_sec, clip_in, sample_rate=sr
                )
                segment = _run_algorithm(
                    process_file,
                    clip_in,
                    clip_out,
                    output_sr=output_sr,
                    process_kwargs=process_kwargs,
                )
                # Normalizing every span to the same peak makes a distant drive hit
                # as hard as a car alongside, which is the rumble's power gone.
                relative = 1.0
                if loudest > 1e-9:
                    relative = (levels[i] / loudest) ** profile.sustained_level_exp
                relative = float(np.clip(relative, profile.sustained_level_floor, 1.0))
                segment = _scale_by_energy(
                    segment, target=profile.sustained_soft_gain * relative
                )
                target_len = max(1, int(round(span * output_sr)))
                if segment.size < target_len:
                    segment = np.pad(segment, (0, target_len - segment.size))
                elif segment.size > target_len:
                    segment = segment[:target_len]
                start_idx = max(0, int(round(ev.start_sec * output_sr)))
                if start_idx >= total_samples or segment.size == 0:
                    continue
                # A rumble span is minutes of one event; its bursts and lulls are
                # all the rhythm the haptic has. Algorithm A normalizes every frame
                # to a constant level internally, so without this the whole span
                # comes out as one flat buzz.
                shape = dynamics[start_idx : start_idx + segment.size]
                if shape.size < segment.size:
                    shape = np.pad(shape, (0, segment.size - shape.size), mode="edge")
                peak = float(np.max(shape)) if shape.size else 0.0
                if peak > 1e-6:
                    segment = segment * (shape / peak)
                soft_placements.append((start_idx, segment))

        if profile.enabled:
            base = _render_continuous_layer(
                audio,
                sr,
                process_file,
                tmp_path,
                output_sr=output_sr,
                total_samples=total_samples,
                process_kwargs=process_kwargs,
                profile=profile,
                dynamics=dynamics,
            )
            if intermittent:
                # Rumble on only where sustained events fired; quiet gaps stay quiet
                mask = build_event_mask(
                    total_samples, output_sr, sustained, fade_ms=40.0
                )
                base *= mask
            spans = [(start, start + len(seg)) for start, seg in placements]
            base *= _duck_envelope(
                total_samples,
                spans,
                output_sr,
                duck=profile.event_duck,
                ramp_ms=profile.duck_ramp_ms,
            )
            timeline += base

    for start_idx, segment in soft_placements:
        end_idx = min(total_samples, start_idx + len(segment))
        seg_len = end_idx - start_idx
        if seg_len > 0:
            timeline[start_idx:end_idx] += segment[:seg_len]

    for start_idx, segment in placements:
        end_idx = min(total_samples, start_idx + len(segment))
        seg_len = end_idx - start_idx
        if seg_len > 0:
            timeline[start_idx:end_idx] += segment[:seg_len]

    np.clip(timeline, -1.0, 1.0, out=timeline)
    _write_wav(output_path, timeline, output_sr)
""",
    "pipeline.py": """\"\"\"End-to-end pipeline aligned with Sound2Hap signal processing.\"\"\"

from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

from typing import Any

from haptic_gt.algorithms import freq_shift, haptic_gen, percept, pitch_match, rule_based
from haptic_gt.audio_io import INPUT_SR, VIB_SR, extract_audio_from_video, prepare_source_wav
from haptic_gt.context import detect_events
from haptic_gt.context.detector import EVENTS_JSON_NAME, GATED_AUDIO_NAME, EventResult
from haptic_gt.context.frozen_fusion import DetectedEvent
from haptic_gt.context.manual_events import events_from_manual, vehicle_events_from_peaks
from haptic_gt.context.mask import apply_gate, events_for_haptic_gate, resolve_gate_categories
from haptic_gt.context.taxonomy import load_taxonomy
from haptic_gt.haptic_synthesis import ContinuousProfile, stitch_algorithm_output


def _replace_vehicle_with_manual_peaks(
    events: list[DetectedEvent],
    peaks_sec: list[float],
    taxonomy,
) -> list[DetectedEvent]:
    \"\"\"Keep impulsive auto events; replace vehicle spans with hand-marked rumble peaks.\"\"\"
    kept = []
    for ev in events:
        cat = taxonomy.categories.get(ev.category)
        if cat is not None and not cat.impulsive and ev.category == "vehicle":
            continue
        if ev.category == "vehicle":
            continue
        kept.append(ev)
    kept.extend(vehicle_events_from_peaks(peaks_sec, taxonomy=taxonomy))
    kept.sort(key=lambda e: e.start_sec)
    return kept

OUTPUT_NAMES = {
    "source_audio": "source_audio.wav",
    "gated_audio": "gated_audio.wav",
    "events_json": "events.json",
    "algorithm_a_perception_mapping": "algorithm_a_perception_mapping.wav",
    "algorithm_b_frequency_shifting": "algorithm_b_frequency_shifting.wav",
    "algorithm_c_pitch_matching": "algorithm_c_pitch_matching.wav",
    "algorithm_d_haptic_gen": "algorithm_d_haptic_gen.wav",
    "algorithm_e_rule_based": "algorithm_e_rule_based.wav",
    "algorithm_e_rule_based_json": "algorithm_e_rule_based.json",
}


@dataclass
class CandidateTracks:
    \"\"\"Paths to source audio, Sound2Hap A–D, and rule-based E.\"\"\"

    source_wav: Path
    algorithm_a: Path | None
    algorithm_b: Path | None
    algorithm_c: Path | None
    algorithm_d: Path | None
    algorithm_e: Path | None
    algorithm_e_json: Path | None
    output_dir: Path
    haptic_input_wav: Path | None
    input_sample_rate: int = INPUT_SR
    output_sample_rate: int = VIB_SR
    pitch_match_info: dict | None = None
    events: list[DetectedEvent] | None = None
    events_json: Path | None = None
    no_events_detected: bool = False
    no_haptic_events: bool = False
    gate_categories_used: list[str] = field(default_factory=list)

    def save_all(self) -> dict[str, Path]:
        out: dict[str, Path] = {"source_audio": self.source_wav}
        if self.algorithm_a and self.algorithm_a.exists():
            out["algorithm_a_perception_mapping"] = self.algorithm_a
        if self.algorithm_b and self.algorithm_b.exists():
            out["algorithm_b_frequency_shifting"] = self.algorithm_b
        if self.algorithm_c and self.algorithm_c.exists():
            out["algorithm_c_pitch_matching"] = self.algorithm_c
        if self.algorithm_d and self.algorithm_d.exists():
            out["algorithm_d_haptic_gen"] = self.algorithm_d
        if self.algorithm_e and self.algorithm_e.exists():
            out["algorithm_e_rule_based"] = self.algorithm_e
        if self.algorithm_e_json and self.algorithm_e_json.exists():
            out["algorithm_e_rule_based_json"] = self.algorithm_e_json
        if self.events_json and self.events_json.exists():
            out["events_json"] = self.events_json
        if (
            self.haptic_input_wav is not None
            and self.haptic_input_wav.exists()
            and self.haptic_input_wav != self.source_wav
        ):
            out["gated_audio"] = self.haptic_input_wav
        return out


def _update_events_json_haptics(
    events_json: Path,
    haptic_paths: dict[str, str],
) -> None:
    if not events_json.exists():
        return
    payload = json.loads(events_json.read_text(encoding="utf-8"))
    existing = payload.get("haptic_outputs", {})
    existing.update(haptic_paths)
    payload["haptic_outputs"] = existing
    events_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def generate_candidate_tracks(
    input_path: str | Path,
    output_dir: str | Path,
    *,
    from_video: bool = True,
    content_type: str = "game",
    enable_context_detection: bool = True,
    taxonomy_path: str | Path | None = None,
    gate_categories: list[str] | None = None,
    continuous_haptics: bool = True,
    continuous_profile: ContinuousProfile | None = None,
    manual_events: list[dict[str, Any]] | dict[str, Any] | str | Path | None = None,
    manual_rumble_peaks: list[float] | None = None,
) -> CandidateTracks:
    \"\"\"
    Run context detection (optional), Sound2Hap A–D, and rule-based E.

    When gate-eligible events are detected, A–D are stitched onto a full-length
    timeline (one WAV per algorithm). When none match gate_categories, A–D are
    skipped. Rule-based E always runs on the ungated mix (plus video frames
    when ``from_video`` is true).

    With `continuous_haptics` on, each algorithm also renders the full clip as a
    low-level continuous layer underneath the event accents, so sustained sounds
    (rumble, rain, engines) keep vibrating instead of leaving silent gaps.

    Pass ``manual_events`` (list of dicts, single event dict, or path to
    events.json) to skip AST/ViViT and trust hand-labeled start/peak/end times.

    Pass ``manual_rumble_peaks`` to replace auto ``vehicle`` events with your
    marked rumble times (keeps auto gunshot/explosion). Use when calib shows
    RMS rise cannot separate true rumbles from engine-bed false positives.
    \"\"\"
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    taxonomy = load_taxonomy(taxonomy_path)
    gate_cats = resolve_gate_categories(taxonomy, gate_categories)

    source_wav = output_dir / OUTPUT_NAMES["source_audio"]
    if from_video:
        extract_audio_from_video(input_path, source_wav, sr=INPUT_SR)
    else:
        prepare_source_wav(input_path, source_wav, sr=INPUT_SR)

    events: list[DetectedEvent] | None = None
    events_json_path: Path | None = None
    no_events_detected = False
    no_haptic_events = False
    haptic_input: Path | None = None
    gate_events: list[DetectedEvent] = []

    out_a = output_dir / OUTPUT_NAMES["algorithm_a_perception_mapping"]
    out_b = output_dir / OUTPUT_NAMES["algorithm_b_frequency_shifting"]
    out_c = output_dir / OUTPUT_NAMES["algorithm_c_pitch_matching"]
    out_d = output_dir / OUTPUT_NAMES["algorithm_d_haptic_gen"]
    out_e = output_dir / OUTPUT_NAMES["algorithm_e_rule_based"]
    out_e_json = output_dir / OUTPUT_NAMES["algorithm_e_rule_based_json"]

    if manual_events is not None:
        events = events_from_manual(manual_events, taxonomy)
        gate_events = events_for_haptic_gate(events, taxonomy, gate_categories=gate_cats)
        no_events_detected = len(events) == 0
        no_haptic_events = len(gate_events) == 0
        result = EventResult(
            events=events,
            no_events_detected=no_events_detected,
            no_haptic_events=no_haptic_events,
            timeline_hz=taxonomy.timeline_hz,
            gate_categories_used=gate_cats,
        )
        events_json_path = output_dir / EVENTS_JSON_NAME
        events_json_path.write_text(
            json.dumps(result.to_dict(output_dir=output_dir), indent=2),
            encoding="utf-8",
        )
        result.events_json = events_json_path
        if gate_events:
            gated = output_dir / GATED_AUDIO_NAME
            apply_gate(
                source_wav,
                gated,
                events,
                taxonomy=taxonomy,
                gate_categories=gate_cats,
            )
            haptic_input = gated
            result.gated_wav = gated
            result.haptic_outputs["gated_audio"] = str(gated)
            # Refresh JSON so gated path is recorded
            events_json_path.write_text(
                json.dumps(result.to_dict(output_dir=output_dir), indent=2),
                encoding="utf-8",
            )
    elif enable_context_detection and from_video:
        event_result = detect_events(
            input_path,
            source_wav,
            output_dir,
            taxonomy_path=taxonomy_path,
            write_gated=True,
            gate_categories=gate_categories,
        )
        events = event_result.events
        events_json_path = event_result.events_json
        if manual_rumble_peaks:
            events = _replace_vehicle_with_manual_peaks(
                events or [], manual_rumble_peaks, taxonomy
            )
            # Rewrite events.json + gated audio with replaced vehicle spans
            gate_events = events_for_haptic_gate(events, taxonomy, gate_categories=gate_cats)
            result = EventResult(
                events=events,
                no_events_detected=len(events) == 0,
                no_haptic_events=len(gate_events) == 0,
                timeline_hz=taxonomy.timeline_hz,
                gate_categories_used=gate_cats,
            )
            events_json_path = output_dir / EVENTS_JSON_NAME
            events_json_path.write_text(
                json.dumps(result.to_dict(output_dir=output_dir), indent=2),
                encoding="utf-8",
            )
            if gate_events:
                gated = output_dir / GATED_AUDIO_NAME
                apply_gate(
                    source_wav,
                    gated,
                    events,
                    taxonomy=taxonomy,
                    gate_categories=gate_cats,
                )
                haptic_input = gated
            no_events_detected = len(events) == 0
            no_haptic_events = len(gate_events) == 0
        else:
            no_events_detected = event_result.no_events_detected
            no_haptic_events = event_result.no_haptic_events
            gate_events = events_for_haptic_gate(
                events or [], taxonomy, gate_categories=gate_cats
            )
            if event_result.gated_wav is not None and event_result.gated_wav.exists():
                haptic_input = event_result.gated_wav
    elif enable_context_detection:
        events_json_path = output_dir / OUTPUT_NAMES["events_json"]
        payload = {
            "no_events_detected": True,
            "no_haptic_events": True,
            "gate_categories_used": gate_cats,
            "timeline_hz": taxonomy.timeline_hz,
            "haptic_outputs": {},
            "events": [],
        }
        events_json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        no_events_detected = True
        no_haptic_events = True
    else:
        gate_events = []

    profile = continuous_profile or ContinuousProfile(enabled=continuous_haptics)

    pitch_info = None
    if gate_events:
        stitch_algorithm_output(
            source_wav,
            gate_events,
            out_a,
            percept.process_file,
            process_kwargs={"content": content_type},
            continuous=profile,
            taxonomy=taxonomy,
        )
        stitch_algorithm_output(
            source_wav,
            gate_events,
            out_b,
            freq_shift.process_file,
            continuous=profile,
            taxonomy=taxonomy,
        )
        stitch_algorithm_output(
            source_wav,
            gate_events,
            out_c,
            pitch_match.process_file,
            continuous=profile,
            taxonomy=taxonomy,
        )
        stitch_algorithm_output(
            source_wav,
            gate_events,
            out_d,
            haptic_gen.process_file,
            continuous=profile,
            taxonomy=taxonomy,
        )

        if events_json_path is not None:
            _update_events_json_haptics(
                events_json_path,
                {
                    "algorithm_a": OUTPUT_NAMES["algorithm_a_perception_mapping"],
                    "algorithm_b": OUTPUT_NAMES["algorithm_b_frequency_shifting"],
                    "algorithm_c": OUTPUT_NAMES["algorithm_c_pitch_matching"],
                    "algorithm_d": OUTPUT_NAMES["algorithm_d_haptic_gen"],
                },
            )

    rule_based.process_file(
        source_wav,
        out_e,
        video_path=input_path if from_video else None,
        json_path=out_e_json,
    )
    if events_json_path is not None:
        _update_events_json_haptics(
            events_json_path,
            {"algorithm_e": OUTPUT_NAMES["algorithm_e_rule_based"]},
        )

    return CandidateTracks(
        source_wav=source_wav,
        algorithm_a=out_a if gate_events else None,
        algorithm_b=out_b if gate_events else None,
        algorithm_c=out_c if gate_events else None,
        algorithm_d=out_d if gate_events else None,
        algorithm_e=out_e,
        algorithm_e_json=out_e_json,
        output_dir=output_dir,
        haptic_input_wav=haptic_input,
        pitch_match_info=pitch_info,
        events=events,
        events_json=events_json_path,
        no_events_detected=no_events_detected,
        no_haptic_events=no_haptic_events,
        gate_categories_used=gate_cats,
    )
""",
    "utils/__init__.py": """\"\"\"Utils package.\"\"\"
""",
    "utils/normalization.py": """\"\"\"Peak/RMS/loudness normalization (from Sound2Hap).\"\"\"

from __future__ import annotations

import sys
import typing as tp

import torch
import torchaudio


def normalize_loudness(
    wav: torch.Tensor,
    sample_rate: int,
    loudness_headroom_db: float = 14,
    loudness_compressor: bool = False,
    energy_floor: float = 2e-3,
) -> torch.Tensor:
    energy = wav.pow(2).mean().sqrt().item()
    if energy < energy_floor:
        return wav
    transform = torchaudio.transforms.Loudness(sample_rate)
    input_loudness_db = transform(wav).item()
    delta_loudness = -loudness_headroom_db - input_loudness_db
    gain = 10.0 ** (delta_loudness / 20.0)
    output = gain * wav
    if loudness_compressor:
        output = torch.tanh(output)
    return output


def _clip_wav(
    wav: torch.Tensor,
    log_clipping: bool = False,
    stem_name: tp.Optional[str] = None,
) -> None:
    max_scale = wav.abs().max()
    if log_clipping and max_scale > 1:
        clamp_prob = (wav.abs() > 1).float().mean().item()
        print(
            f"CLIPPING {stem_name or ''} happening with proba:",
            clamp_prob,
            "maximum scale:",
            max_scale.item(),
            file=sys.stderr,
        )
    wav.clamp_(-1, 1)


def normalize_audio(
    wav: torch.Tensor,
    normalize: bool = True,
    strategy: str = "peak",
    peak_clip_headroom_db: float = 1,
    peak_normalize_db_clamp: float = 0,
    rms_headroom_db: float = 18,
    loudness_headroom_db: float = 14,
    loudness_compressor: bool = False,
    log_clipping: bool = False,
    sample_rate: tp.Optional[int] = None,
    stem_name: tp.Optional[str] = None,
) -> torch.Tensor:
    scale_peak = 10 ** (-peak_clip_headroom_db / 20)
    normalize_peak = 10 ** (peak_normalize_db_clamp / 20)
    scale_rms = 10 ** (-rms_headroom_db / 20)
    if strategy == "peak":
        wav_max = wav.abs().max()
        rescaling = (scale_peak / wav_max).clamp(max=(normalize_peak / wav_max).clamp(min=1))
        if normalize or rescaling < 1:
            wav = wav * rescaling
    elif strategy == "clip":
        wav = wav.clamp(-scale_peak, scale_peak)
    elif strategy == "rms":
        mono = wav.mean(dim=0)
        rescaling = scale_rms / mono.pow(2).mean().sqrt()
        if normalize or rescaling < 1:
            wav = wav * rescaling
        _clip_wav(wav, log_clipping=log_clipping, stem_name=stem_name)
    elif strategy == "loudness":
        assert sample_rate is not None, "Loudness normalization requires sample rate."
        wav = normalize_loudness(
            wav, sample_rate, loudness_headroom_db, loudness_compressor
        )
        _clip_wav(wav, log_clipping=log_clipping, stem_name=stem_name)
    else:
        assert wav.abs().max() < 1
        assert strategy in ("", "none"), f"Unexpected strategy: '{strategy}'"
    return wav
""",
    "context/taxonomy.yaml": """timeline_hz: 100
context_detector_threshold: 0.85
encoder_threshold: 0.35
impulsive_encoder_threshold: 0.30
# Lower threshold for sustained categories (vehicle, weather) — they score lower than explosions
sustained_encoder_threshold: 0.25
# Minimum seconds between impulsive peaks (separate shots / blasts)
# 0.45 keeps volley shots ~0.5 s apart (11.16 then 11.66)
impulsive_min_peak_distance_sec: 0.45
# Half-width used as minimum post-onset content for impulsive gates
impulsive_event_half_width_sec: 0.45
impulsive_onset_search_radius_sec: 0.75
impulsive_decay_tail_sec: 0.85
impulsive_decay_threshold: 0.12
# Short pre-roll before detected onset (gate start)
impulsive_pre_roll_sec: 0.08
# Cap only used as a soft metadata hint; refine no longer recenters spans
sustained_max_gate_sec: 2.5
# Only glue nearly-touching chips — larger gaps are separate rumble bursts
sustained_merge_gap_sec: 0.35
# Rise is diagnostic only (calib). Vehicle gate uses absolute local RMS.
sustained_burst_rise_ratio: 1.10
sustained_burst_pre_sec: 0.7
sustained_burst_post_sec: 1.5
sustained_burst_max_sec: 2.5
# Loudness gate: haptic windows = RMS islands (not AST start/end slabs)
sustained_salience_mid_pct: 50
sustained_salience_loud_pct: 99
sustained_salience_mix: 0.40
sustained_salience_min_sep: 1.45
# Frames below this fraction of the loud level are silence, excluded from percentiles
sustained_salience_silence_frac: 0.05
# A rumble that eases off and comes back this many times is a rhythm, and its gaps
# must stay quiet. Once bursts fill more than half a scene the median sits inside
# them, so loudness percentiles alone read the scene as steady; what separates an
# intermittent rumble from a drive filmed from a wider angle is that the quiet
# stretches repeat and are short, instead of being one long stretch.
sustained_rhythm_min_gaps: 2
sustained_rhythm_gap_max_sec: 2.0
# ...and that the two levels really are two levels. A steady idle ripples around
# its own split threshold, which would otherwise look like a rhythm.
sustained_rhythm_level_ratio: 2.0
# Scenes: the loudness floor is measured per scene, not per clip, so a car rumble
# recorded close cannot set the floor for a tank drive recorded far away in the
# same cut-together clip.
# A scene boundary is a level step this large that holds on both sides...
sustained_scene_level_ratio: 2.5
# ...for at least this long. A 2 s rumble burst over an engine bed is not a scene:
# it has to keep being measured against that bed, not against itself.
sustained_scene_min_sec: 3.5
sustained_scene_block_sec: 2.0
# Each side of a boundary has to hold its level within this factor. Without it a
# loud passage inside one scene reads as two boundaries and the stretch between
# them gets a floor of its own.
sustained_scene_stable_ratio: 2.0
# No scene floor may fall below this fraction of the clip's loudest level. Without
# it a scene holding nothing but room tone or a quiet engine bed measures itself,
# finds one loudness mode and gates the whole thing on as rumble. Measured cases
# sit at 0.12-0.13 (bed, must stay out) and 0.21-0.23 (drive recorded far away,
# must stay in).
sustained_salience_scene_floor_frac: 0.16
# Island edges walk out to this fraction of the threshold (rumble ramps before it gates)
sustained_island_edge_frac: 0.75
sustained_island_max_extend_sec: 0.4
sustained_salience_min_sec: 0.50
# Only bridge tiny holes; 0.3 s was gluing 24s rumble through 28s dip to 29–32
sustained_salience_gap_sec: 0.18
sustained_salience_peak_win_sec: 0.15
# Drop spiky blobs (34–35) whose closed island is mostly below threshold
sustained_salience_min_duty: 0.55
# Short loud onset with no island: crop AST span, never keep a 10 s slab
sustained_salience_max_orphan_sec: 1.2
# Onset proposal stage (before AST/ViViT)
proposal_rms_hop_ms: 5.0
proposal_threshold_ratio: 0.18
proposal_search_pad_sec: 0.75
proposal_window_sec: 1.0
# Sparse scan for sustained sounds (vehicle, human activity) without sharp onsets
proposal_sustained_hop_sec: 1.0
# Impulsive peak snap: search around classifier hint, snap via spectral flux.
# Lookback stays short: a 0.9 s window let a clank 0.88 s earlier steal the onset.
impulsive_onset_back_sec: 0.35
impulsive_onset_forward_sec: 1.2
# Only tiny clips look back to t=0 (4 s muzzle clip). Mixed 16 s clips must not.
impulsive_short_clip_sec: 8.0
onset_flux_hop_ms: 5.0
onset_flux_min_ratio: 0.45
# Earliest onset must still be this fraction of the strongest attack in the window
onset_flux_early_rel: 0.55
# Proposal stage: quieter volley shots vs the loudest cannon
onset_flux_proposal_ratio: 0.16
# Mask continuous bed by sustained spans when rumble is intermittent
sustained_mask_min_events: 3
sustained_mask_min_coverage: 0.15

# --- Frame-level SED (DCASE-style decoding) ---------------------------------
# Window tagging localizes onsets to ~1 s. These posteriors are on a 100 ms grid
# (PANNs framewise is ~10 ms when installed), then decoded with hysteresis.
sed_enabled: true
sed_backend: auto        # auto | ast_dense | panns
sed_window_sec: 1.0
sed_hop_sec: 0.1
sed_median_impulsive_sec: 0.15
sed_median_sustained_sec: 0.45
sed_onset_high: 0.30
sed_onset_low: 0.15
sed_min_event_sec_impulsive: 0.10
sed_min_event_sec_sustained: 0.40
sed_merge_gap_sec: 0.20
# Splitting a volley: a shot peak must reach this fraction of the span's strongest
sed_peak_rel: 0.60

# --- Picture flashes (not ViViT) --------------------------------------------
# Audio peaks on this clip often sit on a boom or a cut, while the fireball is
# a different frame. Orange-pixel onsets snap gunshot/explosion peaks onto the
# picture and add flashes the soundtrack missed. Audio bangs with no flash are
# kept (do not drop off-screen / low-orange fires).
visual_flash_enabled: true
visual_flash_min_d_warm: 0.025
visual_flash_min_warm: 0.025
visual_flash_min_d_hot: -0.005
visual_flash_match_sec: 0.50
visual_flash_min_sep_sec: 0.45

# --- Video branch -----------------------------------------------------------
# Disabled on purpose. ViViT is Kinetics-400 (human actions: "driving car",
# "exploding firecrackers"), which does not cover explosion / engine rumble as
# acoustic events, so video_score came out null on every event and contributed
# nothing to fusion. Revisit with an audio-visual model trained on VGGSound or a
# learned head on VideoMAE features, then set use_video: true.
use_video: false

categories:
  weather:
    audioset_labels:
      - Thunder
      - Rain
      - Wind
      - Rain on surface
      - Storm
    kinetics_labels: []
    context_token_labels:
      - thunder
      - storm
      - rain
    audio_weight: 0.8
    video_weight: 0.2
    # Thunder is an impulsive transient — snap onset via spectral flux
    impulsive: true
    include_in_haptic_gate: true

  gunshot:
    audioset_labels:
      - Gunshot, gunfire
      - Machine gun
      - Cap gun
      - Fusillade
    kinetics_labels:
      - shooting gun
      - playing paintball
    context_token_labels:
      - gunshot
      - gunfire
    audio_weight: 0.7
    video_weight: 0.3
    require_context_or_both: false
    impulsive: true
    include_in_haptic_gate: true

  explosion:
    audioset_labels:
      - Explosion
      - Artillery fire
      - Fireworks
      - Burst, pop
      - Boom
    kinetics_labels:
      - exploding firecrackers
    context_token_labels:
      - explosion
      - boom
      - artillery
      - cannon
    audio_weight: 0.7
    video_weight: 0.3
    require_context_or_both: false
    impulsive: true
    include_in_haptic_gate: true

  vehicle:
    audioset_labels:
      - Vehicle
      - Engine
      - Truck
      - Motor vehicle (road)
      - Idling
      - Tank
    kinetics_labels:
      - driving car
      - riding a bike
      - motorcycling
    context_token_labels:
      - engine_rumble
      - vehicle
      - tank
    audio_weight: 0.75
    video_weight: 0.25
    # Sustained rumble — not impulsive, lower confidence threshold handled by encoder_threshold
    impulsive: false
    include_in_haptic_gate: true

  human_activity:
    audioset_labels:
      - Chainsaw
      - Sawing
      - Wood
    kinetics_labels:
      - chopping wood
      - cutting trees
      - using axe
    context_token_labels:
      - sawing
      - chainsaw
    audio_weight: 0.4
    video_weight: 0.6
    include_in_haptic_gate: false
"""

}

for name, source in FILES.items():
    target = PKG_DIR / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

sys.path.insert(0, str(PROJECT_ROOT))
print("Installed haptic_gt at", PKG_DIR)
print("Modules:", ", ".join(sorted(FILES)))
from haptic_gt.context.encoders import AST_MODEL_ID, VIVIT_MODEL_ID
print("AST model:", AST_MODEL_ID)
print("ViViT model:", VIVIT_MODEL_ID)


In [ ]:
# Workspace folders on this Colab VM (no Google Drive needed)
from pathlib import Path

WORK_DIR = Path("/content/haptic-workspace")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Upload a video in the next cell.")
print("Results will be written to:", OUTPUT_DIR)
print("Download the ZIP at the end — Colab deletes /content when the session ends.")


In [ ]:
from pathlib import Path
from IPython.display import Audio, display

candidates = []
for folder in (Path("/content"), Path("/content/haptic-workspace/input")):
    if folder.exists():
        candidates.extend(sorted(folder.glob("*.mp4")))
        candidates.extend(sorted(folder.glob("*.mkv")))
        candidates.extend(sorted(folder.glob("*.webm")))
# Prefer a clip already on the runtime so Run All does not wait on a file picker.
video_path = next((p for p in candidates if p.is_file()), None)
if video_path is None:
    from google.colab import files
    print("Choose a video file to upload...")
    uploaded = files.upload()
    video_name = next(iter(uploaded))
    video_path = Path("/content") / video_name

print("Using video:", video_path)
print("Output folder:", OUTPUT_DIR)


In [ ]:
%%time
import json
import soundfile as sf
from pathlib import Path
from haptic_gt.pipeline import OUTPUT_NAMES, generate_candidate_tracks

CONTENT_TYPE = "game"
ENABLE_CONTEXT = True  # set False to skip AST/ViViT and re-run Sound2Hap only
# Categories to include in gated haptics
GATE_CATEGORIES = ["weather", "gunshot", "explosion", "vehicle", "human_activity"]

# Auto-detect timing (model alone). Only set a dict for HITL override.
MANUAL_EVENTS = None
# Example override (optional):
# MANUAL_EVENTS = {
#     "category": "explosion",
#     "start_sec": 0.22,
#     "peak_sec": 0.24,
#     "end_sec": 3.66,
# }

# Replace auto vehicle with your rumble marks (keeps auto gunshot/explosion).
# Calib: true rumbles often rise ~1.11–1.23 while bed FPs rise ~1.38+ — not separable by RMS.
MANUAL_RUMBLE_PEAKS = None
# MANUAL_RUMBLE_PEAKS = [6.0, 24.0, 29.0, 30.0, 31.0, 31.9, 41.0, 42.0, 43.0, 45.0, 48.0, 49.0, 51.0]

source_wav = OUTPUT_DIR / OUTPUT_NAMES["source_audio"]
if source_wav.exists() and not ENABLE_CONTEXT and MANUAL_EVENTS is None:
  print("Reusing existing source audio:", source_wav)
  input_path = source_wav
  from_video = False
else:
  input_path = video_path
  from_video = True

tracks = generate_candidate_tracks(
    input_path,
    OUTPUT_DIR,
    from_video=from_video,
    content_type=CONTENT_TYPE,
    enable_context_detection=ENABLE_CONTEXT,
    gate_categories=GATE_CATEGORIES,
    manual_events=MANUAL_EVENTS,
    manual_rumble_peaks=MANUAL_RUMBLE_PEAKS,
)
saved = tracks.save_all()

src_audio, src_sr = sf.read(saved["source_audio"])
print(f"Source: {len(src_audio)/src_sr:.1f}s @ {src_sr} Hz")
print(f"Haptic input: {tracks.haptic_input_wav}")
print(f"No events detected: {tracks.no_events_detected}")
print(f"No haptic events (gate): {tracks.no_haptic_events}")
print(f"Gate categories: {tracks.gate_categories_used}")
if tracks.events_json and tracks.events_json.exists():
    print(f"Events: {tracks.events_json}")
    print(json.dumps(json.loads(tracks.events_json.read_text()), indent=2)[:2000])

print("Saved files:")
missing = []
for name, path in saved.items():
    ok = path.exists()
    print(f"  {name}: {path} [{'OK' if ok else 'MISSING'}]")
    if not ok:
        missing.append(name)

sound2hap_keys = [
    "algorithm_a_perception_mapping",
    "algorithm_b_frequency_shifting",
    "algorithm_c_pitch_matching",
    "algorithm_d_haptic_gen",
]
if tracks.no_haptic_events:
    print("No gate-eligible events — Sound2Hap A–D skipped (see events.json). Rule-based E still ran.")
    missing = [name for name in missing if name not in sound2hap_keys]
if "algorithm_e_rule_based" not in saved:
    missing.append("algorithm_e_rule_based")
if missing:
    raise RuntimeError(
        "Generation incomplete — missing outputs: "
        + ", ".join(missing)
        + ". Scroll up for the first error, fix it, then re-run this cell."
    )
print("Generation complete.")


In [ ]:
# Events timeline visualization (run after generation cell)
import json
from pathlib import Path
import matplotlib.pyplot as plt

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

events_path = Path(OUTPUT_DIR) / "events.json"
if events_path.exists():
    data = json.loads(events_path.read_text())
    events = data.get("events", [])
    if events:
        fig, ax = plt.subplots(figsize=(12, 2 + len(events) * 0.3))
        colors = {
            "weather": "tab:blue",
            "gunshot": "tab:red",
            "explosion": "tab:purple",
            "vehicle": "tab:orange",
            "human_activity": "tab:green",
        }
        for i, ev in enumerate(events):
            c = colors.get(ev["category"], "tab:gray")
            ax.barh(i, ev["end_sec"] - ev["start_sec"], left=ev["start_sec"],
                    height=0.6, color=c, alpha=0.7)
            ax.plot(ev["peak_sec"], i, "k|", markersize=12)
            ax.text(ev["end_sec"], i,
                    f" {ev['label']} ({ev['confidence']:.0%})",
                    va="center", fontsize=9)
        ax.set_xlabel("Time (s)")
        ax.set_yticks(range(len(events)))
        ax.set_yticklabels([e["category"] for e in events])
        ax.set_title("Detected events timeline")
        plt.tight_layout()
        plt.show()
    else:
        print("No events in events.json.")
else:
    print("Run the generation cell first to create events.json")


In [ ]:
# Calibrate vehicle rumble against YOUR marked peaks (span coverage + local RMS).
# Scoring only: fill TRUTH_RUMBLE_TIMES, run after generation. Auto gates on loudness.
from pathlib import Path
import json
import soundfile as sf
from haptic_gt.context.rumble_calib import calibrate_rumble_thresholds, format_calibration_table
from haptic_gt.pipeline import OUTPUT_NAMES

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

# Rumble times you hear in THIS clip (seconds). Empty by default on purpose:
# marks left over from another clip score this one against the wrong video and
# every number in the report comes out zero.
# GRADING KEY ONLY -- this does not steer detection. The detector already ran,
# unaided, in the generate cell; these marks just say where you expected it to
# fire so the report can count hits and misses. (The HITL override that *does*
# change detection is MANUAL_RUMBLE_PEAKS in the generate cell, left at None.)
TRUTH_RUMBLE_TIMES = []

source = OUTPUT_DIR / OUTPUT_NAMES["source_audio"]
events_path = OUTPUT_DIR / OUTPUT_NAMES["events_json"]
if not source.exists() or not events_path.exists():
    raise FileNotFoundError("Run the generation cell first (need source_audio.wav + events.json).")

report = calibrate_rumble_thresholds(
    source,
    TRUTH_RUMBLE_TIMES,
    events_path,
    match_tolerance_sec=1.0,
)
print(format_calibration_table(report))
print(
    "\nAuto rumble windows are loud RMS islands (not AST start/end slabs). "
    "A 29–32s island hits every mark inside it; quiet gaps should stay unmarked. "
    "These marks grade the run; they never feed the detector."
)
(OUTPUT_DIR / "rumble_calibration.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
print("Wrote", OUTPUT_DIR / "rumble_calibration.json")

# Dense 100 ms scan. The whole clip by default: a fixed window belongs to
# whichever video it was typed for, and on the next clip it scans a second of
# nothing. Narrow it only to zoom in on a stretch you are arguing about.
from haptic_gt.context.rumble_calib import scan_rumble_timeline, format_timeline_scan
import matplotlib.pyplot as plt

_dur = float(sf.info(str(source)).duration)
SCAN_START = 0.0
SCAN_END = _dur
SCAN_HOP = 0.1  # 100 ms. Set 0.01 for 10 ms (more rows).

SCAN_START = max(0.0, min(SCAN_START, _dur))
SCAN_END = min(SCAN_END, _dur)

scan = scan_rumble_timeline(
    source,
    start_sec=SCAN_START,
    end_sec=SCAN_END,
    hop_sec=SCAN_HOP,
    events_json=events_path,
    manual_peaks_sec=TRUTH_RUMBLE_TIMES,
)
print(format_timeline_scan(scan, only_manual_windows=bool(TRUTH_RUMBLE_TIMES)))
print("\n(Full scan saved; the table shows ticks near your marks when you set any.)")
SCAN_NAME = "rumble_scan.json"
(OUTPUT_DIR / SCAN_NAME).write_text(json.dumps(scan, indent=2), encoding="utf-8")

ts = [s["t_sec"] for s in scan["samples"]]
rises = [s["rms_rise_ratio"] or 0.0 for s in scan["samples"]]
rmss = [s["local_rms"] for s in scan["samples"]]
thr = scan["summary"].get("salience_threshold")
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(ts, rises, lw=1)
ax[0].axhline(1.10, color="gray", ls="--", lw=0.8, label="1.10")
ax[0].axhline(1.25, color="orange", ls="--", lw=0.8, label="1.25")
ax[0].axhline(1.35, color="red", ls="--", lw=0.8, label="1.35")
for m in TRUTH_RUMBLE_TIMES:
    if SCAN_START <= m <= SCAN_END:
        ax[0].axvline(m, color="green", alpha=0.4, lw=1)
ax[0].set_ylabel("RMS rise (unused)")
ax[0].legend(loc="upper right", fontsize=8)
ax[0].set_title(f"Rumble scan {SCAN_START:.1f}–{SCAN_END:.1f}s @ {SCAN_HOP}s")
ax[1].plot(ts, rmss, lw=1, color="tab:blue")
if thr is not None:
    ax[1].axhline(thr, color="purple", ls="--", lw=1, label=f"salience {thr:.3f}")
for m in TRUTH_RUMBLE_TIMES:
    if SCAN_START <= m <= SCAN_END:
        ax[1].axvline(m, color="green", alpha=0.4, lw=1)
ax[1].set_ylabel("local RMS (auto gate)")
ax[1].set_xlabel("Time (s)")
ax[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()
print("Wrote", OUTPUT_DIR / SCAN_NAME)

# What the loudness gate decided, so a missing rumble is diagnosable: one floor
# per scene, plus every span it dropped and how far under the floor it was.
gate = json.loads(events_path.read_text(encoding="utf-8")).get("sustained_gate") or {}
if gate:
    print("\nrumble gate")
    print("  clip floor {0:.4f}  proposals {1}  kept {2}".format(
        gate.get("clip_threshold", 0.0), gate.get("proposals", 0), gate.get("kept", 0)
    ))
    for sc in gate.get("scenes", []):
        print("  scene {0:6.2f}-{1:6.2f}s  floor {2:.4f}".format(
            sc["start_sec"], sc["end_sec"], sc["threshold"]
        ))
    for d in gate.get("dropped", []):
        print("  dropped {0:.2f}-{1:.2f}s  local RMS {2:.4f} vs floor {3:.4f}  ({4})".format(
            d["start_sec"], d["end_sec"], d["local_rms"], d["scene_threshold"], d["reason"]
        ))
    if not gate.get("dropped"):
        print("  nothing dropped: rumble you cannot feel was never proposed, not gated out")


In [ ]:
# Calibrate cannon / gunshot timing against YOUR marked shot times.
# Shows, for each mark, the nearest real attack in the audio and how strong it is
# relative to this clip's confident blasts. Use it to tell a detector error from
# a mis-typed mark.
from pathlib import Path
import json
import matplotlib.pyplot as plt
from haptic_gt.context.shot_calib import (
    calibrate_shot_times,
    format_shot_calibration_table,
    format_shot_scan,
    scan_shot_attacks,
)
from haptic_gt.pipeline import OUTPUT_NAMES

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

# Shot times you hear in THIS clip (seconds). Empty by default: another clip's
# times score this one against the wrong video, and the whole report reads as a
# detector failure when the marks are simply not from this edit.
# GRADING KEY ONLY: detection already happened, unaided, in the generate cell.
TRUTH_SHOT_TIMES = []

source = OUTPUT_DIR / OUTPUT_NAMES["source_audio"]
events_path = OUTPUT_DIR / OUTPUT_NAMES["events_json"]
if not source.exists() or not events_path.exists():
    raise FileNotFoundError("Run the generation cell first (need source_audio.wav + events.json).")

report = calibrate_shot_times(
    source,
    TRUTH_SHOT_TIMES,
    events_path,
    match_tolerance_sec=0.35,
)
print(format_shot_calibration_table(report))
(OUTPUT_DIR / "shot_calibration.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

# Every strong attack in the clip, whether or not it became an event.
scan = scan_shot_attacks(source, top_n=40)
print()
print(format_shot_scan(scan))
(OUTPUT_DIR / "shot_scan.json").write_text(json.dumps(scan, indent=2), encoding="utf-8")

ts = [a["t_sec"] for a in scan["attacks"]]
vals = [a["flux_rel_max"] for a in scan["attacks"]]
fig, ax = plt.subplots(figsize=(12, 3))
ax.stem(ts, vals, basefmt=" ")
for m in TRUTH_SHOT_TIMES:
    ax.axvline(m, color="green", alpha=0.45, lw=1)
for e in json.loads(events_path.read_text(encoding="utf-8"))["events"]:
    if e["category"] in ("explosion", "gunshot"):
        ax.axvline(e["peak_sec"], color="red", ls="--", alpha=0.6, lw=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("attack / clip max")
ax.set_title("Flux attacks — green = your marks, red dashed = detected shots")
plt.tight_layout()
plt.show()
print("Wrote", OUTPUT_DIR / "shot_calibration.json", "and", OUTPUT_DIR / "shot_scan.json")


In [ ]:
# DCASE-style scoring of events.json against your ground truth.
# Event-based F1 uses a one-to-one onset match inside a collar (DCASE Task 4 uses
# 200 ms); segment-based F1 ignores onset jitter and asks "did we find it at all".
from pathlib import Path
import json
from haptic_gt.context.taxonomy import load_taxonomy
from haptic_gt.eval.sed_metrics import evaluate_events, format_evaluation
from haptic_gt.pipeline import OUTPUT_NAMES
import soundfile as sf

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

# Ground truth comes from the marks you entered in the calibration cells above,
# so one clip cannot be scored against another clip's marks. Run those first, or
# set GROUND_TRUTH here by hand.
GROUND_TRUTH = {
    "explosion": list(globals().get("TRUTH_SHOT_TIMES") or []),
    "vehicle": list(globals().get("TRUTH_RUMBLE_TIMES") or []),
}
GROUND_TRUTH = {k: v for k, v in GROUND_TRUTH.items() if v}

source = OUTPUT_DIR / OUTPUT_NAMES["source_audio"]
events_path = OUTPUT_DIR / OUTPUT_NAMES["events_json"]
if not source.exists() or not events_path.exists():
    raise FileNotFoundError("Run the generation cell first.")

info = sf.info(str(source))
duration = float(info.duration)
events = json.loads(events_path.read_text(encoding="utf-8"))
print("detector:", events.get("detector", {}))
print("clip duration: {0:.2f}s".format(duration))
print("ground truth:", {k: len(v) for k, v in GROUND_TRUTH.items()})

detected_cats = {e["category"] for e in events.get("events", [])}
for cat, marks in GROUND_TRUTH.items():
    if any(m > duration for m in marks):
        print(
            "WARNING: {0} marks fall past the end of this clip -- these look like "
            "another video's marks, scores will be meaningless.".format(cat)
        )
    if cat not in detected_cats:
        print(
            "WARNING: {0} marks exist but nothing of that category was detected. "
            "If this clip has no {0}, clear those marks -- otherwise every one "
            "counts as a miss and drags F1 to 0.".format(cat)
        )

if not GROUND_TRUTH:
    print(
        "Nothing to score yet. Put what YOU hear in this clip into "
        "TRUTH_SHOT_TIMES / TRUTH_RUMBLE_TIMES in the calibration cells above and "
        "re-run them, then run this cell. Detection does not need them; scoring "
        "does. Detected so far:"
    )
    for e in events.get("events", []):
        print("  {0:14} {1:7.2f}-{2:7.2f}s  peak {3:7.2f}s".format(
            e["category"], e["start_sec"], e["end_sec"], e["peak_sec"]
        ))
else:
    # Rumble marks are moments inside a burst, not onsets, so they get coverage
    # (recall + how much of the clip vibrates) instead of an onset-collar F1.
    sustained = tuple(
        name for name, cfg in load_taxonomy().categories.items() if not cfg.impulsive
    )
    report = evaluate_events(
        GROUND_TRUTH,
        events,
        duration_sec=duration,
        collars_sec=(0.2, 0.5),
        segment_sec=1.0,
        sustained_categories=sustained,
    )
    print()
    print(format_evaluation(report))
    (OUTPUT_DIR / "sed_evaluation.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print()
    print("0.2 s is the DCASE onset collar. Hand marks drift ~0.5 s, so read both.")
    print("Wrote", OUTPUT_DIR / "sed_evaluation.json")


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
from haptic_gt.pipeline import OUTPUT_NAMES

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

labels = {
    "source_audio": "Source audio",
    "algorithm_a_perception_mapping": "A — Perception mapping",
    "algorithm_b_frequency_shifting": "B — Frequency shifting",
    "algorithm_c_pitch_matching": "C — Pitch matching",
    "algorithm_d_haptic_gen": "D — HapticGen",
    "algorithm_e_rule_based": "E — Rule-based (video + RMS)",
}

paths = {}
if "saved" in globals():
    paths.update({k: v for k, v in saved.items() if k in labels and v.exists()})
for key in labels:
    if key not in paths:
        candidate = OUTPUT_DIR / OUTPUT_NAMES[key]
        if candidate.exists():
            paths[key] = candidate

missing = [labels[k] for k in labels if k not in paths]
if missing:
    print("Some tracks are missing — re-run the generation cell:")
    for name in missing:
        print(" -", name)
    on_disk = sorted(p.name for p in OUTPUT_DIR.glob("*"))
    print(f"\nCurrently in {OUTPUT_DIR}:", on_disk or "(empty)")
    if not paths:
        raise RuntimeError("No audio outputs found yet. Run the generation cell first.")

for key, label in labels.items():
    if key not in paths:
        continue
    print(label)
    display(Audio(str(paths[key])))


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/haptic-workspace/output")

zip_base = OUTPUT_DIR.parent / "haptic_candidates"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR))
print("ZIP created:", zip_path)

files.download(str(zip_path))
print("Download started.")


## Human-in-the-loop (next step)

1. Play **source audio** (44.1 kHz) in headphones while feeling each **8 kHz** candidate on haptic hardware.
2. Compare **E** (rule-based) against Sound2Hap **A–D**. Rate **realism** and **similarity** (e.g. 1–7 Likert).
3. If agreement < threshold, tune parameters in `haptic_gt/algorithms/*.py` and re-run.
4. Approved tracks become your **ground truth dataset**.